# RaceEng Challenge API Example

## Important Warnings
- This notebook can launch a large number of Canopy simulations.
- The actual simulation count depends on your settings, retries, resume state, worksheet recovery, verification work, and endgame or final-validation behaviour.
- Before any new studies are launched, the notebook estimates the remaining simulation budget, estimates compute credits on a 1 dynamic lap simulation = 1 compute credit basis, and asks for explicit consent.
- This notebook is AI-generated example code. Review, test, and adapt it before using it in a production or customer workflow.
- This notebook is intended as inspiration and example code for further vibe-coding applications, including worksheet interactions, API reconnect and token refresh handling, auto-resume flows, and optimisation algorithms.
- Credentials and access tokens are kept in memory only. This notebook does not intentionally persist credentials, access tokens, or refresh tokens to disk.

## What This Notebook Does
- Prompts for Canopy credentials at runtime.
- Loads the canonical challenge car, weather, track, and user maths configs from Canopy cloud defaults.
- Writes progress to a worksheet while also keeping local state, resume, and reporting files.
- Uses continuation-token paging for study job enumeration and proactively refreshes access tokens before direct OpenAPI calls.

## Local Files Written
- Campaign state and resume JSON files.
- Pending and launched study trace files.
- Worksheet manifest, worksheet row trace, and reconnect trace files.
- Result JSON, sweep summaries, sensitivity summaries, and iteration CSV outputs.

## Customer Notes
- Edit the **Customer Settings** cell for normal use.
- Only edit the **Advanced Settings** cell if you want to tune runtime behaviour.
- The run cell accepts 4 required credential lines in this order: `username`, `password`, `client_id`, `client_secret`. You may optionally add a 5th line for `tenant_name`; if omitted, the notebook uses `client_id` for both values.

## Reference Docs
- [Using worksheets via the API](https://support.canopysimulations.com/hc/en-gb/articles/9386198563357-Using-worksheets-via-the-API)
- [Continuation Tokens](https://support.canopysimulations.com/hc/en-gb/articles/5663976105245-Continuation-Tokens)
- [Authenticating Programmatically](https://support.canopysimulations.com/hc/en-gb/articles/5664140831005-Authenticating-Programmatically)
- [Swagger / OpenAPI](https://api.canopysimulations.com/swagger/index.html)


## Customer Settings

Edit this cell for normal customer use.

`worksheet_destination` can be either:
- a 32-character worksheet ID, or
- a base worksheet name.

If `auto_increment_named_worksheets=True` and the destination is a name, the notebook resolves or creates numbered worksheets using the local registry file.


In [ ]:
from dataclasses import dataclass, field


@dataclass(frozen=True)
class RaceEngCustomerSettings:
    # `target_tenant_id` is the 32-character tenant ID used by worksheet/study API calls.
    # The credential prompt later will separately ask for the OAuth `client_id` and optional `tenant_name` used at `/token`.
    target_tenant_id: str
    worksheet_destination: str
    auto_increment_named_worksheets: bool
    create_worksheet_if_missing: bool
    base_run_prefix: str
    total_target_iterations: int


@dataclass(frozen=True)
class RaceEngAdvancedSettings:
    worksheet_registry_file: str
    max_auto_restarts: int
    retry_delay_seconds: int
    study_timeout_seconds: int
    sweep_points_per_parameter: int
    focus_fraction: float
    endgame_fraction: float
    endgame_verify_top_k: int
    endgame_extra_cycles_max: int
    endgame_stall_patience: int
    endgame_noise_margin_factor: float
    elite_archive_size: int
    endgame_verification_debt_threshold: float
    endgame_unverified_improvement_reset: float
    final_candidate_policy: str
    force_two_main_starts: bool
    worksheet_row_batch_seconds: float
    worksheet_row_batch_size: int


@dataclass(frozen=True)
class RaceEngExampleConfig:
    customer: RaceEngCustomerSettings
    advanced: RaceEngAdvancedSettings
    default_config_urls: dict[str, str] = field(default_factory=dict)


RUN_CUSTOMER_SETTINGS = RaceEngCustomerSettings(
    target_tenant_id="a4ed02d5506c4237a58d520e151f74be",
    worksheet_destination="RaceEng Challenge API Example",
    auto_increment_named_worksheets=True,
    create_worksheet_if_missing=True,
    base_run_prefix="raceeng-challenge-api-example",
    total_target_iterations=30,
)

RUN_CUSTOMER_SETTINGS


## Advanced Settings

These settings control retry behaviour, study sizing, worksheet batching, and the endgame/final-validation policy. Most customers should leave them unchanged.


In [ ]:
RUN_ADVANCED_SETTINGS = RaceEngAdvancedSettings(
    worksheet_registry_file="worksheet_registry.json",
    max_auto_restarts=40,
    retry_delay_seconds=20,
    study_timeout_seconds=2400,
    sweep_points_per_parameter=5,
    focus_fraction=1.0,
    endgame_fraction=0.35,
    endgame_verify_top_k=2,
    endgame_extra_cycles_max=20,
    endgame_stall_patience=6,
    endgame_noise_margin_factor=0.75,
    elite_archive_size=8,
    endgame_verification_debt_threshold=0.002,
    endgame_unverified_improvement_reset=0.001,
    final_candidate_policy="top2",
    force_two_main_starts=True,
    worksheet_row_batch_seconds=2.0,
    worksheet_row_batch_size=8,
)

RUN_CONFIG = RaceEngExampleConfig(
    customer=RUN_CUSTOMER_SETTINGS,
    advanced=RUN_ADVANCED_SETTINGS,
    default_config_urls={
        "car": "https://portal.canopysimulations.com/default-configs/1.14301/cars/Canopy%20F1%20Car%202025%20Race%20Engineering%20Challenge/edit",
        "weather": "https://portal.canopysimulations.com/default-configs/1.14301/weather/25%20deg,%20dry/edit",
        "track": "https://portal.canopysimulations.com/default-configs/1.14301/tracks/Barcelona-F1/edit",
        "userMaths": "https://portal.canopysimulations.com/default-configs/1.14301/userMaths/Canopy%202025%20Race%20Engineering%20Challenge/edit",
    },
)

RUN_CONFIG


## Canopy / API Helpers

This section contains the authentication, retry, worksheet merge, continuation-token paging, study loading, and resume helpers used by the run orchestration below.


In [ ]:
import asyncio
import copy
import getpass
import hashlib
import json
import math
import logging
import re
import statistics
from dataclasses import dataclass, field
from datetime import UTC, datetime
from pathlib import Path
from time import perf_counter
from typing import Any, Callable
from urllib.parse import unquote, urlparse
from uuid import uuid4

import aiohttp
import canopy
import numpy as np
import pandas as pd
from canopy.openapi import (
    ConfigReferenceTenant,
    StudyApi,
    WorksheetApi,
    WorksheetConfig,
    ConfigResolvedLabelsReference,
    WorksheetPostWorksheetRequest,
    WorksheetPutWorksheetRequest,
    WorksheetRow,
    WorksheetRowStudy,
    StudyResolvedLabelsReference,
)

if hasattr(asyncio, "WindowsSelectorEventLoopPolicy"):
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

# Keep Canopy transient blob fetch warnings out of notebook output; retry logic handles them.
logging.getLogger("canopy").setLevel(logging.ERROR)
logging.getLogger("canopy.openapi").setLevel(logging.ERROR)

# RaceEngChallengeAPIExample runs without external input files: credentials stay
# in memory and canonical inputs are pulled from Canopy cloud defaults at runtime.
_WORKSHEET_ROW_UPSERT_LOCKS: dict[str, asyncio.Lock] = {}
_WORKSHEET_ROW_UPSERT_LOCKS_GUARD = asyncio.Lock()
_RACEENG_LAUNCH_SUFFIX_RE = re.compile(r"^(?P<base>.+?)--L(?P<launch>[0-9a-fA-F]{8})$")


async def _get_worksheet_upsert_lock(tenant_id: str, worksheet_id: str) -> asyncio.Lock:
    key = f"{tenant_id}:{worksheet_id}"
    async with _WORKSHEET_ROW_UPSERT_LOCKS_GUARD:
        lock = _WORKSHEET_ROW_UPSERT_LOCKS.get(key)
        if lock is None:
            lock = asyncio.Lock()
            _WORKSHEET_ROW_UPSERT_LOCKS[key] = lock
        return lock


def raceeng_new_launch_id() -> str:
    return uuid4().hex[:8]


def raceeng_make_physical_row_name(logical_row_name: str, launch_id: str) -> str:
    return f"{str(logical_row_name)}--L{str(launch_id)}"


def raceeng_strip_launch_suffix(name: str) -> str:
    text = str(name or "")
    m = _RACEENG_LAUNCH_SUFFIX_RE.match(text)
    return str(m.group("base")) if m is not None else text



class RaceEngUserCancelledRun(RuntimeError):
    pass


def raceeng_clear_cached_auth_data():
    globals().pop("RACEENG_AUTH_DATA", None)


def raceeng_clone_auth_data(auth_data: canopy.AuthenticationData | None) -> canopy.AuthenticationData | None:
    if auth_data is None:
        return None
    return canopy.AuthenticationData(
        client_id=getattr(auth_data, "client_id", None),
        client_secret=getattr(auth_data, "client_secret", None),
        username=getattr(auth_data, "username", None),
        tenant_name=getattr(auth_data, "tenant_name", None),
        password=getattr(auth_data, "password", None),
    )


def raceeng_build_auth_data(
    username: str,
    password: str,
    client_id: str,
    client_secret: str,
    tenant_name: str | None = None,
) -> canopy.AuthenticationData:
    username = str(username or "").strip()
    password = str(password or "")
    client_id = str(client_id or "").strip()
    client_secret = str(client_secret or "")
    tenant_name = str(tenant_name or "").strip() or client_id
    if not username or not password or not client_id or not client_secret:
        raise ValueError("Username, password, client ID, and client secret must all be provided.")
    return canopy.AuthenticationData(
        client_id=client_id,
        client_secret=client_secret,
        username=username,
        tenant_name=tenant_name,
        password=password,
    )


def raceeng_parse_credential_block(block: str) -> canopy.AuthenticationData:
    lines = [line.rstrip("\r") for line in str(block or "").splitlines()]
    while lines and not str(lines[-1]).strip():
        lines.pop()
    if len(lines) == 4:
        username, password, client_id, client_secret = lines
        return raceeng_build_auth_data(username, password, client_id, client_secret)
    if len(lines) == 5:
        username, password, client_id, client_secret, tenant_name = lines
        return raceeng_build_auth_data(username, password, client_id, client_secret, tenant_name)
    raise ValueError(
        "Expected either 4 credential lines in the order `username, password, client_id, client_secret` "
        "or 5 lines in the order `username, password, client_id, client_secret, tenant_name`. "
        f"Received {len(lines)} line(s)."
    )


def _raceeng_prompt_auth_fields_tk() -> str:
    import tkinter as tk
    from tkinter import ttk

    result: dict[str, str | None] = {"value": None}
    root = tk.Tk()
    root.title("RaceEngChallengeAPIExample Canopy Credentials")
    root.attributes("-topmost", True)
    root.resizable(False, False)

    frame = ttk.Frame(root, padding=12)
    frame.grid(row=0, column=0, sticky="nsew")

    ttk.Label(
        frame,
        text=(
            "Paste 4 required lines in this order:\n"
            "1. username\n"
            "2. password\n"
            "3. client_id\n"
            "4. client_secret\n\n"
            "Optional:\n"
            "5. tenant_name\n\n"
            "If line 5 is omitted, this notebook will use `client_id` for both values."
        ),
        justify="left",
    ).grid(row=0, column=0, columnspan=2, sticky="w")

    text = tk.Text(frame, width=72, height=8, wrap="none")
    text.grid(row=1, column=0, columnspan=2, pady=(8, 10), sticky="nsew")
    text.focus_set()

    def _submit():
        result["value"] = text.get("1.0", "end-1c")
        root.quit()

    def _cancel():
        root.quit()

    root.protocol("WM_DELETE_WINDOW", _cancel)

    ttk.Button(frame, text="Use Credentials", command=_submit).grid(row=2, column=0, sticky="ew", padx=(0, 6))
    ttk.Button(frame, text="Cancel", command=_cancel).grid(row=2, column=1, sticky="ew")

    root.update_idletasks()
    root.mainloop()
    try:
        root.destroy()
    except tk.TclError:
        pass

    value = result.get("value")
    if value is None:
        raise RaceEngUserCancelledRun("Run cancelled during credential entry.")
    return str(value)


def raceeng_prompt_for_authentication_data() -> canopy.AuthenticationData:
    try:
        block = _raceeng_prompt_auth_fields_tk()
        return raceeng_parse_credential_block(block)
    except RaceEngUserCancelledRun:
        raise
    except Exception as ex:
        print(
            f"Credential paste prompt unavailable ({type(ex).__name__}: {ex}). "
            "Falling back to sequential prompts."
        )
        username = input("Username: ").strip()
        password = getpass.getpass(prompt="Password: ")
        client_id = input("Client ID: ").strip()
        client_secret = getpass.getpass(prompt="Client Secret: ")
        tenant_name = input("Tenant Name (optional; press Enter to use Client ID): ").strip()
        return raceeng_build_auth_data(username, password, client_id, client_secret, tenant_name)


async def authenticate_with_auth_data(auth_data: canopy.AuthenticationData) -> canopy.Session:
    if auth_data is None:
        raise ValueError("Authentication data is required.")
    session = canopy.Session(raceeng_clone_auth_data(auth_data))
    try:
        session.authentication.authenticate()
        print("Authenticated")
        return session
    except Exception as ex:
        await session.close()
        raise RuntimeError(f"Authentication failed: {ex}") from ex


@dataclass
class RaceEngRuntime:
    auth_data: canopy.AuthenticationData | None = None
    session: canopy.Session | None = None
    campaign_state_stem: str | None = None
    worksheet_tenant_id: str | None = None
    worksheet_id: str | None = None
    worksheet_name: str | None = None
    worksheet_number: int | None = None
    pending_studies: dict[str, dict] = field(default_factory=dict)
    recovered_payloads: dict[str, dict] = field(default_factory=dict)
    api_reauth_count: int = 0
    api_retry_count: int = 0
    recovered_study_count: int = 0
    reconnect_rows: list[dict] = field(default_factory=list)
    finalization_only_pass_used: bool = False
    worksheet_row_manifest: dict[str, dict] = field(default_factory=dict)
    worksheet_row_queue: list[str] = field(default_factory=list)
    worksheet_row_pending_names: set[str] = field(default_factory=set)
    worksheet_known_rows: dict[str, dict] = field(default_factory=dict)
    worksheet_rows_written: int = 0
    worksheet_rows_reinserted: int = 0
    worksheet_reconcile_retry_count: int = 0
    worksheet_reconcile_rows: list[dict] = field(default_factory=list)
    worksheet_row_batch_seconds: float = 2.0
    worksheet_row_batch_size: int = 8
    worksheet_manifest_order_next: int = 0
    worksheet_writer_event: asyncio.Event = field(default_factory=asyncio.Event)
    worksheet_writer_lock: asyncio.Lock = field(default_factory=asyncio.Lock)
    worksheet_writer_task: asyncio.Task | None = None
    worksheet_writer_shutdown: bool = False


def raceeng_error_status(ex: Exception) -> int | None:
    status = getattr(ex, "status", None)
    try:
        return None if status is None else int(status)
    except Exception:
        return None


def raceeng_error_message(ex: Exception) -> str:
    return str(ex or "").strip()


def raceeng_is_session_error(ex: Exception) -> bool:
    msg = raceeng_error_message(ex)
    status = raceeng_error_status(ex)
    return bool(
        status == 401
        or "Reason: Unauthorized" in msg
        or "invalid_token" in msg
        or "invalid_grant" in msg
        or "refresh token" in msg.lower()
        or "Session is closed" in msg
    )


def raceeng_is_retryable_api_error(ex: Exception) -> bool:
    msg = raceeng_error_message(ex)
    status = raceeng_error_status(ex)
    if raceeng_is_session_error(ex):
        return True
    if isinstance(ex, aiohttp.ClientError):
        return True
    if status in {400, 408, 409, 425, 429, 500, 502, 503, 504}:
        if status != 400:
            return True
        return bool("Invalid study job ID" in msg)
    return bool(
        "DynamicLap_ScalarResults.csv" in msg
        or "The specified blob does not exist" in msg
        or "Invalid study job ID" in msg
        or "zero successful simulations" in msg
    )


async def raceeng_runtime_ensure_session(runtime: RaceEngRuntime) -> canopy.Session:
    if runtime.session is None:
        if runtime.auth_data is None:
            raise RuntimeError("Runtime authentication data is missing.")
        runtime.session = await authenticate_with_auth_data(runtime.auth_data)
        runtime.api_reauth_count += 1
    return runtime.session


async def raceeng_runtime_reauthenticate(runtime: RaceEngRuntime, op_name: str, reason: str):
    old_session = runtime.session
    runtime.session = None
    if old_session is not None:
        try:
            await old_session.close()
        except Exception:
            pass
    runtime.reconnect_rows.append(
        {
            "timestamp_utc": datetime.now(UTC).isoformat(),
            "op_name": str(op_name),
            "attempt": 0,
            "action": "session_rebuild",
            "error_type": "session_refresh",
            "status": float("nan"),
            "message": str(reason),
        }
    )
    if runtime.auth_data is None:
        raise RuntimeError("Runtime authentication data is missing.")
    runtime.session = await authenticate_with_auth_data(runtime.auth_data)
    runtime.api_reauth_count += 1


async def raceeng_call_with_retry(
    runtime: RaceEngRuntime | None,
    op_name: str,
    func: Callable[[canopy.Session], Any],
    *,
    max_attempts: int = 6,
    base_delay_seconds: float = 2.0,
    max_delay_seconds: float = 30.0,
) -> Any:
    if runtime is None:
        raise RuntimeError(f"Runtime is required for retryable API call '{op_name}'")

    last_ex = None
    attempts = max(1, int(max_attempts))
    for attempt in range(1, attempts + 1):
        session = await raceeng_runtime_ensure_session(runtime)
        try:
            token_before = str(getattr(session.sync_client.configuration, "access_token", "") or "")
            session.authentication.authenticate()
            token_after = str(getattr(session.sync_client.configuration, "access_token", "") or "")
            if token_before and token_after and token_before != token_after:
                runtime.reconnect_rows.append(
                    {
                        "timestamp_utc": datetime.now(UTC).isoformat(),
                        "op_name": str(op_name),
                        "attempt": int(attempt),
                        "action": "token_refresh",
                        "error_type": "",
                        "status": float("nan"),
                        "message": "Proactively refreshed access token before API call.",
                    }
                )
            return await func(session)
        except asyncio.CancelledError:
            raise
        except Exception as ex:
            last_ex = ex
            status = raceeng_error_status(ex)
            msg = raceeng_error_message(ex)
            retryable = raceeng_is_retryable_api_error(ex)
            runtime.reconnect_rows.append(
                {
                    "timestamp_utc": datetime.now(UTC).isoformat(),
                    "op_name": str(op_name),
                    "attempt": int(attempt),
                    "action": "retry" if retryable and attempt < attempts else "raise",
                    "error_type": type(ex).__name__,
                    "status": float("nan") if status is None else int(status),
                    "message": str(msg),
                }
            )
            if (not retryable) or attempt >= attempts:
                raise
            runtime.api_retry_count += 1
            if raceeng_is_session_error(ex):
                await raceeng_runtime_reauthenticate(runtime, op_name=op_name, reason=msg or type(ex).__name__)
            delay = min(float(max_delay_seconds), float(base_delay_seconds) * (2.0 ** float(attempt - 1)))
            delay *= float(0.85 + 0.30 * np.random.random())
            print(
                f"Warning: API call '{op_name}' failed on attempt {attempt}/{attempts}: "
                f"{type(ex).__name__}: {msg}. Retrying in {delay:.1f}s"
            )
            await asyncio.sleep(float(max(0.5, delay)))

    if last_ex is not None:
        raise last_ex
    raise RuntimeError(f"Retry wrapper exhausted without result for '{op_name}'")



_DEFAULT_CONFIG_TYPE_MAP = {
    "cars": "car",
    "weather": "weather",
    "tracks": "track",
    "userMaths": "userMaths",
}


def raceeng_parse_default_config_url(url: str) -> dict:
    parsed = urlparse(str(url or "").strip())
    parts = [part for part in parsed.path.split("/") if part]
    if len(parts) < 5 or parts[0] != "default-configs":
        raise ValueError(
            "Expected a Canopy default-config URL of the form "
            "https://portal.canopysimulations.com/default-configs/<simVersion>/<type>/<name>/edit"
        )
    sim_version = str(parts[1]).strip()
    config_type = _DEFAULT_CONFIG_TYPE_MAP.get(str(parts[2]).strip())
    config_name = unquote(str(parts[3]).strip())
    if not sim_version or not config_type or not config_name:
        raise ValueError(f"Could not parse default config URL: {url}")
    return {
        "url": str(url),
        "sim_version": sim_version,
        "config_type": config_type,
        "name": config_name,
    }


def raceeng_local_config_to_payload(local_config: canopy.LocalConfig, sim_version: str | None = None) -> dict:
    return {
        "config": copy.deepcopy(local_config.raw_data),
        "customProperties": copy.deepcopy(local_config.properties or {}),
        "notes": local_config.notes,
        "simVersion": sim_version,
        "name": str(local_config.name or "").strip(),
        "config_type": str(local_config.config_type or "").strip(),
    }


def raceeng_config_result_to_payload(config_result, fallback_name: str = "") -> dict:
    document = getattr(config_result, "document", None)
    return {
        "config": copy.deepcopy(getattr(document, "data", {}) or {}),
        "customProperties": copy.deepcopy(getattr(document, "properties", {}) or {}),
        "notes": getattr(document, "notes", None),
        "simVersion": getattr(document, "sim_version", None),
        "name": str(getattr(document, "name", None) or fallback_name or "").strip(),
        "config_type": str(getattr(document, "config_type", None) or "").strip(),
    }


async def load_default_config_payload_from_url(
    session,
    default_config_url: str,
    runtime: RaceEngRuntime | None = None,
) -> dict:
    spec = raceeng_parse_default_config_url(default_config_url)

    async def _load(active_session):
        return await canopy.load_default_config(
            active_session,
            spec["config_type"],
            spec["name"],
            sim_version=spec["sim_version"],
        )

    local_config = (
        await raceeng_call_with_retry(
            runtime,
            f"load_default_config:{spec['config_type']}:{spec['name']}",
            _load,
        )
        if runtime is not None
        else await _load(session)
    )
    payload = raceeng_local_config_to_payload(local_config, sim_version=spec["sim_version"])
    print(
        f"Loaded cloud default {spec['config_type']}: {payload.get('name') or spec['name']} "
        f"(simVersion={spec['sim_version']})"
    )
    return payload


def get_original_config_name(payload: dict, source_hint: str | None, fallback: str) -> str:
    if isinstance(payload.get("name"), str) and payload.get("name").strip():
        return payload.get("name").strip()
    cfg = payload.get("config") or {}
    if isinstance(cfg.get("name"), str) and cfg.get("name").strip():
        return cfg.get("name").strip()
    if isinstance(source_hint, str) and source_hint.strip():
        try:
            return raceeng_parse_default_config_url(source_hint)["name"]
        except Exception:
            try:
                stem = Path(source_hint).stem
                if stem:
                    return stem
            except Exception:
                pass
    return fallback


def worksheet_config(tenant_id: str, config_type: str, config_id: str) -> WorksheetConfig:
    return WorksheetConfig(
        config_type=config_type,
        reference=ConfigResolvedLabelsReference(
            tenant=ConfigReferenceTenant(
                tenant_id=tenant_id,
                target_id=config_id,
            )
        ),
        inherit_reference=False,
    )


async def upsert_worksheet_row(
    session,
    tenant_id: str,
    worksheet_id: str,
    row_name: str,
    worksheet_configs: list[WorksheetConfig],
    study_id: str,
    runtime: RaceEngRuntime | None = None,
):
    if runtime is not None:
        await raceeng_enqueue_worksheet_row(
            runtime=runtime,
            tenant_id=tenant_id,
            worksheet_id=worksheet_id,
            row_name=row_name,
            worksheet_configs=worksheet_configs,
            study_id=study_id,
        )
        return

    lock = await _get_worksheet_upsert_lock(tenant_id, worksheet_id)
    async with lock:
        session.authentication.authenticate()
        worksheet_api = WorksheetApi(session.async_client)
        worksheet_result = await worksheet_api.worksheet_get_worksheet(tenant_id, worksheet_id)
        worksheet = worksheet_result.worksheet

        new_row = WorksheetRow(
            name=row_name,
            configs=worksheet_configs,
            study=WorksheetRowStudy(
                reference=StudyResolvedLabelsReference(
                    tenant_id=tenant_id,
                    target_id=study_id,
                )
            ),
        )

        rows = list(worksheet.outline.rows or [])
        rows.append(new_row)
        worksheet.outline.rows = rows
        request = WorksheetPutWorksheetRequest(
            name=worksheet.name,
            properties=worksheet.properties,
            outline=worksheet.outline,
            notes=worksheet.notes,
        )
        await worksheet_api.worksheet_put_worksheet(
            tenant_id,
            worksheet_id,
            request,
        )


def sorted_jobs_by_index(jobs):
    def idx(job):
        doc_id = getattr(job.document, "document_id", "")
        try:
            return int(doc_id.rsplit("-", 1)[-1])
        except Exception:
            return 10**9

    return sorted(jobs, key=idx)

def sanitize_notes(notes):
    if notes is None:
        return None
    if not isinstance(notes, str):
        return notes

    cleaned = re.sub(r"(?i)test", "", notes)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned or None


def raceeng_format_eta(seconds: float) -> str:
    if not math.isfinite(float(seconds)):
        return "?"
    total = float(max(0.0, seconds))
    if total < 90.0:
        return f"{total:.0f}s"
    if total < 5400.0:
        return f"{total/60.0:.1f}m"
    return f"{total/3600.0:.2f}h"


def raceeng_safe_file_stem(token: str) -> str:
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(token or "")).strip("_") or "raceeng"


def raceeng_payload_hash(payload: dict) -> str:
    normalized = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=True)
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


def raceeng_campaign_state_paths(campaign_state_stem: str) -> dict[str, str]:
    stem = raceeng_safe_file_stem(campaign_state_stem)
    return {
        "json": f"{stem}_state.json",
        "history": f"{stem}_history.npz",
        "launches": f"{stem}_launches.jsonl",
        "pending": f"{stem}_pending_studies.json",
        "worksheet_rows": f"{stem}_worksheet_rows.jsonl",
        "worksheet_manifest": f"{stem}_worksheet_row_manifest.json",
        "worksheet_reconcile_trace": f"{stem}_worksheet_reconcile_trace.csv",
    }


def raceeng_append_jsonl(path: str, payload: dict):
    with Path(path).open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, ensure_ascii=True) + "\n")


def raceeng_serialize_worksheet_config(cfg: WorksheetConfig) -> dict:
    ref = getattr(cfg, "reference", None)
    tenant_ref = None if ref is None else getattr(ref, "tenant", None)
    return {
        "config_type": str(getattr(cfg, "config_type", "")),
        "tenant_id": str(getattr(tenant_ref, "tenant_id", "")),
        "target_id": str(getattr(tenant_ref, "target_id", "")),
        "inherit_reference": bool(getattr(cfg, "inherit_reference", False)),
    }


def raceeng_deserialize_worksheet_config(payload: dict) -> WorksheetConfig:
    return WorksheetConfig(
        config_type=str(payload.get("config_type", "")),
        reference=ConfigResolvedLabelsReference(
            tenant=ConfigReferenceTenant(
                tenant_id=str(payload.get("tenant_id", "")),
                target_id=str(payload.get("target_id", "")),
            )
        ),
        inherit_reference=bool(payload.get("inherit_reference", False)),
    )


def raceeng_worksheet_row_from_entry(entry: dict) -> WorksheetRow:
    configs = [raceeng_deserialize_worksheet_config(cfg) for cfg in list(entry.get("configs") or [])]
    study_id = str(entry.get("study_id", ""))
    tenant_id = str(entry.get("tenant_id", ""))
    return WorksheetRow(
        name=str(entry.get("row_name", "")),
        configs=configs,
        study=WorksheetRowStudy(
            reference=StudyResolvedLabelsReference(
                tenant_id=tenant_id,
                target_id=study_id,
            )
        ),
    )


def raceeng_load_worksheet_manifest(campaign_state_stem: str) -> dict[str, dict]:
    path = Path(raceeng_campaign_state_paths(campaign_state_stem)["worksheet_manifest"])
    if not path.exists():
        return {}
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}
    rows = dict((data or {}).get("rows") or {})
    out: dict[str, dict] = {}
    for row_name, entry in rows.items():
        if isinstance(entry, dict):
            out[str(row_name)] = dict(entry)
    return out


def raceeng_write_worksheet_manifest(campaign_state_stem: str, rows: dict[str, dict]):
    path = Path(raceeng_campaign_state_paths(campaign_state_stem)["worksheet_manifest"])
    ordered = {
        name: rows[name]
        for name in sorted(
            rows,
            key=lambda key: (
                int((rows.get(key) or {}).get("order_index", 10**9)),
                str(key),
            ),
        )
    }
    payload = {
        "timestamp_utc": datetime.now(UTC).isoformat(),
        "row_count": int(len(ordered)),
        "rows": ordered,
    }
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def raceeng_record_reconcile_event(runtime: RaceEngRuntime | None, payload: dict):
    if runtime is None:
        return
    row = dict(payload)
    row.setdefault("timestamp_utc", datetime.now(UTC).isoformat())
    runtime.worksheet_reconcile_rows.append(row)
    if runtime.campaign_state_stem:
        pd.DataFrame(runtime.worksheet_reconcile_rows).to_csv(
            raceeng_campaign_state_paths(runtime.campaign_state_stem)["worksheet_reconcile_trace"],
            index=False,
        )


def raceeng_runtime_ensure_manifest_loaded(runtime: RaceEngRuntime):
    if runtime is None or not runtime.campaign_state_stem:
        return
    if runtime.worksheet_row_manifest:
        runtime.worksheet_manifest_order_next = max(
            int(runtime.worksheet_manifest_order_next),
            1 + max(
                [int((entry or {}).get("order_index", -1)) for entry in runtime.worksheet_row_manifest.values()],
                default=-1,
            ),
        )
        return
    runtime.worksheet_row_manifest = raceeng_load_worksheet_manifest(runtime.campaign_state_stem)
    runtime.worksheet_manifest_order_next = 1 + max(
        [int((entry or {}).get("order_index", -1)) for entry in runtime.worksheet_row_manifest.values()],
        default=-1,
    )


async def raceeng_flush_worksheet_rows(
    runtime: RaceEngRuntime,
    *,
    force: bool = False,
    reason: str = "flush",
) -> int:
    if runtime is None or runtime.session is None:
        return 0
    tenant_id = str(runtime.worksheet_tenant_id or "")
    worksheet_id = str(runtime.worksheet_id or "")
    if not tenant_id or not worksheet_id:
        return 0
    raceeng_runtime_ensure_manifest_loaded(runtime)
    async with runtime.worksheet_writer_lock:
        pending_names = list(dict.fromkeys(runtime.worksheet_row_queue))
        if not pending_names and not force:
            return 0
        worksheet_result = await raceeng_call_with_retry(
            runtime,
            f"worksheet_get:{worksheet_id}:writer",
            lambda s: WorksheetApi(s.async_client).worksheet_get_worksheet(tenant_id, worksheet_id),
        )
        worksheet = worksheet_result.worksheet
        remote_rows = list(worksheet.outline.rows or [])
        remote_by_name = {str(row.name or ""): row for row in remote_rows}
        runtime.worksheet_known_rows.update({name: {"present": True} for name in remote_by_name if name})

        combined_rows = list(remote_rows)
        appended = 0
        reinserted = 0
        manifest_names = sorted(
            runtime.worksheet_row_manifest,
            key=lambda key: (
                int((runtime.worksheet_row_manifest.get(key) or {}).get("order_index", 10**9)),
                str(key),
            ),
        )
        for row_name in manifest_names:
            if row_name in remote_by_name:
                continue
            entry = dict(runtime.worksheet_row_manifest[row_name])
            combined_rows.append(raceeng_worksheet_row_from_entry(entry))
            remote_by_name[row_name] = combined_rows[-1]
            appended += 1
            if bool(entry.get("ever_verified_present", False)):
                reinserted += 1
                runtime.worksheet_rows_reinserted += 1

        if appended > 0:
            request = WorksheetPutWorksheetRequest(
                name=worksheet.name,
                properties=worksheet.properties,
                outline=worksheet.outline,
                notes=worksheet.notes,
            )
            request.outline.rows = combined_rows
            await raceeng_call_with_retry(
                runtime,
                f"worksheet_put:{worksheet_id}:writer",
                lambda s: WorksheetApi(s.async_client).worksheet_put_worksheet(tenant_id, worksheet_id, request),
            )

        verify_result = await raceeng_call_with_retry(
            runtime,
            f"worksheet_get:{worksheet_id}:writer_verify",
            lambda s: WorksheetApi(s.async_client).worksheet_get_worksheet(tenant_id, worksheet_id),
        )
        verified_names = {
            str(row.name or "")
            for row in list(verify_result.worksheet.outline.rows or [])
            if str(row.name or "")
        }
        missing = [
            row_name
            for row_name in runtime.worksheet_row_manifest
            if row_name not in verified_names
        ]
        if missing:
            runtime.worksheet_reconcile_retry_count += 1
            for row_name in missing:
                entry = dict(runtime.worksheet_row_manifest.get(row_name) or {})
                entry["verify_failures"] = int(entry.get("verify_failures", 0)) + 1
                entry["last_missing_utc"] = datetime.now(UTC).isoformat()
                runtime.worksheet_row_manifest[row_name] = entry
            if runtime.campaign_state_stem:
                raceeng_write_worksheet_manifest(runtime.campaign_state_stem, runtime.worksheet_row_manifest)
            raceeng_record_reconcile_event(
                runtime,
                {
                    "reason": str(reason),
                    "action": "verify_missing",
                    "missing_count": int(len(missing)),
                    "pending_count": int(len(runtime.worksheet_row_queue)),
                    "appended_count": int(appended),
                },
            )
            raise RuntimeError(
                f"Worksheet reconciliation missing {len(missing)} rows after PUT for worksheet {worksheet_id}"
            )

        for row_name, entry in list(runtime.worksheet_row_manifest.items()):
            entry = dict(entry)
            entry["ever_verified_present"] = True
            entry["last_verified_utc"] = datetime.now(UTC).isoformat()
            runtime.worksheet_row_manifest[row_name] = entry
        if runtime.campaign_state_stem:
            raceeng_write_worksheet_manifest(runtime.campaign_state_stem, runtime.worksheet_row_manifest)
        if pending_names:
            runtime.worksheet_row_queue = [name for name in runtime.worksheet_row_queue if name not in set(pending_names)]
            runtime.worksheet_row_pending_names -= set(pending_names)
        raceeng_record_reconcile_event(
            runtime,
            {
                "reason": str(reason),
                "action": "flush_ok",
                "pending_count": int(len(pending_names)),
                "appended_count": int(appended),
                "reinserted_count": int(reinserted),
                "manifest_row_count": int(len(runtime.worksheet_row_manifest)),
            },
        )
        return int(appended)


async def raceeng_worksheet_writer_loop(runtime: RaceEngRuntime):
    while True:
        try:
            if runtime.worksheet_writer_shutdown:
                if runtime.worksheet_row_queue:
                    await raceeng_flush_worksheet_rows(runtime, force=True, reason="shutdown")
                break
            if not runtime.worksheet_row_queue:
                runtime.worksheet_writer_event.clear()
                await runtime.worksheet_writer_event.wait()
                continue
            runtime.worksheet_writer_event.clear()
            if len(runtime.worksheet_row_queue) < int(max(1, runtime.worksheet_row_batch_size)):
                try:
                    await asyncio.wait_for(
                        runtime.worksheet_writer_event.wait(),
                        timeout=float(max(0.2, runtime.worksheet_row_batch_seconds)),
                    )
                    continue
                except asyncio.TimeoutError:
                    pass
            await raceeng_flush_worksheet_rows(runtime, force=False, reason="batch")
        except asyncio.CancelledError:
            if runtime.worksheet_row_queue:
                try:
                    await raceeng_flush_worksheet_rows(runtime, force=True, reason="cancel")
                except Exception:
                    pass
            raise
        except Exception as ex:
            runtime.worksheet_reconcile_retry_count += 1
            raceeng_record_reconcile_event(
                runtime,
                {
                    "reason": "writer_exception",
                    "action": "retry",
                    "error_type": type(ex).__name__,
                    "message": str(ex),
                    "pending_count": int(len(runtime.worksheet_row_queue)),
                },
            )
            await asyncio.sleep(2.0)


async def raceeng_ensure_worksheet_writer(
    runtime: RaceEngRuntime,
    tenant_id: str,
    worksheet_id: str,
    *,
    worksheet_name: str | None = None,
    worksheet_number: int | None = None,
    batch_seconds: float | None = None,
    batch_size: int | None = None,
):
    if runtime is None:
        return
    raceeng_runtime_ensure_manifest_loaded(runtime)
    runtime.worksheet_tenant_id = str(tenant_id)
    runtime.worksheet_id = str(worksheet_id)
    if worksheet_name is not None:
        runtime.worksheet_name = str(worksheet_name)
    if worksheet_number is not None:
        runtime.worksheet_number = int(worksheet_number)
    if batch_seconds is not None:
        runtime.worksheet_row_batch_seconds = float(max(0.2, batch_seconds))
    if batch_size is not None:
        runtime.worksheet_row_batch_size = int(max(1, batch_size))
    if runtime.worksheet_writer_task is None or runtime.worksheet_writer_task.done():
        runtime.worksheet_writer_shutdown = False
        runtime.worksheet_writer_task = asyncio.create_task(raceeng_worksheet_writer_loop(runtime))


async def raceeng_enqueue_worksheet_row(
    runtime: RaceEngRuntime,
    tenant_id: str,
    worksheet_id: str,
    row_name: str,
    worksheet_configs: list[WorksheetConfig],
    study_id: str,
) -> None:
    await raceeng_ensure_worksheet_writer(runtime, tenant_id, worksheet_id)
    logical_row_name = raceeng_strip_launch_suffix(row_name)
    launch_match = _RACEENG_LAUNCH_SUFFIX_RE.match(str(row_name))
    launch_id = "" if launch_match is None else str(launch_match.group("launch"))
    if row_name not in runtime.worksheet_row_manifest:
        runtime.worksheet_rows_written += 1
    entry = dict(runtime.worksheet_row_manifest.get(row_name) or {})
    if "order_index" not in entry:
        entry["order_index"] = int(runtime.worksheet_manifest_order_next)
        runtime.worksheet_manifest_order_next += 1
    entry.update(
        {
            "row_name": str(row_name),
            "logical_row_name": str(logical_row_name),
            "launch_id": str(launch_id),
            "study_id": str(study_id),
            "tenant_id": str(tenant_id),
            "worksheet_id": str(worksheet_id),
            "configs": [raceeng_serialize_worksheet_config(cfg) for cfg in worksheet_configs],
            "updated_utc": datetime.now(UTC).isoformat(),
            "ever_verified_present": bool(entry.get("ever_verified_present", False)),
        }
    )
    runtime.worksheet_row_manifest[str(row_name)] = entry
    if runtime.campaign_state_stem:
        paths = raceeng_campaign_state_paths(runtime.campaign_state_stem)
        raceeng_write_worksheet_manifest(runtime.campaign_state_stem, runtime.worksheet_row_manifest)
        raceeng_append_jsonl(
            paths["worksheet_rows"],
            {
                "timestamp_utc": datetime.now(UTC).isoformat(),
                "event": "enqueue",
                "row_name": str(row_name),
                "logical_row_name": str(logical_row_name),
                "launch_id": str(launch_id),
                "study_id": str(study_id),
                "worksheet_id": str(worksheet_id),
            },
        )
    if row_name not in runtime.worksheet_row_pending_names:
        runtime.worksheet_row_pending_names.add(str(row_name))
        runtime.worksheet_row_queue.append(str(row_name))
    runtime.worksheet_writer_event.set()
    if len(runtime.worksheet_row_queue) >= int(max(1, runtime.worksheet_row_batch_size)):
        await raceeng_flush_worksheet_rows(runtime, force=False, reason="enqueue_threshold")


async def raceeng_reconcile_worksheet_manifest(runtime: RaceEngRuntime | None) -> int:
    if runtime is None or not runtime.worksheet_row_manifest:
        return 0
    await raceeng_ensure_worksheet_writer(
        runtime,
        str(runtime.worksheet_tenant_id or ""),
        str(runtime.worksheet_id or ""),
        worksheet_name=runtime.worksheet_name,
        worksheet_number=runtime.worksheet_number,
    )
    return await raceeng_flush_worksheet_rows(runtime, force=True, reason="startup_reconcile")


async def raceeng_shutdown_worksheet_writer(runtime: RaceEngRuntime | None):
    if runtime is None:
        return
    task = runtime.worksheet_writer_task
    if task is None:
        return
    runtime.worksheet_writer_shutdown = True
    runtime.worksheet_writer_event.set()
    try:
        await task
    except asyncio.CancelledError:
        pass
    except Exception:
        pass
    runtime.worksheet_writer_task = None


def raceeng_write_pending_studies_file(campaign_state_stem: str, pending_studies: dict[str, dict]):
    paths = raceeng_campaign_state_paths(campaign_state_stem)
    pending_rows = sorted(
        [{k: v for k, v in dict(entry).items()} for entry in pending_studies.values()],
        key=lambda row: (str(row.get("logical_row_name", "")), str(row.get("launch_id", ""))),
    )
    Path(paths["pending"]).write_text(
        json.dumps(
            {
                "timestamp_utc": datetime.now(UTC).isoformat(),
                "pending_studies": pending_rows,
            },
            indent=2,
        ),
        encoding="utf-8",
    )


def raceeng_register_study_launch(runtime: RaceEngRuntime | None, entry: dict):
    if runtime is None or not runtime.campaign_state_stem:
        return
    payload = {k: v for k, v in dict(entry).items()}
    payload.setdefault("timestamp_utc", datetime.now(UTC).isoformat())
    payload.setdefault("event", "launch")
    runtime.pending_studies[str(payload["launch_id"])] = payload
    raceeng_append_jsonl(raceeng_campaign_state_paths(runtime.campaign_state_stem)["launches"], payload)
    raceeng_write_pending_studies_file(runtime.campaign_state_stem, runtime.pending_studies)


def raceeng_mark_study_status(
    runtime: RaceEngRuntime | None,
    launch_id: str,
    status: str,
    *,
    drop_from_pending: bool = False,
    **updates,
):
    if runtime is None or not runtime.campaign_state_stem:
        return
    key = str(launch_id)
    existing = dict(runtime.pending_studies.get(key) or {"launch_id": key})
    existing.update({k: v for k, v in updates.items()})
    existing["status"] = str(status)
    existing["updated_utc"] = datetime.now(UTC).isoformat()
    event_payload = dict(existing)
    event_payload["event"] = "status"
    raceeng_append_jsonl(raceeng_campaign_state_paths(runtime.campaign_state_stem)["launches"], event_payload)
    if drop_from_pending:
        runtime.pending_studies.pop(key, None)
    else:
        runtime.pending_studies[key] = existing
    raceeng_write_pending_studies_file(runtime.campaign_state_stem, runtime.pending_studies)


def raceeng_fit_surrogates_from_history(
    specs: list["RaceEngParameter"],
    history_X: list[list[float]],
    history_obj: list[float],
    history_cons: dict[str, list[float]],
) -> tuple[dict | None, dict[str, dict | None]]:
    x_arr = np.array(history_X, dtype=float) if history_X else np.zeros((0, len(specs)), dtype=float)
    y_obj_arr = np.array(history_obj, dtype=float) if history_obj else np.zeros(0, dtype=float)
    obj_model = raceeng_fit_ridge_linear(x_arr, y_obj_arr)
    constraint_models = {
        c.name: raceeng_fit_ridge_linear(x_arr, np.array(history_cons.get(c.name, []), dtype=float))
        for c in RACEENG_CONSTRAINTS
    }
    return obj_model, constraint_models


def raceeng_load_campaign_state(campaign_state_stem: str) -> dict | None:
    paths = raceeng_campaign_state_paths(campaign_state_stem)
    state_path = Path(paths["json"])
    if not state_path.exists():
        return None
    data = json.loads(state_path.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        return None

    history_path = Path(paths["history"])
    history_X: list[list[float]] = []
    history_obj: list[float] = []
    history_cons: dict[str, list[float]] = {c.name: [] for c in RACEENG_CONSTRAINTS}
    if history_path.exists():
        with np.load(history_path, allow_pickle=False) as npz:
            if "history_X" in npz:
                history_X = np.array(npz["history_X"], dtype=float).tolist()
            if "history_obj" in npz:
                history_obj = np.array(npz["history_obj"], dtype=float).tolist()
            for c in RACEENG_CONSTRAINTS:
                key = f"history_cons__{c.name}"
                if key in npz:
                    history_cons[c.name] = np.array(npz[key], dtype=float).tolist()
    pending_path = Path(paths["pending"])
    pending_studies: dict[str, dict] = {}
    if pending_path.exists():
        try:
            pending_doc = json.loads(pending_path.read_text(encoding="utf-8"))
            for row in list((pending_doc or {}).get("pending_studies") or []):
                if isinstance(row, dict) and row.get("launch_id"):
                    pending_studies[str(row["launch_id"])] = dict(row)
        except Exception:
            pending_studies = {}
    data["history_X"] = history_X
    data["history_obj"] = history_obj
    data["history_cons"] = history_cons
    data["pending_studies"] = pending_studies
    data["_campaign_paths"] = paths
    return data


def raceeng_write_campaign_state(
    campaign_state_stem: str,
    state_payload: dict,
    history_X: list[list[float]],
    history_obj: list[float],
    history_cons: dict[str, list[float]],
):
    paths = raceeng_campaign_state_paths(campaign_state_stem)
    Path(paths["json"]).write_text(json.dumps(state_payload, indent=2), encoding="utf-8")
    save_payload: dict[str, np.ndarray] = {
        "history_X": np.array(history_X, dtype=float) if history_X else np.zeros((0, 0), dtype=float),
        "history_obj": np.array(history_obj, dtype=float) if history_obj else np.zeros(0, dtype=float),
    }
    for c in RACEENG_CONSTRAINTS:
        save_payload[f"history_cons__{c.name}"] = np.array(history_cons.get(c.name, []), dtype=float)
    np.savez_compressed(paths["history"], **save_payload)
    raceeng_write_pending_studies_file(
        campaign_state_stem,
        {
            str(row.get("launch_id")): dict(row)
            for row in list(state_payload.get("pending_studies") or [])
            if isinstance(row, dict) and row.get("launch_id")
        },
    )


def raceeng_vector_distance(
    specs: list["RaceEngParameter"],
    vector_a: dict[str, float] | None,
    vector_b: dict[str, float] | None,
) -> float:
    if vector_a is None or vector_b is None or not specs:
        return float("inf")
    diffs = []
    for spec in specs:
        scale = float(max(1e-12, spec.delta))
        va = float((vector_a or {}).get(spec.name, float("nan")))
        vb = float((vector_b or {}).get(spec.name, float("nan")))
        if not math.isfinite(va) or not math.isfinite(vb):
            return float("inf")
        diffs.append(((va - vb) / scale) ** 2)
    if not diffs:
        return float("inf")
    return float(math.sqrt(float(np.mean(np.array(diffs, dtype=float)))))


def raceeng_make_archive_entry(
    *,
    specs: list["RaceEngParameter"],
    vector: dict[str, float],
    metrics: dict[str, float],
    source_phase: str,
    source_iteration: int,
    source_label: str,
    verified: bool,
    last_verified_median: float | None = None,
) -> dict | None:
    lap = float((metrics or {}).get(RACEENG_OBJECTIVE, float("nan")))
    if (not math.isfinite(lap)) or (not raceeng_is_legal(metrics)):
        return None
    signature = ",".join(
        f"{float(vector.get(spec.name, float('nan'))):.8f}"
        for spec in specs
    )
    return {
        "lap": float(lap),
        "vector": {str(k): float(v) for k, v in dict(vector).items()},
        "metrics": {str(k): float(v) for k, v in dict(metrics).items()},
        "source_phase": str(source_phase),
        "source_iteration": int(source_iteration),
        "source_label": str(source_label),
        "verified": bool(verified),
        "last_verified_median": float(lap if last_verified_median is None and verified else (last_verified_median or float("nan"))),
        "signature": str(signature),
        "updated_utc": datetime.now(UTC).isoformat(),
    }


def raceeng_update_elite_archive(
    elite_archive: list[dict],
    entry: dict | None,
    *,
    specs: list["RaceEngParameter"],
    max_size: int = 8,
    distance_threshold: float = 0.35,
) -> list[dict]:
    if not isinstance(entry, dict):
        return list(elite_archive or [])
    updated = [dict(row) for row in list(elite_archive or []) if isinstance(row, dict)]
    candidate_vector = dict(entry.get("vector") or {})
    candidate_lap = float(entry.get("lap", float("nan")))
    if not math.isfinite(candidate_lap):
        return updated
    replaced = False
    for idx, existing in enumerate(updated):
        dist = raceeng_vector_distance(specs, candidate_vector, dict(existing.get("vector") or {}))
        if dist > float(distance_threshold):
            continue
        existing_lap = float(existing.get("lap", float("nan")))
        existing_verified = bool(existing.get("verified", False))
        candidate_verified = bool(entry.get("verified", False))
        should_replace = bool(
            (candidate_verified and not existing_verified)
            or (candidate_verified == existing_verified and candidate_lap < existing_lap)
        )
        if should_replace:
            merged = dict(existing)
            merged.update(dict(entry))
            if candidate_verified:
                merged["verified"] = True
                merged["last_verified_median"] = float(
                    entry.get("last_verified_median", entry.get("lap", float("nan")))
                )
            updated[idx] = merged
        replaced = True
        break
    if not replaced:
        updated.append(dict(entry))
    updated.sort(
        key=lambda row: (
            float(row.get("lap", float("inf"))),
            0 if bool(row.get("verified", False)) else 1,
            int(row.get("source_iteration", 10**9)),
        )
    )
    return updated[: int(max(1, max_size))]


def raceeng_best_archive_entry(
    elite_archive: list[dict],
    *,
    require_unverified: bool = False,
) -> dict | None:
    rows = [dict(row) for row in list(elite_archive or []) if isinstance(row, dict)]
    if require_unverified:
        rows = [row for row in rows if not bool(row.get("verified", False))]
    rows = [
        row
        for row in rows
        if math.isfinite(float(row.get("lap", float("nan"))))
    ]
    if not rows:
        return None
    rows.sort(key=lambda row: float(row.get("lap", float("inf"))))
    return dict(rows[0])


def raceeng_best_archive_legal_lap(elite_archive: list[dict]) -> float:
    entry = raceeng_best_archive_entry(elite_archive, require_unverified=False)
    return float(entry.get("lap", float("nan"))) if isinstance(entry, dict) else float("nan")


def raceeng_has_verification_debt(
    elite_archive: list[dict],
    best_verified_lap: float,
    threshold: float,
) -> bool:
    challenger = raceeng_best_archive_entry(elite_archive, require_unverified=True)
    if challenger is None:
        return False
    challenger_lap = float(challenger.get("lap", float("nan")))
    if not math.isfinite(challenger_lap):
        return False
    if not math.isfinite(float(best_verified_lap)):
        return True
    return bool(challenger_lap + float(threshold) < float(best_verified_lap))


async def create_user_config_from_payload(
    session=None,
    config_type: str = "",
    payload: dict | None = None,
    name: str = "",
    runtime: RaceEngRuntime | None = None,
) -> str:
    payload = dict(payload or {})

    async def _create_config(active_session):
        return await canopy.create_config(
            active_session,
            config_type,
            name,
            payload["config"],
            properties=payload.get("customProperties") or {},
            notes=sanitize_notes(payload.get("notes")),
            sim_version=payload.get("simVersion"),
        )

    if runtime is not None:
        return await raceeng_call_with_retry(runtime, f"create_config:{config_type}:{name}", _create_config)
    return await _create_config(session)


@dataclass(frozen=True)
class RaceEngConstraint:
    name: str
    sense: str  # '<=' or '>='
    bound: float
    description: str


@dataclass
class RaceEngParameter:
    name: str
    lower: float
    upper: float
    delta: float
    get_value: Callable[[dict], float]
    apply_value: Callable[[dict, float], None]
    build_sub_sweeps: Callable[[dict, float, float], list[tuple[str, list[float]]]]


RACEENG_CONSTRAINTS = [
    RaceEngConstraint("RaceEng_EBottoming", "<=", 0.5, "Bottoming energy"),
    RaceEngConstraint("RaceEng_UndersteerT10Entry", ">=", -4e-3, "Low-speed entry stability"),
    RaceEngConstraint("RaceEng_UndersteerT9", ">=", -1.5e-3, "High-speed stability"),
    RaceEngConstraint("RaceEng_aCamberFEOS", ">=", -3.5, "Front EOS camber"),
    RaceEngConstraint("RaceEng_aCamberREOS", ">=", -1.5, "Rear EOS camber"),
    RaceEngConstraint("RaceEng_kHeaveF", "<=", 850000.0, "Front heave wheel rate (N/m)"),
    RaceEngConstraint("RaceEng_kHeaveR", "<=", 400000.0, "Rear heave wheel rate (N/m)"),
    RaceEngConstraint("RaceEng_kRoll", "<=", 350.0, "Roll stiffness"),
    RaceEngConstraint("RaceEng_rBrakeBalHighPressure", ">=", 0.60, "High-pressure brake balance"),
]

RACEENG_OBJECTIVE = "tLapTotal"
RACEENG_METRICS = [RACEENG_OBJECTIVE, *[c.name for c in RACEENG_CONSTRAINTS]]


RACEENG_SOFT_CONSTRAINT_SLACK = {
    "RaceEng_EBottoming": 0.30,
    "RaceEng_UndersteerT10Entry": 0.0015,
    "RaceEng_UndersteerT9": 0.0010,
    "RaceEng_aCamberFEOS": 0.06,
    "RaceEng_aCamberREOS": 0.05,
    "RaceEng_kHeaveF": 150000.0,
    "RaceEng_kHeaveR": 70000.0,
    "RaceEng_kRoll": 20.0,
    "RaceEng_rBrakeBalHighPressure": 0.03,
}


def _raceeng_path_tokens(path: str) -> list[tuple[str, int | None]]:
    tokens = []
    for part in str(path).split("."):
        m = re.fullmatch(r"([^\[\]]+)(?:\[(\d+)\])?", part)
        if m is None:
            raise ValueError(f"Unsupported path token '{part}' in '{path}'")
        tokens.append((m.group(1), int(m.group(2)) if m.group(2) else None))
    return tokens


def raceeng_get_path_value(container: dict, path: str):
    cur = container
    for key, idx in _raceeng_path_tokens(path):
        if not isinstance(cur, dict):
            raise TypeError(f"Expected dict while traversing {path}, got {type(cur).__name__}")
        cur = cur[key]
        if idx is not None:
            if not isinstance(cur, list):
                raise TypeError(f"Expected list at {key}[{idx}] in {path}")
            cur = cur[idx]
    return cur


def raceeng_set_path_value(container: dict, path: str, value):
    cur = container
    tokens = _raceeng_path_tokens(path)
    for i, (key, idx) in enumerate(tokens):
        last = i == len(tokens) - 1
        if key not in cur:
            raise KeyError(f"Missing key '{key}' in {path}")
        if idx is None:
            if last:
                cur[key] = value
                return
            cur = cur[key]
            if not isinstance(cur, dict):
                raise TypeError(f"Expected dict after {key} in path {path}")
        else:
            seq = cur[key]
            if not isinstance(seq, list):
                raise TypeError(f"Expected list at {key}[{idx}] in path {path}")
            if last:
                seq[idx] = value
                return
            cur = seq[idx]
            if not isinstance(cur, dict):
                raise TypeError(f"Expected dict after {key}[{idx}] in path {path}")
    raise RuntimeError(f"Could not assign {path}")


def _raceeng_clamp(v: float, lo: float, hi: float) -> float:
    return float(max(lo, min(hi, v)))


def _raceeng_exploration_path(path: str) -> str:
    return path if str(path).startswith("car.") else f"car.{path}"


def _raceeng_strip_car_prefix(path: str) -> str:
    p = str(path)
    return p[4:] if p.startswith("car.") else p


def _make_simple_path_parameter(
    name: str,
    path: str,
    lower: float,
    upper: float,
    delta: float,
    cast_type: type = float,
) -> RaceEngParameter:
    def _get(car_cfg: dict) -> float:
        return float(raceeng_get_path_value(car_cfg, path))

    def _apply(car_cfg: dict, value: float):
        v = cast_type(round(value)) if cast_type is int else float(value)
        raceeng_set_path_value(car_cfg, path, v)

    def _sub_sweeps(_car_cfg: dict, low: float, high: float) -> list[tuple[str, list[float]]]:
        lo = cast_type(round(low)) if cast_type is int else float(low)
        hi = cast_type(round(high)) if cast_type is int else float(high)
        return [(_raceeng_exploration_path(path), [lo, hi])]

    return RaceEngParameter(
        name=name,
        lower=float(lower),
        upper=float(upper),
        delta=float(delta),
        get_value=_get,
        apply_value=_apply,
        build_sub_sweeps=_sub_sweeps,
    )


def build_raceeng_exploration(
    sub_sweeps: list[tuple[str, list[float]]],
    *,
    value_type: str = "absolute",
) -> dict:
    if not sub_sweeps:
        raise ValueError("sub_sweeps must not be empty")
    n_points = len(sub_sweeps[0][1])
    if n_points < 2:
        raise ValueError("Need at least two sweep points")
    for path, values in sub_sweeps:
        if len(values) != n_points:
            raise ValueError(f"Mismatched point count for {path}")
    if value_type not in {"absolute", "additive"}:
        raise ValueError(f"Unsupported exploration value_type: {value_type}")
    return {
        "config": {
            "design": {
                "name": "Star",
                "sweeps": [
                    {
                        "dimensionType": "enumeration",
                        "numberOfPoints": int(n_points),
                        "parallelSubSweeps": [
                            {
                                "parameterPath": path,
                                "source": {
                                    "name": "Numerical values",
                                    "valueType": str(value_type),
                                    "values": [float(v) for v in values],
                                },
                            }
                            for path, values in sub_sweeps
                        ],
                    }
                ],
            }
        }
    }


def extract_raceeng_metrics(job) -> dict[str, float]:
    scalar = getattr(job, "scalar_data", None) or {}
    return {
        name: float(scalar.get(name)) if scalar.get(name) is not None else float("nan")
        for name in RACEENG_METRICS
    }


def raceeng_is_legal(metrics: dict[str, float], constraints: list[RaceEngConstraint] = RACEENG_CONSTRAINTS) -> bool:
    for c in constraints:
        v = float(metrics.get(c.name, float("nan")))
        if not math.isfinite(v):
            return False
        if c.sense == "<=" and v > c.bound:
            return False
        if c.sense == ">=" and v < c.bound:
            return False
    return True


def raceeng_total_violation(metrics: dict[str, float], constraints: list[RaceEngConstraint] = RACEENG_CONSTRAINTS) -> float:
    total = 0.0
    for c in constraints:
        v = float(metrics.get(c.name, float("nan")))
        if not math.isfinite(v):
            total += 1e6
            continue
        if c.sense == "<=":
            raw = max(0.0, v - c.bound)
        else:
            raw = max(0.0, c.bound - v)
        total += raw / max(1.0, abs(c.bound))
    return float(total)


def raceeng_legality_breakdown(metrics: dict[str, float], constraints: list[RaceEngConstraint] = RACEENG_CONSTRAINTS) -> dict[str, float]:
    out = {}
    for c in constraints:
        v = float(metrics.get(c.name, float("nan")))
        if not math.isfinite(v):
            out[c.name] = float("inf")
        elif c.sense == "<=":
            out[c.name] = max(0.0, v - c.bound)
        else:
            out[c.name] = max(0.0, c.bound - v)
    return out


def apply_raceeng_vector(base_car_payload: dict, specs: list[RaceEngParameter], vector: dict[str, float]) -> dict:
    payload = copy.deepcopy(base_car_payload)
    cfg = payload["config"]
    for spec in specs:
        spec.apply_value(cfg, float(vector[spec.name]))
    return payload


def extract_raceeng_vector(car_payload: dict, specs: list[RaceEngParameter]) -> dict[str, float]:
    cfg = car_payload["config"]
    return {spec.name: float(spec.get_value(cfg)) for spec in specs}


async def raceeng_materialize_vector_car(
    session,
    base_car_payload: dict,
    specs: list[RaceEngParameter],
    vector: dict[str, float],
    name_prefix: str,
    runtime: RaceEngRuntime | None = None,
) -> tuple[str, dict]:
    payload = apply_raceeng_vector(base_car_payload, specs, vector)
    car_id = await create_user_config_from_payload(
        session=session,
        config_type="car",
        payload=payload,
        name=f"{name_prefix}-car-{uuid4().hex[:8]}",
        runtime=runtime,
    )
    return car_id, payload


@dataclass
class RaceEngLoadedStudy:
    jobs: list[Any]
    metadata_result: Any | None = None
    study_document: Any | None = None


def raceeng_serialize_list_filter(session: canopy.Session, list_filter: Any) -> str:
    return json.dumps(session.sync_client.sanitize_for_serialization(list_filter))


async def raceeng_list_study_job_documents(
    session: canopy.Session,
    tenant_id: str,
    study_id: str,
    *,
    runtime: RaceEngRuntime | None = None,
    items_per_page: int = 250,
) -> list[Any]:
    docs: list[Any] = []
    continuation_token: str | None = None
    page_index = 0
    while True:
        list_filter = canopy.openapi.ListFilter(
            items_per_page=int(max(1, items_per_page)),
            continuation_token=continuation_token,
        )
        filter_json = raceeng_serialize_list_filter(session, list_filter)
        if runtime is not None:
            result = await raceeng_call_with_retry(
                runtime,
                f"study_jobs:{study_id}:page{page_index}",
                lambda s, filter_json=filter_json: StudyApi(s.async_client).study_get_study_jobs(
                    tenant_id,
                    study_id,
                    filter=filter_json,
                ),
            )
        else:
            session.authentication.authenticate()
            result = await StudyApi(session.async_client).study_get_study_jobs(
                tenant_id,
                study_id,
                filter=filter_json,
            )
        query_results = getattr(result, "query_results", None)
        docs.extend(list(getattr(query_results, "documents", None) or []))
        if not bool(getattr(query_results, "has_more_results", False)):
            break
        next_token = str(getattr(query_results, "continuation_token", "") or "")
        if not next_token:
            raise RuntimeError(
                f"Study jobs for {study_id} reported more results but did not provide a continuation token."
            )
        continuation_token = next_token
        page_index += 1
    return docs


async def raceeng_load_study_with_retry(
    session,
    study_id: str,
    sim_type: str = "DynamicLap",
    tenant_id: str | None = None,
    max_attempts: int = 6,
    retry_delay_seconds: float = 20.0,
    runtime: RaceEngRuntime | None = None,
):
    async def _load_job(active_session: canopy.Session, resolved_tenant_id: str, job_id: str):
        return await canopy.load_study_job(
            session=active_session,
            study_id=study_id,
            sim_type=sim_type,
            job_index=job_id,
            tenant_id=resolved_tenant_id,
            include_scalar_results=True,
        )

    async def _load(active_session):
        active_session.authentication.authenticate()
        resolved_tenant_id = str(
            tenant_id
            if tenant_id is not None
            else getattr(active_session.authentication, "tenant_id", "")
        )
        if not resolved_tenant_id:
            raise RuntimeError(f"Tenant ID is required to load study {study_id}.")

        active_session.authentication.authenticate()
        study_api = StudyApi(active_session.async_client)
        metadata_result = await study_api.study_get_study_metadata(resolved_tenant_id, study_id)
        study_document = canopy.get_study_document(active_session, metadata_result.study)
        job_documents = await raceeng_list_study_job_documents(
            active_session,
            resolved_tenant_id,
            study_id,
            runtime=runtime,
        )

        semaphore = asyncio.Semaphore(active_session.default_api_concurrency)

        async def _load_one(job_document: Any):
            job_id = str(getattr(job_document, "document_id", "") or "")
            if not job_id:
                raise RuntimeError(f"Study {study_id} returned a job document without a document_id.")
            async with semaphore:
                if runtime is not None:
                    return await raceeng_call_with_retry(
                        runtime,
                        f"load_study_job:{study_id}:{job_id}",
                        lambda s, resolved_tenant_id=resolved_tenant_id, job_id=job_id: _load_job(
                            s,
                            resolved_tenant_id,
                            job_id,
                        ),
                        max_attempts=max_attempts,
                        base_delay_seconds=max(1.0, float(retry_delay_seconds) / 5.0),
                        max_delay_seconds=float(max(retry_delay_seconds, 20.0)),
                    )
                return await _load_job(active_session, resolved_tenant_id, job_id)

        jobs = await asyncio.gather(*[_load_one(job_document) for job_document in job_documents])
        return RaceEngLoadedStudy(
            jobs=list(jobs),
            metadata_result=metadata_result,
            study_document=study_document,
        )

    if runtime is not None:
        return await raceeng_call_with_retry(
            runtime,
            f"load_study:{study_id}",
            _load,
            max_attempts=max_attempts,
            base_delay_seconds=max(1.0, float(retry_delay_seconds) / 5.0),
            max_delay_seconds=float(max(retry_delay_seconds, 20.0)),
        )
    return await _load(session)


async def raceeng_wait_for_study_progress(
    session,
    tenant_id: str,
    study_id: str,
    timeout_seconds: int,
    *,
    min_completed_fraction: float = 1.0,
    min_completed_jobs: int = 0,
    min_succeeded_jobs: int = 0,
    poll_seconds: float = 8.0,
    runtime: RaceEngRuntime | None = None,
) -> dict:
    start = perf_counter()
    while True:
        if runtime is not None:
            meta = await raceeng_call_with_retry(
                runtime,
                f"study_metadata:{study_id}",
                lambda s: StudyApi(s.async_client).study_get_study_metadata(tenant_id, study_id),
            )
            active_session = runtime.session
        else:
            active_session = session
            session.authentication.authenticate()
            study_api = StudyApi(session.async_client)
            meta = await study_api.study_get_study_metadata(tenant_id, study_id)
        study_doc = canopy.get_study_document(active_session, meta.study)
        total = int(getattr(study_doc, "job_count", 0) or 0)
        completed = int(getattr(study_doc, "completed_job_count", 0) or 0)
        succeeded = int(getattr(study_doc, "succeeded_job_count", 0) or 0)
        elapsed = float(perf_counter() - start)
        is_complete = bool(total > 0 and completed >= total)
        min_completed_target = max(int(min_completed_jobs), int(math.ceil(float(min_completed_fraction) * max(1, total))))
        partial_ready = bool(completed >= min_completed_target and succeeded >= int(min_succeeded_jobs))
        timed_out = bool(timeout_seconds > 0 and elapsed >= float(timeout_seconds))
        if is_complete or partial_ready or timed_out:
            return {
                "is_complete": bool(is_complete),
                "partial_ready": bool(partial_ready),
                "timed_out": bool(timed_out),
                "job_count": int(total),
                "completed_job_count": int(completed),
                "succeeded_job_count": int(succeeded),
                "wait_seconds": float(elapsed),
                "succeeded_simulation_count": int(succeeded),
            }
        await asyncio.sleep(float(max(2.0, poll_seconds)))


def _raceeng_worksheet_configs(
    tenant_id: str,
    car_id: str,
    weather_id: str,
    track_id: str,
    user_maths_id: str,
    exploration_id: str | None = None,
) -> list[WorksheetConfig]:
    cfgs = [
        worksheet_config(tenant_id, "car", car_id),
        worksheet_config(tenant_id, "weather", weather_id),
        worksheet_config(tenant_id, "track", track_id),
        worksheet_config(tenant_id, "userMaths", user_maths_id),
    ]
    if exploration_id is not None:
        cfgs.append(worksheet_config(tenant_id, "exploration", exploration_id))
    return cfgs


async def raceeng_run_single_point(
    session,
    tenant_id: str,
    worksheet_id: str,
    row_name: str,
    static_config_ids: dict[str, str],
    car_payload: dict,
    timeout_seconds: int,
    runtime: RaceEngRuntime | None = None,
    phase: str = "single",
    iteration: int | None = None,
) -> dict:
    max_attempts = 3
    last_ex = None

    for attempt in range(1, max_attempts + 1):
        try:
            suffix = uuid4().hex[:8]
            attempt_row_name = row_name if attempt == 1 else f"{row_name}-retry{attempt}"
            launch_id = raceeng_new_launch_id()
            physical_row_name = raceeng_make_physical_row_name(attempt_row_name, launch_id)
            car_id = await create_user_config_from_payload(
                session=session,
                config_type="car",
                payload=car_payload,
                name=f"{attempt_row_name}-car-{suffix}",
                runtime=runtime,
            )
            if runtime is not None:
                study_id = await raceeng_call_with_retry(
                    runtime,
                    f"create_study:{attempt_row_name}",
                    lambda s: canopy.create_study(
                        s,
                        "dynamicLap",
                        f"{attempt_row_name}-dyn-{suffix}",
                        [
                            car_id,
                            static_config_ids["weather"],
                            static_config_ids["track"],
                            static_config_ids["userMaths"],
                        ],
                    ),
                )
            else:
                study_id = await canopy.create_study(
                    session,
                    "dynamicLap",
                    f"{attempt_row_name}-dyn-{suffix}",
                    [
                        car_id,
                        static_config_ids["weather"],
                        static_config_ids["track"],
                        static_config_ids["userMaths"],
                    ],
                )
            raceeng_register_study_launch(
                runtime,
                {
                    "launch_id": str(launch_id),
                    "logical_row_name": str(attempt_row_name),
                    "physical_row_name": str(physical_row_name),
                    "phase": str(phase),
                    "iteration": None if iteration is None else int(iteration),
                    "kind": "single_point",
                    "status": "launched",
                    "study_id": str(study_id),
                    "exploration_id": None,
                    "car_id": str(car_id),
                    "repeat_count": 1,
                    "point_defs": [],
                },
            )

            await upsert_worksheet_row(
                session=session,
                tenant_id=tenant_id,
                worksheet_id=worksheet_id,
                row_name=physical_row_name,
                worksheet_configs=_raceeng_worksheet_configs(
                    tenant_id=tenant_id,
                    car_id=car_id,
                    weather_id=static_config_ids["weather"],
                    track_id=static_config_ids["track"],
                    user_maths_id=static_config_ids["userMaths"],
                ),
                study_id=study_id,
                runtime=runtime,
            )

            wait_result = await raceeng_wait_for_study_progress(
                session=session,
                tenant_id=tenant_id,
                study_id=study_id,
                timeout_seconds=timeout_seconds,
                min_completed_fraction=1.0,
                min_completed_jobs=1,
                min_succeeded_jobs=1,
                poll_seconds=8.0,
                runtime=runtime,
            )
            if int(wait_result["succeeded_simulation_count"]) == 0:
                raceeng_mark_study_status(runtime, launch_id, "failed", reason="zero_successful_simulations")
                raise RuntimeError(f"Study {study_id} completed with zero successful simulations")

            study = await raceeng_load_study_with_retry(
                session=session,
                study_id=study_id,
                sim_type="DynamicLap",
                runtime=runtime,
            )
            jobs = sorted_jobs_by_index(study.jobs)
            if not jobs:
                raceeng_mark_study_status(runtime, launch_id, "failed", reason="no_jobs_returned")
                raise RuntimeError(f"Study {study_id} returned no jobs")
            duration_seconds = raceeng_estimate_study_duration_seconds(jobs)
            metrics = extract_raceeng_metrics(jobs[0])
            raceeng_mark_study_status(
                runtime,
                launch_id,
                "finalized",
                drop_from_pending=True,
                succeeded_simulation_count=int(wait_result["succeeded_simulation_count"]),
                duration_seconds=float(duration_seconds),
            )
            return {
                "row_name": attempt_row_name,
                "physical_row_name": physical_row_name,
                "launch_id": str(launch_id),
                "study_id": study_id,
                "car_id": car_id,
                "job_id": jobs[0].document.document_id,
                "metrics": metrics,
                "duration_seconds": float(duration_seconds),
                "succeeded_simulation_count": int(wait_result["succeeded_simulation_count"]),
            }
        except Exception as ex:
            last_ex = ex
            msg = str(ex)
            retryable = (
                "zero successful simulations" in msg
                or "DynamicLap_ScalarResults.csv" in msg
                or "The specified blob does not exist" in msg
            )
            if attempt < max_attempts and retryable:
                print(
                    f"Warning: single-point run failed for '{row_name}' on attempt "
                    f"{attempt}/{max_attempts}: {msg}. Retrying..."
                )
                await asyncio.sleep(6.0)
                continue
            raise

    if last_ex is not None:
        raise last_ex
    raise RuntimeError(f"Single-point run failed for '{row_name}'")


def raceeng_median_metrics_from_runs(runs: list[dict]) -> dict[str, float]:
    med: dict[str, float] = {}
    for metric in RACEENG_METRICS:
        vals = []
        for run in runs:
            metrics = dict(run.get("metrics") or {})
            val = float(metrics.get(metric, float("nan")))
            if math.isfinite(val):
                vals.append(val)
        med[metric] = float(statistics.median(vals)) if vals else float("nan")
    return med


async def raceeng_run_single_point_replicates(
    session,
    tenant_id: str,
    worksheet_id: str,
    row_name: str,
    static_config_ids: dict[str, str],
    car_payload: dict,
    timeout_seconds: int,
    repeat_count: int = 3,
    noop_parameter_path: str = "car.chassis.hRideFSetup",
    runtime: RaceEngRuntime | None = None,
    phase: str = "replicate",
    iteration: int | None = None,
    bundle_role: str = "generic",
) -> dict:
    n = int(max(2, repeat_count))
    max_attempts = 3
    print(f"{row_name}: launching no-op exploration replicates ({n} points)")
    noop_local_path = _raceeng_strip_car_prefix(noop_parameter_path)
    last_ex = None

    for attempt in range(1, max_attempts + 1):
        try:
            suffix = uuid4().hex[:8]
            attempt_row_name = row_name if attempt == 1 else f"{row_name}-retry{attempt}"
            launch_id = raceeng_new_launch_id()
            physical_row_name = raceeng_make_physical_row_name(attempt_row_name, launch_id)
            car_id = await create_user_config_from_payload(
                session=session,
                config_type="car",
                payload=car_payload,
                name=f"{attempt_row_name}-car-{suffix}",
                runtime=runtime,
            )
            exploration_payload = build_raceeng_exploration(
                [
                    (
                        _raceeng_exploration_path(noop_local_path),
                        [0.0] * int(n),
                    )
                ],
                value_type="additive",
            )
            exploration_id = await create_user_config_from_payload(
                session=session,
                config_type="exploration",
                payload={"config": exploration_payload["config"]},
                name=f"{attempt_row_name}-noop-exp-{suffix}",
                runtime=runtime,
            )
            if runtime is not None:
                study_id = await raceeng_call_with_retry(
                    runtime,
                    f"create_study:{attempt_row_name}",
                    lambda s: canopy.create_study(
                        s,
                        "dynamicLap",
                        f"{attempt_row_name}-noop-{suffix}",
                        [
                            car_id,
                            static_config_ids["weather"],
                            static_config_ids["track"],
                            static_config_ids["userMaths"],
                            exploration_id,
                        ],
                    ),
                )
            else:
                study_id = await canopy.create_study(
                    session,
                    "dynamicLap",
                    f"{attempt_row_name}-noop-{suffix}",
                    [
                        car_id,
                        static_config_ids["weather"],
                        static_config_ids["track"],
                        static_config_ids["userMaths"],
                        exploration_id,
                    ],
                )
            raceeng_register_study_launch(
                runtime,
                {
                    "launch_id": str(launch_id),
                    "logical_row_name": str(attempt_row_name),
                    "physical_row_name": str(physical_row_name),
                    "phase": str(phase),
                    "iteration": None if iteration is None else int(iteration),
                    "kind": "noop_bundle",
                    "bundle_role": str(bundle_role),
                    "status": "launched",
                    "study_id": str(study_id),
                    "exploration_id": str(exploration_id),
                    "car_id": str(car_id),
                    "repeat_count": int(n),
                    "point_defs": [],
                },
            )
            await upsert_worksheet_row(
                session=session,
                tenant_id=tenant_id,
                worksheet_id=worksheet_id,
                row_name=physical_row_name,
                worksheet_configs=_raceeng_worksheet_configs(
                    tenant_id=tenant_id,
                    car_id=car_id,
                    weather_id=static_config_ids["weather"],
                    track_id=static_config_ids["track"],
                    user_maths_id=static_config_ids["userMaths"],
                    exploration_id=exploration_id,
                ),
                study_id=study_id,
                runtime=runtime,
            )

            wait_result = await raceeng_wait_for_study_progress(
                session=session,
                tenant_id=tenant_id,
                study_id=study_id,
                timeout_seconds=timeout_seconds,
                min_completed_fraction=1.0,
                min_completed_jobs=int(n),
                min_succeeded_jobs=1,
                poll_seconds=8.0,
                runtime=runtime,
            )
            if int(wait_result["succeeded_simulation_count"]) <= 0:
                raceeng_mark_study_status(runtime, launch_id, "failed", reason="zero_successful_simulations")
                raise RuntimeError(f"Study {study_id} completed with zero successful simulations")

            study = await raceeng_load_study_with_retry(
                session=session,
                study_id=study_id,
                sim_type="DynamicLap",
                runtime=runtime,
            )
            jobs = sorted_jobs_by_index(study.jobs)
            if len(jobs) < n:
                raceeng_mark_study_status(runtime, launch_id, "failed", reason="insufficient_jobs", job_count=len(jobs))
                raise RuntimeError(
                    f"Study {study_id} returned only {len(jobs)} jobs for replicate bundle of {n}"
                )
            duration_seconds = raceeng_estimate_study_duration_seconds(jobs)

            runs: list[dict] = []
            laps: list[float] = []
            legal_count = 0
            for i in range(n):
                metrics = extract_raceeng_metrics(jobs[i])
                lap = float(metrics.get(RACEENG_OBJECTIVE, float("nan")))
                is_legal = bool(raceeng_is_legal(metrics))
                lap_txt = "nan" if not math.isfinite(lap) else f"{lap:.6f}"
                print(f"{row_name}: replicate {i+1}/{n} -> tLapTotal={lap_txt}, hard_legal={is_legal}")
                runs.append(
                    {
                        "row_name": attempt_row_name,
                        "study_id": study_id,
                        "exploration_id": exploration_id,
                        "car_id": car_id,
                        "launch_id": str(launch_id),
                        "job_id": jobs[i].document.document_id,
                        "metrics": metrics,
                        "duration_seconds": float(duration_seconds),
                        "succeeded_simulation_count": int(wait_result["succeeded_simulation_count"]),
                    }
                )
                laps.append(float(lap))
                legal_count += int(is_legal)

            median_metrics = raceeng_median_metrics_from_runs(runs)
            median_lap = float(median_metrics.get(RACEENG_OBJECTIVE, float("nan")))
            median_legal = bool(raceeng_is_legal(median_metrics))
            representative_idx = 0
            finite = [(idx, float(lap)) for idx, lap in enumerate(laps) if math.isfinite(float(lap))]
            if finite:
                med = float(statistics.median([lap for _, lap in finite]))
                representative_idx = int(min(finite, key=lambda pair: abs(pair[1] - med))[0])
            raceeng_mark_study_status(
                runtime,
                launch_id,
                "finalized",
                drop_from_pending=True,
                succeeded_simulation_count=int(wait_result["succeeded_simulation_count"]),
                duration_seconds=float(duration_seconds),
                legal_count=int(legal_count),
            )

            return {
                "runs": runs,
                "representative_eval": dict(runs[representative_idx]),
                "median_metrics": dict(median_metrics),
                "median_lap": float(median_lap),
                "median_legal": bool(median_legal),
                "laps": [float(v) for v in laps],
                "legal_count": int(legal_count),
                "repeat_count": int(n),
                "study_id": study_id,
                "exploration_id": exploration_id,
                "logical_row_name": attempt_row_name,
                "physical_row_name": physical_row_name,
                "launch_id": str(launch_id),
            }
        except Exception as ex:
            last_ex = ex
            msg = str(ex)
            retryable = (
                "zero successful simulations" in msg
                or "DynamicLap_ScalarResults.csv" in msg
                or "The specified blob does not exist" in msg
                or "returned only" in msg
            )
            if attempt < max_attempts and retryable:
                print(
                    f"Warning: no-op replicate bundle failed for '{row_name}' on attempt "
                    f"{attempt}/{max_attempts}: {msg}. Retrying..."
                )
                await asyncio.sleep(6.0)
                continue
            raise

    if last_ex is not None:
        raise last_ex
    raise RuntimeError(f"No-op replicate bundle failed for '{row_name}'")


def increment_worksheet_name(name: str) -> str:
    base = str(name or "").strip()
    if not base:
        return "Race Engineering Worksheet 2"
    m = re.search(r"(.*?)(\d+)(\D*)$", base)
    if m is None:
        return f"{base} 2"
    prefix, digits, suffix = m.group(1), m.group(2), m.group(3)
    next_num = int(digits) + 1
    return f"{prefix}{next_num:0{len(digits)}d}{suffix}"


def build_raceeng_parameter_specs(base_car_config: dict) -> list[RaceEngParameter]:
    def _append_vector_components(
        target: list[RaceEngParameter],
        *,
        name_prefix: str,
        path: str,
        span: float,
        delta: float,
    ):
        base_values = raceeng_get_path_value(base_car_config, path)
        if not isinstance(base_values, list):
            raise TypeError(f"Expected list at {path}, got {type(base_values).__name__}")
        for i, base_val in enumerate(base_values):
            center = float(base_val)
            lo = center - float(span)
            hi = center + float(span)
            target.append(
                _make_simple_path_parameter(
                    name=f"{name_prefix}{i}",
                    path=f"{path}[{i}]",
                    lower=lo,
                    upper=hi,
                    delta=float(delta),
                )
            )

    base_diff_outputs = [
        float(v)
        for v in raceeng_get_path_value(base_car_config, "powertrain.rearAxleTransmission.diff.MDiffDemandOutputs")
    ]
    base_bbw_values = [
        float(v)
        for v in raceeng_get_path_value(base_car_config, "control.brakeBalanceOptimisation.mapOutput.values")
    ]

    specs: list[RaceEngParameter] = [
        _make_simple_path_parameter("hRideFSetup", "chassis.hRideFSetup", 0.015, 0.060, 0.0020),
        _make_simple_path_parameter("hRideRSetup", "chassis.hRideRSetup", 0.040, 0.110, 0.0030),
        _make_simple_path_parameter("aFlapF", "aero.flapAngles.aFlapF", math.radians(10.0), math.radians(40.0), math.radians(1.0)),
        _make_simple_path_parameter("kAntiRollBarFront", "suspension.front.internal.antiRollBar.kAntiRollBar", 0.0, 6000.0, 200.0),
        _make_simple_path_parameter("kAntiRollBarRear", "suspension.rear.internal.antiRollBar.kAntiRollBar", 500.0, 5000.0, 200.0),
        _make_simple_path_parameter("kTorsionBarFront", "suspension.front.internal.torsionBar.kTorsionBar", 4500.0, 11000.0, 500.0),
        _make_simple_path_parameter("kSpringRearTri", "suspension.rear.internal.triSpring.kSpring", 120000.0, 320000.0, 10000.0),
        _make_simple_path_parameter("rWeightBalF", "chassis.carRunningMass.rWeightBalF", 0.45, 0.47, 0.0020),
        _make_simple_path_parameter(
            "aToeSetupFront",
            "suspension.front.external.aToeSetupAlignment.aToeSetup",
            -0.020,
            0.020,
            0.0010,
        ),
        _make_simple_path_parameter(
            "aToeSetupRear",
            "suspension.rear.external.aToeSetupAlignment.aToeSetup",
            -0.020,
            0.020,
            0.0010,
        ),
        _make_simple_path_parameter(
            "aCamberSetupFront",
            "suspension.front.external.aCamberSetupAlignment.aCamberSetup",
            math.radians(-4.8),
            math.radians(-0.7),
            math.radians(0.20),
        ),
        _make_simple_path_parameter(
            "aCamberSetupRear",
            "suspension.rear.external.aCamberSetupAlignment.aCamberSetup",
            math.radians(-2.0),
            math.radians(0.3),
            math.radians(0.20),
        ),
        _make_simple_path_parameter(
            "frontBumpStop_pLinear",
            "suspension.front.internal.bumpStop.pLinear",
            200000.0,
            6000000.0,
            200000.0,
        ),
        _make_simple_path_parameter(
            "frontBumpStop_pExponential",
            "suspension.front.internal.bumpStop.pExponential",
            250.0,
            15000.0,
            500.0,
        ),
        _make_simple_path_parameter(
            "frontBumpStop_pExponentialScaling",
            "suspension.front.internal.bumpStop.pExponentialScaling",
            0.5,
            8.0,
            0.25,
        ),
        _make_simple_path_parameter(
            "frontBumpStop_xFreeGap",
            "suspension.front.internal.bumpStop.xFreeGap",
            0.0,
            0.02,
            0.0015,
        ),
        _make_simple_path_parameter(
            "rearTriBumpStop_pLinear",
            "suspension.rear.internal.triBumpStop.pLinear",
            50000.0,
            2000000.0,
            75000.0,
        ),
        _make_simple_path_parameter(
            "rearTriBumpStop_pExponential",
            "suspension.rear.internal.triBumpStop.pExponential",
            200.0,
            12000.0,
            400.0,
        ),
        _make_simple_path_parameter(
            "rearTriBumpStop_xFreeGap",
            "suspension.rear.internal.triBumpStop.xFreeGap",
            0.0,
            0.05,
            0.0020,
        ),
    ]

    # Front and rear external pickup arrays: each component sweeps independently.
    for axle in ("front", "rear"):
        for key in ("rFLWBI", "rRLWBI", "rFUWBI", "rRUWBI", "rPRI"):
            path = f"suspension.{axle}.external.pickUpPts.{key}"
            _append_vector_components(
                specs,
                name_prefix=f"{axle}_{key}_",
                path=path,
                span=0.03,
                delta=0.003,
            )

    # Front and rear internal ARB pickup arrays: each component sweeps independently.
    for axle in ("front", "rear"):
        for key in ("rARBRockerPickup", "rARBPickupInboard"):
            path = f"suspension.{axle}.internal.pickUpPts.{key}"
            _append_vector_components(
                specs,
                name_prefix=f"{axle}_{key}_",
                path=path,
                span=0.03,
                delta=0.003,
            )

    # Rear tri bump-stop pickup coordinates, each component independent.
    _append_vector_components(
        specs,
        name_prefix="rear_rTriBumpStop_",
        path="suspension.rear.internal.pickUpPts.rTriBumpStop",
        span=0.03,
        delta=0.003,
    )

    # Diff demand map: expose all three torque outputs independently.
    for i, base in enumerate(base_diff_outputs):
        upper = max(2500.0, 1.8 * abs(base) + 500.0)
        delta = max(120.0, 0.10 * upper)
        specs.append(
            _make_simple_path_parameter(
                name=f"diffDemandOutput{i}",
                path=f"powertrain.rearAxleTransmission.diff.MDiffDemandOutputs[{i}]",
                lower=0.0,
                upper=upper,
                delta=delta,
            )
        )

    # BBW map: expose each output independently.
    for i, _base in enumerate(base_bbw_values):
        specs.append(
            _make_simple_path_parameter(
                name=f"BBWMapOutput{i}",
                path=f"control.brakeBalanceOptimisation.mapOutput.values[{i}]",
                lower=0.15,
                upper=0.85,
                delta=0.010,
            )
        )

    return specs


def raceeng_fit_linear_gradient(values: list[float], responses: list[float]) -> float:
    finite_pairs = [
        (float(x), float(y))
        for x, y in zip(values, responses)
        if math.isfinite(float(x)) and math.isfinite(float(y))
    ]
    if len(finite_pairs) < 2:
        return 0.0

    buckets: dict[float, list[float]] = {}
    for x, y in finite_pairs:
        key = round(float(x), 12)
        buckets.setdefault(key, []).append(float(y))

    xs = sorted(buckets.keys())
    if len(xs) < 2:
        return 0.0
    x_arr = np.array([float(x) for x in xs], dtype=float)
    y_arr = np.array([float(statistics.median(buckets[x])) for x in xs], dtype=float)
    x_center = x_arr - float(np.mean(x_arr))
    y_center = y_arr - float(np.mean(y_arr))
    denom = float(np.dot(x_center, x_center))
    if denom <= 1e-16:
        return 0.0
    return float(np.dot(x_center, y_center) / denom)


def raceeng_constraint_profile(iteration: int, max_iterations: int) -> dict:
    harden_iters = max(3, int(math.ceil(float(max_iterations) * 0.60)))
    hardness = float(min(1.0, max(0.0, (float(iteration) - 1.0) / float(harden_iters - 1))))
    bounds: dict[str, float] = {}
    rows: list[dict] = []
    for c in RACEENG_CONSTRAINTS:
        base_slack = float(RACEENG_SOFT_CONSTRAINT_SLACK.get(c.name, max(1.0, abs(c.bound) * 0.05)))
        active_slack = float((1.0 - hardness) * base_slack)
        if c.sense == "<=":
            effective = float(c.bound + active_slack)
        else:
            effective = float(c.bound - active_slack)
        bounds[c.name] = effective
        rows.append(
            {
                "constraint": c.name,
                "sense": c.sense,
                "hard_bound": float(c.bound),
                "effective_bound": float(effective),
                "active_slack": float(active_slack),
                "hardness": float(hardness),
            }
        )
    return {"hardness": hardness, "bounds": bounds, "rows": rows}


def raceeng_is_legal_with_bounds(
    metrics: dict[str, float],
    bounds: dict[str, float],
    constraints: list[RaceEngConstraint] = RACEENG_CONSTRAINTS,
) -> bool:
    for c in constraints:
        v = float(metrics.get(c.name, float("nan")))
        if not math.isfinite(v):
            return False
        b = float(bounds.get(c.name, c.bound))
        if c.sense == "<=" and v > b:
            return False
        if c.sense == ">=" and v < b:
            return False
    return True


def raceeng_total_violation_with_bounds(
    metrics: dict[str, float],
    bounds: dict[str, float],
    constraints: list[RaceEngConstraint] = RACEENG_CONSTRAINTS,
) -> float:
    total = 0.0
    for c in constraints:
        v = float(metrics.get(c.name, float("nan")))
        if not math.isfinite(v):
            total += 1e6
            continue
        b = float(bounds.get(c.name, c.bound))
        if c.sense == "<=":
            raw = max(0.0, v - b)
        else:
            raw = max(0.0, b - v)
        total += raw / max(1.0, abs(c.bound))
    return float(total)


def raceeng_estimate_study_duration_seconds(jobs) -> float:
    starts = []
    ends = []
    for job in (jobs or []):
        doc = getattr(job, "document", None)
        if doc is None:
            continue
        cd = getattr(doc, "creation_date", None)
        md = getattr(doc, "modified_date", None)
        if cd is not None:
            starts.append(pd.to_datetime(cd))
        if md is not None:
            ends.append(pd.to_datetime(md))
    if not starts or not ends:
        return float("nan")
    delta = max(ends) - min(starts)
    return float(delta.total_seconds())


def raceeng_fit_ridge_linear(
    X: np.ndarray,
    y: np.ndarray,
    ridge: float = 1e-3,
) -> dict | None:
    if X.size == 0 or y.size == 0:
        return None
    mask = np.isfinite(y)
    mask &= np.all(np.isfinite(X), axis=1)
    Xf = X[mask]
    yf = y[mask]
    if Xf.shape[0] < max(8, Xf.shape[1] + 2):
        return None

    x_mu = np.mean(Xf, axis=0)
    x_sigma = np.std(Xf, axis=0)
    x_sigma = np.where(x_sigma < 1e-9, 1.0, x_sigma)
    z = (Xf - x_mu) / x_sigma
    F = np.hstack([np.ones((z.shape[0], 1), dtype=float), z])

    I = np.eye(F.shape[1], dtype=float)
    I[0, 0] = 0.0
    lhs = F.T @ F + float(ridge) * I
    rhs = F.T @ yf
    beta = np.linalg.solve(lhs, rhs)
    y_hat = F @ beta
    resid = yf - y_hat
    ss_res = float(np.dot(resid, resid))
    y_center = yf - float(np.mean(yf))
    ss_tot = float(np.dot(y_center, y_center))
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 1e-12 else float("nan")
    resid_std = float(np.std(resid)) if resid.size else float("nan")
    return {
        "x_mu": x_mu,
        "x_sigma": x_sigma,
        "beta": beta,
        "r2": r2,
        "resid_std": resid_std,
        "train_count": int(Xf.shape[0]),
    }


def raceeng_predict_ridge_linear(model: dict | None, Xq: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    if Xq.ndim == 1:
        Xq = Xq[None, :]
    n = Xq.shape[0]
    if model is None:
        return np.full(n, float("nan"), dtype=float), np.full(n, float("inf"), dtype=float)
    z = (Xq - model["x_mu"]) / model["x_sigma"]
    F = np.hstack([np.ones((z.shape[0], 1), dtype=float), z])
    y = F @ model["beta"]
    radius = np.sqrt(np.mean(z * z, axis=1))
    base_unc = float(model.get("resid_std", float("nan")))
    if not math.isfinite(base_unc):
        base_unc = 0.02
    unc = np.maximum(1e-6, base_unc * (1.0 + 0.25 * radius))
    return y.astype(float), unc.astype(float)


def raceeng_estimate_noise_sigma(
    point_results: list[dict],
    spec_names: list[str],
) -> tuple[float, int]:
    buckets: dict[tuple[float, ...], list[float]] = {}
    for p in point_results:
        vector = p.get("vector") or {}
        key = tuple(round(float(vector.get(n, float("nan"))), 9) for n in spec_names)
        lap = float((p.get("metrics") or {}).get(RACEENG_OBJECTIVE, float("nan")))
        if not math.isfinite(lap):
            continue
        buckets.setdefault(key, []).append(lap)
    sigmas = []
    for laps in buckets.values():
        if len(laps) >= 2:
            sigmas.append(float(np.std(np.array(laps, dtype=float), ddof=1)))
    if not sigmas:
        return 0.0, 0
    return float(np.median(np.array(sigmas, dtype=float))), int(len(sigmas))


def raceeng_job_state(job) -> str:
    if job is None:
        return "missing"
    doc = getattr(job, "document", None)
    if doc is None:
        return "missing"
    data = getattr(doc, "data", None)
    if isinstance(data, dict):
        return str(data.get("state", "unknown") or "unknown").lower()
    if hasattr(data, "to_dict"):
        data_dict = data.to_dict()
        if isinstance(data_dict, dict):
            return str(data_dict.get("state", "unknown") or "unknown").lower()
    return "unknown"


def raceeng_compute_parameter_results(
    point_results: list[dict],
    active_specs: list[RaceEngParameter],
    current_vector: dict[str, float],
    study_id: str,
    exploration_id: str,
) -> dict[str, dict]:
    parameter_results: dict[str, dict] = {}
    for spec in active_specs:
        rows = list(point_results)
        gradients = {}
        for metric_name in RACEENG_METRICS:
            x_vals = [float(r["vector"][spec.name]) for r in rows]
            y_vals = [float(r["metrics"].get(metric_name, float("nan"))) for r in rows]
            gradients[metric_name] = raceeng_fit_linear_gradient(x_vals, y_vals)
        finite_obj = [
            float(r["metrics"].get(RACEENG_OBJECTIVE, float("nan")))
            for r in rows
            if math.isfinite(float(r["metrics"].get(RACEENG_OBJECTIVE, float("nan"))))
        ]
        x_vals = [float(r["vector"][spec.name]) for r in rows]
        parameter_results[spec.name] = {
            "center": float(current_vector[spec.name]),
            "low": float(min(x_vals)) if x_vals else float("nan"),
            "high": float(max(x_vals)) if x_vals else float("nan"),
            "n_points": int(len(rows)),
            "n_finite_objective": int(len(finite_obj)),
            "gradient": gradients,
            "study_id": study_id,
            "exploration_id": exploration_id,
        }
    return parameter_results


def raceeng_build_joint_point_definitions(
    specs: list[RaceEngParameter],
    active_specs: list[RaceEngParameter],
    current_vector: dict[str, float],
    points_per_parameter: int,
    trust_radius: float,
    mutation_scales: dict[str, float] | None = None,
    rng: np.random.Generator | None = None,
    current_repeat_count: int = 3,
    obj_model: dict | None = None,
    constraint_models: dict[str, dict | None] | None = None,
    soft_bounds: dict[str, float] | None = None,
    hardness: float = 1.0,
) -> list[dict]:
    if not active_specs:
        return []
    if rng is None:
        rng = np.random.default_rng(20260311)
    mutation_scales = mutation_scales or {}
    constraint_models = constraint_models or {}
    soft_bounds = soft_bounds or {}

    active_names = {s.name for s in active_specs}
    current_arr = np.array([float(current_vector[s.name]) for s in specs], dtype=float)
    lo = np.array([float(s.lower) for s in specs], dtype=float)
    hi = np.array([float(s.upper) for s in specs], dtype=float)
    delta = np.array([float(max(1e-12, s.delta)) for s in specs], dtype=float)
    scale = np.array([float(mutation_scales.get(s.name, 1.0)) for s in specs], dtype=float)
    active_idx = np.array([i for i, s in enumerate(specs) if s.name in active_names], dtype=int)
    if active_idx.size == 0:
        active_idx = np.arange(len(specs), dtype=int)

    n_points = int(max(120, len(active_specs) * max(3, int(points_per_parameter))))
    n_points = max(n_points, int(current_repeat_count) + 6)
    n_pool = int(max(n_points, math.ceil(2.5 * n_points)))

    point_defs: list[dict] = []
    for _ in range(int(current_repeat_count)):
        vec = {s.name: float(current_vector[s.name]) for s in specs}
        point_defs.append(
            {
                "parameter": "__batch__",
                "value": float("nan"),
                "vector": vec,
                "strategy": "current_replicate",
            }
        )

    # Geometry groups for correlated perturbations across pickup coordinates.
    groups: list[list[int]] = []
    tmp_groups: dict[str, list[int]] = {}
    for i, s in enumerate(specs):
        m = re.fullmatch(r"(.+)_([0-9]+)", s.name)
        if m is None:
            continue
        prefix = str(m.group(1))
        if "_r" not in prefix:
            continue
        tmp_groups.setdefault(prefix, []).append(i)
    for idxs in tmp_groups.values():
        if len(idxs) >= 2:
            groups.append(sorted(idxs))

    raw_candidates: list[np.ndarray] = []
    raw_strategies: list[str] = []
    for _ in range(max(0, n_pool - len(point_defs))):
        candidate = np.array(current_arr, dtype=float)
        r = float(rng.random())
        if r < 0.45:
            sigma = delta[active_idx] * float(trust_radius) * scale[active_idx] * 0.90
            candidate[active_idx] += rng.normal(loc=0.0, scale=sigma, size=active_idx.size)
            strategy = "gauss_full"
        elif r < 0.80:
            k = int(max(3, math.ceil(0.10 * active_idx.size)))
            idx_local = rng.choice(active_idx.size, size=k, replace=False)
            idx = active_idx[idx_local]
            sigma = delta[idx] * float(trust_radius) * scale[idx] * 2.0
            candidate[idx] += rng.normal(loc=0.0, scale=sigma, size=k)
            strategy = "sparse"
        elif groups:
            group = groups[int(rng.integers(len(groups)))]
            sigma = delta[group] * float(trust_radius) * scale[group] * 1.8
            candidate[group] += rng.normal(loc=0.0, scale=sigma, size=len(group))
            strategy = "grouped_geometry"
        else:
            j = int(active_idx[int(rng.integers(active_idx.size))])
            candidate[j] = lo[j] if float(rng.random()) < 0.5 else hi[j]
            strategy = "boundary_probe"
        candidate = np.clip(candidate, lo, hi)
        raw_candidates.append(candidate)
        raw_strategies.append(strategy)

    # Surrogate ranking to focus the batch, still keeping exploration diversity.
    chosen_idx = list(range(len(raw_candidates)))
    need = max(0, n_points - len(point_defs))
    if need < len(raw_candidates):
        if obj_model is not None and soft_bounds:
            X_cand = np.array(raw_candidates, dtype=float)
            lap_pred, lap_unc = raceeng_predict_ridge_linear(obj_model, X_cand)
            soft_v_pred = np.zeros(X_cand.shape[0], dtype=float)
            hard_v_pred = np.zeros(X_cand.shape[0], dtype=float)
            for c in RACEENG_CONSTRAINTS:
                cm = constraint_models.get(c.name)
                y_pred, _ = raceeng_predict_ridge_linear(cm, X_cand)
                hb = float(c.bound)
                sb = float(soft_bounds.get(c.name, hb))
                if c.sense == "<=":
                    hard_raw = np.maximum(0.0, y_pred - hb)
                    soft_raw = np.maximum(0.0, y_pred - sb)
                else:
                    hard_raw = np.maximum(0.0, hb - y_pred)
                    soft_raw = np.maximum(0.0, sb - y_pred)
                hard_v_pred += hard_raw / max(1.0, abs(hb))
                soft_v_pred += soft_raw / max(1.0, abs(hb))
            score = lap_pred + (35.0 + 125.0 * hardness) * soft_v_pred + (80.0 + 200.0 * hardness) * hard_v_pred
            score += 0.30 * lap_unc
            score = np.where(np.isfinite(score), score, np.inf)
            order = np.argsort(score)
            top = order[: max(need, int(math.ceil(1.8 * need)))]
            if top.size <= need:
                chosen_idx = list(map(int, top))
            else:
                weights = np.linspace(1.6, 0.5, top.size, dtype=float)
                weights = weights / np.sum(weights)
                sampled = rng.choice(top, size=need, replace=False, p=weights)
                chosen_idx = [int(x) for x in sampled]
        else:
            sampled = rng.choice(len(raw_candidates), size=need, replace=False)
            chosen_idx = [int(x) for x in sampled]

    for idx in chosen_idx[:need]:
        vec = {specs[i].name: float(raw_candidates[idx][i]) for i in range(len(specs))}
        point_defs.append(
            {
                "parameter": "__batch__",
                "value": float("nan"),
                "vector": vec,
                "strategy": str(raw_strategies[idx]),
            }
        )

    return point_defs


def raceeng_point_defs_to_sub_sweeps(
    current_car_config: dict,
    specs: list[RaceEngParameter],
    point_defs: list[dict],
) -> list[tuple[str, list[float]]]:
    point_path_values: list[dict[str, float]] = []
    all_paths: set[str] = set()
    for point in point_defs:
        row_paths: dict[str, float] = {}
        vec = point["vector"]
        for spec in specs:
            value = float(vec[spec.name])
            for path, vals in spec.build_sub_sweeps(current_car_config, value, value):
                row_paths[str(path)] = float(vals[0])
                all_paths.add(str(path))
        point_path_values.append(row_paths)

    ordered_paths = sorted(all_paths)
    return [
        (
            _raceeng_exploration_path(path),
            [float(point_path_values[i][path]) for i in range(len(point_path_values))],
        )
        for path in ordered_paths
    ]


async def raceeng_start_joint_parameter_exploration(
    session,
    tenant_id: str,
    worksheet_id: str,
    row_name: str,
    static_config_ids: dict[str, str],
    base_car_id: str,
    current_car_config: dict,
    specs: list[RaceEngParameter],
    active_specs: list[RaceEngParameter],
    current_vector: dict[str, float],
    points_per_parameter: int,
    trust_radius: float = 1.0,
    mutation_scales: dict[str, float] | None = None,
    rng: np.random.Generator | None = None,
    current_repeat_count: int = 3,
    obj_model: dict | None = None,
    constraint_models: dict[str, dict | None] | None = None,
    soft_bounds: dict[str, float] | None = None,
    hardness: float = 1.0,
    point_defs_override: list[dict] | None = None,
    runtime: RaceEngRuntime | None = None,
    phase: str = "main",
    iteration: int | None = None,
) -> dict:
    if point_defs_override is None:
        point_defs = raceeng_build_joint_point_definitions(
            specs=specs,
            active_specs=active_specs,
            current_vector=current_vector,
            points_per_parameter=points_per_parameter,
            trust_radius=trust_radius,
            mutation_scales=mutation_scales,
            rng=rng,
            current_repeat_count=current_repeat_count,
            obj_model=obj_model,
            constraint_models=constraint_models,
            soft_bounds=soft_bounds,
            hardness=hardness,
        )
    else:
        point_defs = [dict(p) for p in point_defs_override]
    if not point_defs:
        raise RuntimeError("No points generated for joint exploration")
    sub_sweeps = raceeng_point_defs_to_sub_sweeps(current_car_config, specs, point_defs)
    exploration_payload = build_raceeng_exploration(sub_sweeps)
    suffix = uuid4().hex[:8]
    launch_id = raceeng_new_launch_id()
    physical_row_name = raceeng_make_physical_row_name(row_name, launch_id)
    exploration_id = await create_user_config_from_payload(
        session=session,
        config_type="exploration",
        payload=exploration_payload,
        name=f"{row_name}-joint-exp-{suffix}",
        runtime=runtime,
    )
    if runtime is not None:
        study_id = await raceeng_call_with_retry(
            runtime,
            f"create_study:{row_name}",
            lambda s: canopy.create_study(
                s,
                "dynamicLap",
                f"{row_name}-joint-sweep-{suffix}",
                [
                    base_car_id,
                    static_config_ids["weather"],
                    static_config_ids["track"],
                    static_config_ids["userMaths"],
                    exploration_id,
                ],
            ),
        )
    else:
        study_id = await canopy.create_study(
            session,
            "dynamicLap",
            f"{row_name}-joint-sweep-{suffix}",
            [
                base_car_id,
                static_config_ids["weather"],
                static_config_ids["track"],
                static_config_ids["userMaths"],
                exploration_id,
            ],
        )
    raceeng_register_study_launch(
        runtime,
        {
            "launch_id": str(launch_id),
            "logical_row_name": str(row_name),
            "physical_row_name": str(physical_row_name),
            "phase": str(phase),
            "iteration": None if iteration is None else int(iteration),
            "kind": "joint_exploration",
            "status": "launched",
            "study_id": str(study_id),
            "exploration_id": str(exploration_id),
            "car_id": str(base_car_id),
            "active_spec_names": [str(s.name) for s in active_specs],
            "current_vector": {k: float(v) for k, v in current_vector.items()},
            "point_defs": [{k: v for k, v in dict(point).items()} for point in point_defs],
            "repeat_count": int(len(point_defs)),
        },
    )
    await upsert_worksheet_row(
        session=session,
        tenant_id=tenant_id,
        worksheet_id=worksheet_id,
        row_name=physical_row_name,
        worksheet_configs=_raceeng_worksheet_configs(
            tenant_id=tenant_id,
            car_id=base_car_id,
            weather_id=static_config_ids["weather"],
            track_id=static_config_ids["track"],
            user_maths_id=static_config_ids["userMaths"],
            exploration_id=exploration_id,
        ),
        study_id=study_id,
        runtime=runtime,
    )
    return {
        "launch_id": str(launch_id),
        "logical_row_name": str(row_name),
        "physical_row_name": str(physical_row_name),
        "study_id": study_id,
        "exploration_id": exploration_id,
        "point_defs": point_defs,
        "launch_time": float(perf_counter()),
        "point_count": int(len(point_defs)),
    }


async def raceeng_wait_for_partial_study(
    session,
    tenant_id: str,
    study_id: str,
    timeout_seconds: int,
    min_completed_fraction: float = 0.82,
    min_completed_jobs: int = 120,
    min_succeeded_jobs: int = 90,
    poll_seconds: float = 8.0,
    progress_label: str = "",
    runtime: RaceEngRuntime | None = None,
) -> dict:
    return await raceeng_wait_for_study_progress(
        session=session,
        tenant_id=tenant_id,
        study_id=study_id,
        timeout_seconds=timeout_seconds,
        min_completed_fraction=min_completed_fraction,
        min_completed_jobs=min_completed_jobs,
        min_succeeded_jobs=min_succeeded_jobs,
        poll_seconds=poll_seconds,
        runtime=runtime,
    )


async def raceeng_collect_study_point_results(
    session,
    study_id: str,
    point_defs: list[dict],
    runtime: RaceEngRuntime | None = None,
) -> dict:
    study = await raceeng_load_study_with_retry(
        session=session,
        study_id=study_id,
        sim_type="DynamicLap",
        runtime=runtime,
    )
    jobs = sorted_jobs_by_index(study.jobs)
    duration_seconds = raceeng_estimate_study_duration_seconds(jobs)
    point_results: list[dict] = []
    for idx, point in enumerate(point_defs):
        job = jobs[idx] if idx < len(jobs) else None
        state = raceeng_job_state(job)
        metrics = extract_raceeng_metrics(job) if job is not None else {m: float("nan") for m in RACEENG_METRICS}
        lap = float(metrics.get(RACEENG_OBJECTIVE, float("nan")))
        succeeded = bool(state == "successful" and math.isfinite(lap))
        point_results.append(
            {
                "parameter": str(point["parameter"]),
                "value": float(point["value"]),
                "point_index": int(idx),
                "local_point_index": int(idx),
                "job_id": None if job is None else str(job.document.document_id),
                "job_state": str(state),
                "succeeded": bool(succeeded),
                "metrics": metrics,
                "vector": dict(point["vector"]),
                "strategy": str(point["strategy"]),
            }
        )
    succeeded_count = sum(1 for p in point_results if bool(p.get("succeeded")))
    return {
        "point_results": point_results,
        "duration_seconds": float(duration_seconds),
        "succeeded_simulation_count": int(succeeded_count),
        "job_count": int(len(point_defs)),
    }


async def raceeng_finalize_joint_study(
    session,
    tenant_id: str,
    study_id: str,
    exploration_id: str,
    point_defs: list[dict],
    specs: list[RaceEngParameter],
    active_specs: list[RaceEngParameter],
    current_vector: dict[str, float],
    timeout_seconds: int,
    runtime: RaceEngRuntime | None = None,
    launch_id: str | None = None,
) -> dict:
    wait_result = await raceeng_wait_for_study_progress(
        session=session,
        tenant_id=tenant_id,
        study_id=study_id,
        timeout_seconds=timeout_seconds,
        min_completed_fraction=1.0,
        min_completed_jobs=len(point_defs),
        min_succeeded_jobs=1,
        poll_seconds=8.0,
        runtime=runtime,
    )
    collected = await raceeng_collect_study_point_results(
        session=session,
        study_id=study_id,
        point_defs=point_defs,
        runtime=runtime,
    )
    point_results = collected["point_results"]
    parameter_results = raceeng_compute_parameter_results(
        point_results=point_results,
        active_specs=active_specs,
        current_vector=current_vector,
        study_id=study_id,
        exploration_id=exploration_id,
    )
    noise_sigma, noise_groups = raceeng_estimate_noise_sigma(point_results, [s.name for s in specs])
    is_complete = bool(wait_result["is_complete"])
    if launch_id is not None:
        raceeng_mark_study_status(
            runtime,
            launch_id,
            "finalized" if is_complete else "partial_harvested",
            drop_from_pending=bool(is_complete),
            succeeded_simulation_count=int(wait_result["succeeded_simulation_count"]),
            duration_seconds=float(collected["duration_seconds"]),
            completed_job_count=int(wait_result["completed_job_count"]),
            job_count=int(wait_result["job_count"]),
        )
    return {
        "launch_id": None if launch_id is None else str(launch_id),
        "study_id": study_id,
        "exploration_id": exploration_id,
        "point_results": point_results,
        "parameter_results": parameter_results,
        "duration_seconds": float(collected["duration_seconds"]),
        "succeeded_simulation_count": int(wait_result["succeeded_simulation_count"]),
        "noise_sigma_tlap": float(noise_sigma),
        "noise_group_count": int(noise_groups),
        "is_complete": bool(is_complete),
        "job_count": int(wait_result["job_count"]),
        "completed_job_count": int(wait_result["completed_job_count"]),
    }


async def raceeng_run_joint_parameter_exploration(
    session,
    tenant_id: str,
    worksheet_id: str,
    row_name: str,
    static_config_ids: dict[str, str],
    base_car_id: str,
    current_car_config: dict,
    specs: list[RaceEngParameter],
    active_specs: list[RaceEngParameter],
    current_vector: dict[str, float],
    timeout_seconds: int,
    points_per_parameter: int,
    trust_radius: float = 1.0,
    mutation_scales: dict[str, float] | None = None,
    rng: np.random.Generator | None = None,
    current_repeat_count: int = 3,
    obj_model: dict | None = None,
    constraint_models: dict[str, dict | None] | None = None,
    soft_bounds: dict[str, float] | None = None,
    hardness: float = 1.0,
    partial_min_completed_fraction: float = 0.82,
    partial_min_completed_jobs: int = 120,
    partial_min_succeeded_jobs: int = 90,
    partial_poll_seconds: float = 8.0,
    progress_label: str = "",
    point_defs_override: list[dict] | None = None,
    runtime: RaceEngRuntime | None = None,
    phase: str = "main",
    iteration: int | None = None,
) -> dict:
    started = await raceeng_start_joint_parameter_exploration(
        session=session,
        tenant_id=tenant_id,
        worksheet_id=worksheet_id,
        row_name=row_name,
        static_config_ids=static_config_ids,
        base_car_id=base_car_id,
        current_car_config=current_car_config,
        specs=specs,
        active_specs=active_specs,
        current_vector=current_vector,
        points_per_parameter=points_per_parameter,
        trust_radius=trust_radius,
        mutation_scales=mutation_scales,
        rng=rng,
        current_repeat_count=current_repeat_count,
        obj_model=obj_model,
        constraint_models=constraint_models,
        soft_bounds=soft_bounds,
        hardness=hardness,
        point_defs_override=point_defs_override,
        runtime=runtime,
        phase=phase,
        iteration=iteration,
    )
    wait_info = await raceeng_wait_for_partial_study(
        session=session,
        tenant_id=tenant_id,
        study_id=started["study_id"],
        timeout_seconds=timeout_seconds,
        min_completed_fraction=partial_min_completed_fraction,
        min_completed_jobs=partial_min_completed_jobs,
        min_succeeded_jobs=partial_min_succeeded_jobs,
        poll_seconds=partial_poll_seconds,
        progress_label=progress_label,
        runtime=runtime,
    )
    collected = await raceeng_collect_study_point_results(
        session=session,
        study_id=started["study_id"],
        point_defs=started["point_defs"],
        runtime=runtime,
    )
    point_results = collected["point_results"]
    parameter_results = raceeng_compute_parameter_results(
        point_results=point_results,
        active_specs=active_specs,
        current_vector=current_vector,
        study_id=started["study_id"],
        exploration_id=started["exploration_id"],
    )
    noise_sigma, noise_groups = raceeng_estimate_noise_sigma(point_results, [s.name for s in specs])
    raceeng_mark_study_status(
        runtime,
        str(started["launch_id"]),
        "finalized" if bool(wait_info["is_complete"]) else "partial_harvested",
        drop_from_pending=bool(wait_info["is_complete"]),
        succeeded_simulation_count=int(collected["succeeded_simulation_count"]),
        completed_job_count=int(wait_info["completed_job_count"]),
        job_count=int(wait_info["job_count"]),
        duration_seconds=float(collected["duration_seconds"]),
    )
    return {
        "launch_id": str(started["launch_id"]),
        "logical_row_name": str(started["logical_row_name"]),
        "physical_row_name": str(started["physical_row_name"]),
        "study_id": started["study_id"],
        "exploration_id": started["exploration_id"],
        "point_defs": started["point_defs"],
        "point_results": point_results,
        "parameter_results": parameter_results,
        "duration_seconds": float(collected["duration_seconds"]),
        "succeeded_simulation_count": int(collected["succeeded_simulation_count"]),
        "noise_sigma_tlap": float(noise_sigma),
        "noise_group_count": int(noise_groups),
        "is_complete": bool(wait_info["is_complete"]),
        "job_count": int(wait_info["job_count"]),
        "completed_job_count": int(wait_info["completed_job_count"]),
        "wait_seconds": float(wait_info["wait_seconds"]),
    }


async def raceeng_recover_single_point_eval(
    session,
    launch_entry: dict,
    runtime: RaceEngRuntime | None = None,
) -> dict:
    study = await raceeng_load_study_with_retry(
        session=session,
        study_id=str(launch_entry["study_id"]),
        sim_type="DynamicLap",
        runtime=runtime,
    )
    jobs = sorted_jobs_by_index(study.jobs)
    if not jobs:
        raise RuntimeError(f"Recovered study {launch_entry['study_id']} returned no jobs")
    duration_seconds = raceeng_estimate_study_duration_seconds(jobs)
    metrics = extract_raceeng_metrics(jobs[0])
    raceeng_mark_study_status(
        runtime,
        str(launch_entry["launch_id"]),
        "finalized",
        drop_from_pending=True,
        succeeded_simulation_count=1,
        duration_seconds=float(duration_seconds),
    )
    return {
        "row_name": str(launch_entry.get("logical_row_name", "")),
        "physical_row_name": str(launch_entry.get("physical_row_name", "")),
        "launch_id": str(launch_entry["launch_id"]),
        "study_id": str(launch_entry["study_id"]),
        "car_id": str(launch_entry.get("car_id", "")),
        "job_id": jobs[0].document.document_id,
        "metrics": metrics,
        "duration_seconds": float(duration_seconds),
        "succeeded_simulation_count": 1,
    }


async def raceeng_recover_noop_bundle(
    session,
    launch_entry: dict,
    runtime: RaceEngRuntime | None = None,
) -> dict:
    repeat_count = int(max(2, int(launch_entry.get("repeat_count", 3))))
    study = await raceeng_load_study_with_retry(
        session=session,
        study_id=str(launch_entry["study_id"]),
        sim_type="DynamicLap",
        runtime=runtime,
    )
    jobs = sorted_jobs_by_index(study.jobs)
    if len(jobs) < repeat_count:
        raise RuntimeError(
            f"Recovered study {launch_entry['study_id']} returned only {len(jobs)} jobs for replicate bundle of {repeat_count}"
        )
    duration_seconds = raceeng_estimate_study_duration_seconds(jobs)
    runs: list[dict] = []
    laps: list[float] = []
    legal_count = 0
    for idx in range(repeat_count):
        metrics = extract_raceeng_metrics(jobs[idx])
        lap = float(metrics.get(RACEENG_OBJECTIVE, float("nan")))
        is_legal = bool(raceeng_is_legal(metrics))
        runs.append(
            {
                "row_name": str(launch_entry.get("logical_row_name", "")),
                "physical_row_name": str(launch_entry.get("physical_row_name", "")),
                "study_id": str(launch_entry["study_id"]),
                "exploration_id": str(launch_entry.get("exploration_id", "")),
                "car_id": str(launch_entry.get("car_id", "")),
                "launch_id": str(launch_entry["launch_id"]),
                "job_id": jobs[idx].document.document_id,
                "metrics": metrics,
                "duration_seconds": float(duration_seconds),
                "succeeded_simulation_count": repeat_count,
            }
        )
        laps.append(float(lap))
        legal_count += int(is_legal)
    median_metrics = raceeng_median_metrics_from_runs(runs)
    median_lap = float(median_metrics.get(RACEENG_OBJECTIVE, float("nan")))
    median_legal = bool(raceeng_is_legal(median_metrics))
    representative_idx = 0
    finite = [(idx, float(lap)) for idx, lap in enumerate(laps) if math.isfinite(float(lap))]
    if finite:
        med = float(statistics.median([lap for _, lap in finite]))
        representative_idx = int(min(finite, key=lambda pair: abs(pair[1] - med))[0])
    raceeng_mark_study_status(
        runtime,
        str(launch_entry["launch_id"]),
        "finalized",
        drop_from_pending=True,
        succeeded_simulation_count=repeat_count,
        duration_seconds=float(duration_seconds),
        legal_count=int(legal_count),
    )
    return {
        "runs": runs,
        "representative_eval": dict(runs[representative_idx]),
        "median_metrics": dict(median_metrics),
        "median_lap": float(median_lap),
        "median_legal": bool(median_legal),
        "laps": [float(v) for v in laps],
        "legal_count": int(legal_count),
        "repeat_count": int(repeat_count),
        "study_id": str(launch_entry["study_id"]),
        "exploration_id": str(launch_entry.get("exploration_id", "")),
        "logical_row_name": str(launch_entry.get("logical_row_name", "")),
        "physical_row_name": str(launch_entry.get("physical_row_name", "")),
        "launch_id": str(launch_entry["launch_id"]),
        "bundle_role": str(launch_entry.get("bundle_role", "generic")),
    }


async def raceeng_recover_pending_studies(
    session,
    tenant_id: str,
    specs: list[RaceEngParameter],
    runtime: RaceEngRuntime | None = None,
) -> dict:
    if runtime is None or not runtime.pending_studies:
        return {"joint_runs": [], "bundles": [], "single_runs": []}

    joint_runs: list[dict] = []
    bundles: list[dict] = []
    single_runs: list[dict] = []
    pending_entries = [dict(entry) for entry in runtime.pending_studies.values()]
    print(f"Recovering {len(pending_entries)} pending studies before launching new work.")
    for entry in pending_entries:
        launch_id = str(entry.get("launch_id", ""))
        kind = str(entry.get("kind", ""))
        try:
            if kind == "joint_exploration":
                active_names = {str(v) for v in list(entry.get("active_spec_names") or [])}
                active_specs = [s for s in specs if s.name in active_names]
                recovered = await raceeng_finalize_joint_study(
                    session=session,
                    tenant_id=tenant_id,
                    study_id=str(entry["study_id"]),
                    exploration_id=str(entry.get("exploration_id", "")),
                    point_defs=[dict(point) for point in list(entry.get("point_defs") or [])],
                    specs=specs,
                    active_specs=active_specs,
                    current_vector={k: float(v) for k, v in dict(entry.get("current_vector") or {}).items()},
                    timeout_seconds=180,
                    runtime=runtime,
                    launch_id=launch_id,
                )
                joint_runs.append({"entry": entry, "result": recovered})
            elif kind == "noop_bundle":
                bundle = await raceeng_recover_noop_bundle(
                    session=session,
                    launch_entry=entry,
                    runtime=runtime,
                )
                bundles.append(bundle)
            elif kind == "single_point":
                recovered = await raceeng_recover_single_point_eval(
                    session=session,
                    launch_entry=entry,
                    runtime=runtime,
                )
                single_runs.append(recovered)
            else:
                continue
            runtime.recovered_study_count += 1
        except Exception as ex:
            raceeng_mark_study_status(
                runtime,
                launch_id,
                "recovery_failed",
                drop_from_pending=False,
                recover_error=f"{type(ex).__name__}: {ex}",
            )
            print(f"Pending-study recovery failed for {launch_id}: {type(ex).__name__}: {ex}")
    return {
        "joint_runs": joint_runs,
        "bundles": bundles,
        "single_runs": single_runs,
    }


def raceeng_mad_sigma(values: list[float]) -> float:
    finite = [float(v) for v in values if math.isfinite(float(v))]
    if len(finite) < 2:
        return 0.0
    med = float(statistics.median(finite))
    mad = float(statistics.median([abs(v - med) for v in finite]))
    return float(1.4826 * mad)


def raceeng_update_ewma(prev: float, sample: float, alpha: float = 0.30) -> float:
    s = float(sample)
    if not math.isfinite(s):
        return float(prev)
    if not math.isfinite(float(prev)):
        return float(s)
    a = float(max(0.01, min(0.99, alpha)))
    return float((1.0 - a) * float(prev) + a * s)


def raceeng_select_endgame_active_specs(
    specs: list[RaceEngParameter],
    focus_scores: dict[str, float],
    obj_model: dict | None,
    fraction: float = 0.45,
    min_keep: int = 18,
) -> list[RaceEngParameter]:
    if not specs:
        return []
    keep = int(max(1, min(len(specs), max(min_keep, int(math.ceil(float(fraction) * len(specs)))))))
    model_effect: dict[str, float] = {s.name: 0.0 for s in specs}
    if obj_model is not None:
        beta = np.array(obj_model.get("beta", []), dtype=float)
        sigma = np.array(obj_model.get("x_sigma", []), dtype=float)
        if beta.size >= len(specs) + 1 and sigma.size >= len(specs):
            coef = beta[1 : 1 + len(specs)] / np.maximum(1e-12, sigma[: len(specs)])
            for i, spec in enumerate(specs):
                model_effect[spec.name] = float(abs(float(coef[i])) * max(1e-12, float(spec.delta)))

    blended = []
    for spec in specs:
        score = float(0.60 * model_effect.get(spec.name, 0.0) + 0.40 * float(focus_scores.get(spec.name, 0.0)))
        blended.append((score, spec))
    blended.sort(key=lambda item: float(item[0]), reverse=True)
    selected = [spec for _, spec in blended[:keep]]
    if not selected:
        selected = list(specs)
    return selected


def raceeng_build_es_point_definitions(
    specs: list[RaceEngParameter],
    incumbent_vector: dict[str, float],
    active_specs: list[RaceEngParameter],
    trust_radius: float,
    sigma_mult: float,
    mutation_scales: dict[str, float] | None = None,
    rng: np.random.Generator | None = None,
    points_target: int | None = None,
    obj_model: dict | None = None,
    constraint_models: dict[str, dict | None] | None = None,
    soft_bounds: dict[str, float] | None = None,
) -> list[dict]:
    if rng is None:
        rng = np.random.default_rng(20260312)
    mutation_scales = mutation_scales or {}
    constraint_models = constraint_models or {}
    soft_bounds = soft_bounds or {}
    if not specs:
        return []

    active_names = {s.name for s in active_specs} if active_specs else {s.name for s in specs}
    current_arr = np.array([float(incumbent_vector[s.name]) for s in specs], dtype=float)
    lo = np.array([float(s.lower) for s in specs], dtype=float)
    hi = np.array([float(s.upper) for s in specs], dtype=float)
    delta = np.array([float(max(1e-12, s.delta)) for s in specs], dtype=float)
    scale = np.array([float(mutation_scales.get(s.name, 1.0)) for s in specs], dtype=float)
    active_idx = np.array([i for i, s in enumerate(specs) if s.name in active_names], dtype=int)
    if active_idx.size == 0:
        active_idx = np.arange(len(specs), dtype=int)

    if points_target is None:
        points_target = int(min(168, max(96, 4 * active_idx.size)))
    points_target = int(max(48, points_target))
    pool_size = int(max(points_target, int(math.ceil(2.2 * points_target))))

    point_defs: list[dict] = []
    for _ in range(3):
        point_defs.append(
            {
                "parameter": "__es__",
                "value": float("nan"),
                "vector": {s.name: float(incumbent_vector[s.name]) for s in specs},
                "strategy": "es_center",
            }
        )

    raw_candidates: list[np.ndarray] = []
    raw_strategies: list[str] = []
    sigma_base = delta * float(max(0.25, min(2.5, sigma_mult))) * float(max(0.20, trust_radius)) * scale

    while len(raw_candidates) < max(0, pool_size - len(point_defs)):
        candidate = np.array(current_arr, dtype=float)
        r = float(rng.random())
        if r < 0.70:
            sigma = sigma_base[active_idx] * 0.95
            candidate[active_idx] += rng.normal(0.0, sigma, size=active_idx.size)
            strategy = "es_local_gauss"
        elif r < 0.90:
            k = int(max(2, math.ceil(0.08 * active_idx.size)))
            local_idx = rng.choice(active_idx, size=min(k, active_idx.size), replace=False)
            step = rng.normal(0.0, sigma_base[local_idx] * 1.20, size=local_idx.size)
            if float(rng.random()) < 0.5:
                step = -step
            candidate[local_idx] += step
            strategy = "es_mirrored"
        else:
            j = int(active_idx[int(rng.integers(active_idx.size))])
            candidate[j] = lo[j] if float(rng.random()) < 0.5 else hi[j]
            strategy = "es_boundary_nudge"
        candidate = np.clip(candidate, lo, hi)
        raw_candidates.append(candidate)
        raw_strategies.append(strategy)

    chosen_idx = list(range(len(raw_candidates)))
    need = max(0, points_target - len(point_defs))
    if need < len(raw_candidates):
        if obj_model is not None and soft_bounds:
            X_cand = np.array(raw_candidates, dtype=float)
            lap_pred, lap_unc = raceeng_predict_ridge_linear(obj_model, X_cand)
            soft_v_pred = np.zeros(X_cand.shape[0], dtype=float)
            hard_v_pred = np.zeros(X_cand.shape[0], dtype=float)
            for c in RACEENG_CONSTRAINTS:
                cm = constraint_models.get(c.name)
                y_pred, _ = raceeng_predict_ridge_linear(cm, X_cand)
                hb = float(c.bound)
                sb = float(soft_bounds.get(c.name, hb))
                if c.sense == "<=":
                    hard_raw = np.maximum(0.0, y_pred - hb)
                    soft_raw = np.maximum(0.0, y_pred - sb)
                else:
                    hard_raw = np.maximum(0.0, hb - y_pred)
                    soft_raw = np.maximum(0.0, sb - y_pred)
                hard_v_pred += hard_raw / max(1.0, abs(hb))
                soft_v_pred += soft_raw / max(1.0, abs(hb))
            score = lap_pred + 28.0 * soft_v_pred + 70.0 * hard_v_pred + 0.25 * lap_unc
            score = np.where(np.isfinite(score), score, np.inf)
            top = np.argsort(score)[: max(need, int(math.ceil(1.6 * need)))]
            if top.size <= need:
                chosen_idx = [int(x) for x in top]
            else:
                weights = np.linspace(1.4, 0.6, top.size, dtype=float)
                weights = weights / np.sum(weights)
                sampled = rng.choice(top, size=need, replace=False, p=weights)
                chosen_idx = [int(x) for x in sampled]
        else:
            sampled = rng.choice(len(raw_candidates), size=need, replace=False)
            chosen_idx = [int(x) for x in sampled]

    for idx in chosen_idx[:need]:
        point_defs.append(
            {
                "parameter": "__es__",
                "value": float("nan"),
                "vector": {specs[i].name: float(raw_candidates[idx][i]) for i in range(len(specs))},
                "strategy": str(raw_strategies[idx]),
            }
        )
    return point_defs


## Optimisation Engine

This section contains the challenge-specific optimisation loop and reporting pipeline. It relies on the helper layer above for resilient worksheet writes, reconnect handling, and study result collection.


In [ ]:
def raceeng_validate_optimisation_preflight(total_iterations: int):
    required_globals = [
        "RACEENG_CONSTRAINTS",
        "RACEENG_SOFT_CONSTRAINT_SLACK",
        "RACEENG_OBJECTIVE",
        "RACEENG_METRICS",
    ]
    missing = [name for name in required_globals if name not in globals()]
    if missing:
        raise RaceEngFatalNotebookError(
            "Fatal notebook error during optimisation preflight: missing required globals "
            + ", ".join(sorted(missing))
        )
    try:
        raceeng_constraint_profile(1, int(max(1, total_iterations)))
    except Exception as ex:
        raise RaceEngFatalNotebookError(
            f"Fatal notebook error during optimisation preflight: {type(ex).__name__}: {ex}"
        ) from ex


async def import_configs_and_optimize_balanced_car(
    session,
    tenant_id: str,
    worksheet_id: str,
    worksheet_name: str | None,
    worksheet_number: int | None,
    row_prefix: str,
    max_iterations: int,
    timeout_seconds: int,
    car_payload: dict,
    weather_payload: dict,
    track_payload: dict,
    user_maths_payload: dict,
    auth_data: canopy.AuthenticationData | None = None,
    sweep_points_per_parameter: int = 5,
    focus_fraction: float = 0.70,
    endgame_fraction: float = 0.35,
    endgame_verify_top_k: int = 2,
    endgame_extra_cycles_max: int = 20,
    endgame_stall_patience: int = 6,
    endgame_noise_margin_factor: float = 0.75,
    elite_archive_size: int = 8,
    endgame_verification_debt_threshold: float = 0.002,
    endgame_unverified_improvement_reset: float = 0.001,
    final_candidate_policy: str = "top2",
    force_two_main_starts: bool = True,
    worksheet_row_batch_seconds: float = 2.0,
    worksheet_row_batch_size: int = 8,
    start_iteration: int = 1,
    campaign_total_iterations: int | None = None,
    resume_state: dict | None = None,
    campaign_state_stem: str | None = None,
    finalization_only: bool = False,
):
    sweep_points_per_parameter = max(3, int(sweep_points_per_parameter))
    focus_fraction = float(max(0.30, min(1.0, focus_fraction)))
    endgame_fraction = float(max(0.05, min(0.70, endgame_fraction)))
    endgame_verify_top_k = int(max(1, endgame_verify_top_k))
    endgame_extra_cycles_max = int(max(0, endgame_extra_cycles_max))
    endgame_stall_patience = int(max(1, endgame_stall_patience))
    endgame_noise_margin_factor = float(max(0.10, min(2.00, endgame_noise_margin_factor)))
    elite_archive_size = int(max(2, elite_archive_size))
    endgame_verification_debt_threshold = float(max(0.0005, endgame_verification_debt_threshold))
    endgame_unverified_improvement_reset = float(max(0.0005, endgame_unverified_improvement_reset))
    final_candidate_policy = str(final_candidate_policy or "top2")
    worksheet_row_batch_seconds = float(max(0.2, worksheet_row_batch_seconds))
    worksheet_row_batch_size = int(max(1, worksheet_row_batch_size))
    start_iteration = int(max(1, start_iteration))
    run_iteration_count = int(max(1, max_iterations))
    total_iterations = int(max(start_iteration + run_iteration_count - 1, campaign_total_iterations or max_iterations))
    campaign_state_stem = campaign_state_stem or f"raceeng_campaign_{raceeng_safe_file_stem(row_prefix)}"
    raceeng_validate_optimisation_preflight(total_iterations)
    runtime = RaceEngRuntime(
        auth_data=raceeng_clone_auth_data(auth_data),
        session=session,
        campaign_state_stem=campaign_state_stem,
        worksheet_tenant_id=str(tenant_id),
        worksheet_id=str(worksheet_id),
        worksheet_name=None if worksheet_name is None else str(worksheet_name),
        worksheet_number=None if worksheet_number is None else int(worksheet_number),
        pending_studies={
            str(row.get("launch_id")): dict(row)
            for row in list((resume_state or {}).get("pending_studies") or [])
            if isinstance(row, dict) and row.get("launch_id")
        },
        finalization_only_pass_used=bool(finalization_only),
        worksheet_row_batch_seconds=float(worksheet_row_batch_seconds),
        worksheet_row_batch_size=int(worksheet_row_batch_size),
    )
    await raceeng_ensure_worksheet_writer(
        runtime,
        tenant_id=str(tenant_id),
        worksheet_id=str(worksheet_id),
        worksheet_name=worksheet_name,
        worksheet_number=worksheet_number,
        batch_seconds=worksheet_row_batch_seconds,
        batch_size=worksheet_row_batch_size,
    )

    try:
        car_payload = copy.deepcopy(dict(car_payload or {}))
        weather_payload = copy.deepcopy(dict(weather_payload or {}))
        track_payload = copy.deepcopy(dict(track_payload or {}))
        user_maths_payload = copy.deepcopy(dict(user_maths_payload or {}))

        for label, payload in [
            ("car", car_payload),
            ("weather", weather_payload),
            ("track", track_payload),
            ("userMaths", user_maths_payload),
        ]:
            if not isinstance(payload.get("config"), dict) or not payload.get("config"):
                raise ValueError(f"{label} payload is missing its config data")

        static_specs = [
            (
                "weather",
                weather_payload,
                get_original_config_name(weather_payload, None, "weather"),
                raceeng_payload_hash(weather_payload),
            ),
            (
                "track",
                track_payload,
                get_original_config_name(track_payload, None, "track"),
                raceeng_payload_hash(track_payload),
            ),
            (
                "userMaths",
                user_maths_payload,
                get_original_config_name(user_maths_payload, None, "userMaths"),
                raceeng_payload_hash(user_maths_payload),
            ),
        ]
        saved_static_configs = dict((resume_state or {}).get("static_configs") or {})
        static_config_ids: dict[str, str] = {}
        static_config_state: dict[str, dict] = {}
        created_static_config_keys: list[str] = []
        reused_static_config_keys: list[str] = []
        for cfg_type, payload, cfg_name, payload_hash in static_specs:
            saved_entry = dict(saved_static_configs.get(cfg_type) or {})
            saved_id = str(saved_entry.get("config_id") or "").strip()
            saved_hash = str(saved_entry.get("payload_hash") or "").strip()
            if saved_id and saved_hash == payload_hash:
                static_config_ids[cfg_type] = saved_id
                reused_static_config_keys.append(cfg_type)
            else:
                static_config_ids[cfg_type] = await create_user_config_from_payload(
                    session=session,
                    config_type=cfg_type,
                    payload=payload,
                    name=cfg_name,
                    runtime=runtime,
                )
                created_static_config_keys.append(cfg_type)
            static_config_state[cfg_type] = {
                "config_id": str(static_config_ids[cfg_type]),
                "name": str(cfg_name),
                "payload_hash": str(payload_hash),
            }

        if created_static_config_keys:
            print("Imported static configs:")
            for key in ["weather", "track", "userMaths"]:
                if key in created_static_config_keys:
                    print(f"  {key:9s} {static_config_ids[key]}")
        if reused_static_config_keys:
            print("Reused static configs:")
            for key in ["weather", "track", "userMaths"]:
                if key in reused_static_config_keys:
                    print(f"  {key:9s} {static_config_ids[key]}")

        specs = build_raceeng_parameter_specs(car_payload["config"])
        spec_names = [s.name for s in specs]
        current_vector = extract_raceeng_vector(car_payload, specs)
        trust_radius = 1.20
        focus_scores: dict[str, float] = {s.name: 0.0 for s in specs}
        mutation_scales: dict[str, float] = {
            s.name: (1.40 if (s.name.startswith("front_r") or s.name.startswith("rear_r")) else 1.0) for s in specs
        }
        rng = np.random.default_rng(20260311)
        calibration_residuals: list[float] = []
        calibration_bias = 0.0
        calibration_mae = 0.0
        obj_model = None
        constraint_models: dict[str, dict | None] = {c.name: None for c in RACEENG_CONSTRAINTS}
        history_X: list[list[float]] = []
        history_obj: list[float] = []
        history_cons: dict[str, list[float]] = {c.name: [] for c in RACEENG_CONSTRAINTS}
        seen_job_ids: set[str] = set()
        batch_points = int(sweep_points_per_parameter)
        study_wall_ewma = float("nan")
        island_count = 3 if len(spec_names) >= 3 else (2 if len(spec_names) >= 2 else 1)
        concurrent_starts = int(min(2, island_count))
        concurrency_cooldown = 0
        no_improve_streak = 0
        islands: list[dict] = []
        endgame_sigma_mult = 1.0
        endgame_no_improve_streak = 0
        extension_cycles_used = 0
        rolling_noise_sigma = float("nan")
        campaign_status = str((resume_state or {}).get("campaign_status", "running"))
        final_validation_complete = bool((resume_state or {}).get("final_validation_complete", False))
        result_json_written = bool((resume_state or {}).get("result_json_written", False))
        last_clean_iteration = int((resume_state or {}).get("last_clean_iteration", start_iteration - 1))
        final_result_json_path = str((resume_state or {}).get("final_result_json_path", "") or "")
        final_validation_bundle_state = (
            dict((resume_state or {}).get("final_validation_bundle") or {})
            if isinstance((resume_state or {}).get("final_validation_bundle"), dict)
            else None
        )

        run_stamp = datetime.now(UTC).strftime("%Y%m%d-%H%M%S")
        safe_prefix = re.sub(r"[^A-Za-z0-9_-]+", "_", row_prefix).strip("_") or "raceeng"
        ws_token = (
            f"ws{int(worksheet_number)}"
            if worksheet_number is not None
            else f"ws_{raceeng_safe_file_stem(worksheet_id)}"
        )
        output_stem = f"raceeng_{ws_token}_{safe_prefix}_{run_stamp}"
        output_files = {
            "sweep_results_csv": f"{output_stem}_sweep_results.csv",
            "sensitivities_csv": f"{output_stem}_sensitivities.csv",
            "iteration_summary_csv": f"{output_stem}_iteration_summary.csv",
            "surrogate_diagnostics_csv": f"{output_stem}_surrogate_diagnostics.csv",
            "constraint_schedule_csv": f"{output_stem}_constraint_schedule.csv",
            "noise_estimates_csv": f"{output_stem}_noise_estimates.csv",
            "timing_summary_csv": f"{output_stem}_timing_summary.csv",
            "controller_trace_csv": f"{output_stem}_controller_trace.csv",
            "endgame_cycle_summary_csv": f"{output_stem}_endgame_cycle_summary.csv",
            "verification_replicates_csv": f"{output_stem}_verification_replicates.csv",
            "checkpoint_trace_csv": f"{output_stem}_checkpoint_trace.csv",
            "api_reconnect_trace_csv": f"{output_stem}_api_reconnect_trace.csv",
            "elite_archive_csv": f"{output_stem}_elite_archive.csv",
            "worksheet_reconcile_trace_csv": f"{output_stem}_worksheet_reconcile_trace.csv",
            "checkpoint_json": f"{output_stem}_checkpoint.json",
            "best_car_json": f"{output_stem}_best_car.json",
            "result_json": f"{output_stem}_result.json",
        }
        output_files.update(
            {
                "campaign_state_json": raceeng_campaign_state_paths(campaign_state_stem)["json"],
                "campaign_history_npz": raceeng_campaign_state_paths(campaign_state_stem)["history"],
                "campaign_launches_jsonl": raceeng_campaign_state_paths(campaign_state_stem)["launches"],
                "campaign_pending_json": raceeng_campaign_state_paths(campaign_state_stem)["pending"],
                "campaign_worksheet_rows_jsonl": raceeng_campaign_state_paths(campaign_state_stem)["worksheet_rows"],
                "campaign_worksheet_manifest_json": raceeng_campaign_state_paths(campaign_state_stem)["worksheet_manifest"],
                "campaign_worksheet_reconcile_trace_csv": raceeng_campaign_state_paths(campaign_state_stem)["worksheet_reconcile_trace"],
            }
        )

        sweep_rows: list[dict] = []
        sensitivity_rows: list[dict] = []
        iteration_rows: list[dict] = []
        surrogate_rows: list[dict] = []
        constraint_rows: list[dict] = []
        noise_rows: list[dict] = []
        timing_rows: list[dict] = []
        controller_rows: list[dict] = []
        endgame_rows: list[dict] = []
        verification_rows: list[dict] = []
        checkpoint_rows: list[dict] = []
        elite_archive: list[dict] = []

        best_legal_metrics = None
        best_legal_vector = None
        best_verified_vector = None
        best_verified_metrics = None
        best_verified_lap = float("nan")
        incumbent_reference_lap = float("nan")
        incumbent_reference_bundle = None
        baseline_bundle = None
        current_metrics = {}
        current_eval = {}

        def _merge_recovered_joint_runs(joint_runs_local: list[dict]) -> int:
            merged_count = 0
            nonlocal best_legal_metrics, best_legal_vector, elite_archive
            for recovered in list(joint_runs_local or []):
                rr = dict(recovered.get("result") or {})
                source_iteration = int((recovered.get("entry") or {}).get("iteration") or 0)
                source_phase = str((recovered.get("entry") or {}).get("phase") or "recovered")
                for point in list(rr.get("point_results", [])):
                    point_metrics = dict(point.get("metrics") or {})
                    point_vector = dict(point.get("vector") or current_vector)
                    point_job_id = point.get("job_id")
                    if point_job_id and str(point_job_id) in seen_job_ids:
                        continue
                    if point_job_id:
                        seen_job_ids.add(str(point_job_id))
                    if not math.isfinite(float(point_metrics.get(RACEENG_OBJECTIVE, float("nan")))):
                        continue
                    history_X.append([float(point_vector[s.name]) for s in specs])
                    history_obj.append(float(point_metrics[RACEENG_OBJECTIVE]))
                    for c in RACEENG_CONSTRAINTS:
                        history_cons[c.name].append(float(point_metrics.get(c.name, float("nan"))))
                    if raceeng_is_legal(point_metrics):
                        if best_legal_metrics is None or float(point_metrics[RACEENG_OBJECTIVE]) < float(
                            best_legal_metrics[RACEENG_OBJECTIVE]
                        ):
                            best_legal_metrics = dict(point_metrics)
                            best_legal_vector = dict(point_vector)
                        elite_archive = raceeng_update_elite_archive(
                            elite_archive,
                            raceeng_make_archive_entry(
                                specs=specs,
                                vector=point_vector,
                                metrics=point_metrics,
                                source_phase=source_phase,
                                source_iteration=source_iteration,
                                source_label=str(point.get("job_id") or "recovered_point"),
                                verified=False,
                            ),
                            specs=specs,
                            max_size=elite_archive_size,
                        )
                    merged_count += 1
            return int(merged_count)

        if resume_state:
            current_vector = {
                s.name: float((resume_state.get("current_vector") or {}).get(s.name, current_vector[s.name]))
                for s in specs
            }
            trust_radius = float(resume_state.get("trust_radius", trust_radius))
            focus_scores.update({k: float(v) for k, v in (resume_state.get("focus_scores") or {}).items() if k in focus_scores})
            mutation_scales.update(
                {k: float(v) for k, v in (resume_state.get("mutation_scales") or {}).items() if k in mutation_scales}
            )
            if isinstance(resume_state.get("rng_state"), dict):
                rng.bit_generator.state = dict(resume_state["rng_state"])
            calibration_residuals = [
                float(v) for v in list(resume_state.get("calibration_residuals") or [])[-500:] if math.isfinite(float(v))
            ]
            calibration_bias = float(resume_state.get("calibration_bias", 0.0))
            calibration_mae = float(resume_state.get("calibration_mae", 0.0))
            history_X = [[float(v) for v in row] for row in list(resume_state.get("history_X") or [])]
            history_obj = [float(v) for v in list(resume_state.get("history_obj") or [])]
            history_cons = {
                c.name: [float(v) for v in list((resume_state.get("history_cons") or {}).get(c.name, []))]
                for c in RACEENG_CONSTRAINTS
            }
            obj_model, constraint_models = raceeng_fit_surrogates_from_history(specs, history_X, history_obj, history_cons)
            seen_job_ids = {str(v) for v in list(resume_state.get("seen_job_ids") or [])}
            batch_points = int(resume_state.get("batch_points", batch_points))
            study_wall_ewma = float(resume_state.get("study_wall_ewma_seconds", study_wall_ewma))
            concurrent_starts = int(resume_state.get("concurrent_starts", concurrent_starts))
            concurrency_cooldown = int(resume_state.get("concurrency_cooldown", concurrency_cooldown))
            no_improve_streak = int(resume_state.get("no_improve_streak", no_improve_streak))
            endgame_sigma_mult = float(resume_state.get("endgame_sigma_mult", endgame_sigma_mult))
            endgame_no_improve_streak = int(resume_state.get("endgame_no_improve_streak", endgame_no_improve_streak))
            extension_cycles_used = int(resume_state.get("extension_cycles_used", extension_cycles_used))
            rolling_noise_sigma = float(resume_state.get("rolling_noise_sigma", rolling_noise_sigma))
            elite_archive = [dict(row) for row in list(resume_state.get("elite_archive") or []) if isinstance(row, dict)]
            runtime.worksheet_rows_written = int(resume_state.get("worksheet_rows_written", runtime.worksheet_rows_written))
            runtime.worksheet_rows_reinserted = int(
                resume_state.get("worksheet_rows_reinserted", runtime.worksheet_rows_reinserted)
            )
            runtime.worksheet_reconcile_retry_count = int(
                resume_state.get("worksheet_reconcile_retry_count", runtime.worksheet_reconcile_retry_count)
            )

            best_legal_metrics = (
                None
                if not isinstance(resume_state.get("best_legal_metrics"), dict)
                else {k: float(v) for k, v in resume_state["best_legal_metrics"].items()}
            )
            best_legal_vector = (
                None
                if not isinstance(resume_state.get("best_legal_vector"), dict)
                else {k: float(v) for k, v in resume_state["best_legal_vector"].items()}
            )
            best_verified_metrics = (
                None
                if not isinstance(resume_state.get("best_verified_metrics"), dict)
                else {k: float(v) for k, v in resume_state["best_verified_metrics"].items()}
            )
            best_verified_vector = (
                None
                if not isinstance(resume_state.get("best_verified_vector"), dict)
                else {k: float(v) for k, v in resume_state["best_verified_vector"].items()}
            )
            best_verified_lap = float(resume_state.get("best_verified_lap", best_verified_lap))
            incumbent_reference_lap = float(resume_state.get("incumbent_reference_lap", incumbent_reference_lap))
            incumbent_reference_bundle = resume_state.get("incumbent_reference_bundle")
            baseline_bundle = resume_state.get("baseline_bundle")
            baseline_lap = float(baseline_bundle.get("median_lap", float("nan"))) if isinstance(baseline_bundle, dict) else float("nan")
            baseline_laps = [float(v) for v in list((baseline_bundle or {}).get("laps", []))]
            baseline_legal_count = int((baseline_bundle or {}).get("legal_count", 0))
            baseline_repeat_count = int((baseline_bundle or {}).get("repeat_count", len(baseline_laps)))
            baseline_legal = bool((baseline_bundle or {}).get("median_legal", False))
            current_metrics = (
                {k: float(v) for k, v in (resume_state.get("current_metrics") or {}).items()}
                if isinstance(resume_state.get("current_metrics"), dict)
                else dict(best_legal_metrics or best_verified_metrics or (baseline_bundle or {}).get("median_metrics") or {})
            )
            current_car_id, current_payload = await raceeng_materialize_vector_car(
                session=session,
                base_car_payload=car_payload,
                specs=specs,
                vector=current_vector,
                name_prefix=f"{row_prefix}-resumeSeed",
                runtime=runtime,
            )
            current_eval = {
                "study_id": str(resume_state.get("current_study_id", "__resume__")),
                "car_id": current_car_id,
                "job_id": str(resume_state.get("current_job_id", "__resume__")),
                "metrics": dict(current_metrics),
                "duration_seconds": 0.0,
                "succeeded_simulation_count": 0,
            }
            print(
                "Resumed campaign state: "
                f"start_iteration={start_iteration}, current_lap={float(current_metrics.get(RACEENG_OBJECTIVE, float('nan'))):.6f}, "
                f"history_rows={len(history_obj)}, best_legal="
                f"{float(best_legal_metrics.get(RACEENG_OBJECTIVE, float('nan'))) if best_legal_metrics else float('nan'):.6f}"
            )
        await raceeng_reconcile_worksheet_manifest(runtime)
        recovery = await raceeng_recover_pending_studies(
            session=session,
            tenant_id=tenant_id,
            specs=specs,
            runtime=runtime,
        )
        recovered_joint_point_count = _merge_recovered_joint_runs(list(recovery.get("joint_runs", [])))
        recovered_bundles = [dict(b) for b in list(recovery.get("bundles", []))]
        if recovery.get("joint_runs") or recovery.get("bundles") or recovery.get("single_runs"):
            obj_model, constraint_models = raceeng_fit_surrogates_from_history(specs, history_X, history_obj, history_cons)
            print(
                f"Recovered pending studies: joint={len(recovery.get('joint_runs', []))}, "
                f"bundles={len(recovery.get('bundles', []))}, single={len(recovery.get('single_runs', []))}, "
                f"joint_points={recovered_joint_point_count}"
            )
        if (not isinstance(baseline_bundle, dict)) and recovered_bundles:
            baseline_candidates = [b for b in recovered_bundles if str(b.get("bundle_role", "")) == "baseline"]
            if baseline_candidates:
                baseline_bundle = dict(baseline_candidates[-1])
                baseline_lap = float(baseline_bundle["median_lap"])
                baseline_laps = [float(v) for v in baseline_bundle.get("laps", [])]
                baseline_legal_count = int(baseline_bundle.get("legal_count", 0))
                baseline_repeat_count = int(baseline_bundle.get("repeat_count", len(baseline_laps)))
                baseline_legal = bool(baseline_bundle.get("median_legal", False))
        if resume_state:
            if not isinstance(baseline_bundle, dict):
                raise RuntimeError("Resume state is missing baseline_bundle")
        else:
            current_payload = apply_raceeng_vector(car_payload, specs, current_vector)
            baseline_bundle = await raceeng_run_single_point_replicates(
                session=session,
                tenant_id=tenant_id,
                worksheet_id=worksheet_id,
                row_name=f"{row_prefix}-baseline",
                static_config_ids=static_config_ids,
                car_payload=current_payload,
                timeout_seconds=timeout_seconds,
                repeat_count=3,
                runtime=runtime,
                phase="baseline",
                iteration=0,
                bundle_role="baseline",
            )
            current_eval = dict(baseline_bundle["representative_eval"])
            current_metrics = dict(current_eval["metrics"])
            baseline_lap = float(baseline_bundle["median_lap"])
            baseline_laps = [float(v) for v in baseline_bundle["laps"]]
            baseline_legal_count = int(baseline_bundle["legal_count"])
            baseline_repeat_count = int(baseline_bundle["repeat_count"])
            baseline_legal = bool(baseline_bundle["median_legal"])
            print(
                "Baseline median-of-3: "
                f"laps={[round(v, 6) for v in baseline_laps]}, "
                f"median={baseline_lap:.6f}, legal_runs={baseline_legal_count}/{baseline_repeat_count}, median_hard_legal={baseline_legal}"
            )
            if baseline_legal:
                best_legal_metrics = dict(baseline_bundle["median_metrics"])
                best_legal_vector = dict(current_vector)
            elif raceeng_is_legal(current_metrics):
                best_legal_metrics = dict(current_metrics)
                best_legal_vector = dict(current_vector)
            best_verified_vector = dict(current_vector) if baseline_legal else None
            best_verified_metrics = dict(baseline_bundle["median_metrics"]) if baseline_legal else None
            best_verified_lap = float(baseline_lap) if baseline_legal else float("nan")
            incumbent_reference_lap = float(baseline_lap)
            incumbent_reference_bundle = dict(baseline_bundle)
            if baseline_legal:
                elite_archive = raceeng_update_elite_archive(
                    elite_archive,
                    raceeng_make_archive_entry(
                        specs=specs,
                        vector=current_vector,
                        metrics=dict(baseline_bundle["median_metrics"]),
                        source_phase="baseline",
                        source_iteration=max(0, start_iteration - 1),
                        source_label="baseline",
                        verified=True,
                        last_verified_median=float(baseline_lap),
                    ),
                    specs=specs,
                    max_size=elite_archive_size,
                )

        current_lap = float(current_metrics.get(RACEENG_OBJECTIVE, float("nan")))
        if best_verified_vector is None and baseline_bundle is not None and bool(baseline_bundle.get("median_legal", False)):
            best_verified_vector = dict(current_vector)
            best_verified_metrics = dict(baseline_bundle["median_metrics"])
            best_verified_lap = float(baseline_bundle["median_lap"])
        if not math.isfinite(incumbent_reference_lap):
            incumbent_reference_lap = float(current_lap)

        saved_islands = list(resume_state.get("islands") or []) if isinstance(resume_state, dict) else []
        if saved_islands:
            for ii, island_raw in enumerate(saved_islands[:island_count]):
                ivec = {
                    s.name: float((island_raw.get("vector") or {}).get(s.name, current_vector[s.name]))
                    for s in specs
                }
                imetrics = {
                    k: float(v)
                    for k, v in (island_raw.get("metrics") or current_metrics).items()
                }
                islands.append(
                    {
                        "id": int(island_raw.get("id", ii + 1)),
                        "vector": dict(ivec),
                        "metrics": dict(imetrics),
                        "trust_radius": float(island_raw.get("trust_radius", trust_radius)),
                        "mutation_scales": {
                            k: float(v)
                            for k, v in dict(island_raw.get("mutation_scales") or mutation_scales).items()
                        },
                    }
                )
        if not islands:
            for ii in range(int(island_count)):
                ivec = dict(current_vector)
                if ii > 0:
                    picks = rng.choice(len(specs), size=min(10, len(specs)), replace=False)
                    for pidx in picks:
                        s = specs[int(pidx)]
                        span = float(max(1e-12, s.delta) * 1.8)
                        trial = float(ivec[s.name] + rng.normal(loc=0.0, scale=span))
                        ivec[s.name] = float(_raceeng_clamp(trial, s.lower, s.upper))
                islands.append(
                    {
                        "id": int(ii + 1),
                        "vector": dict(ivec),
                        "metrics": dict(current_metrics),
                        "trust_radius": float(trust_radius),
                        "mutation_scales": dict(mutation_scales),
                    }
                )

        main_iterations = int(max(1, int(math.ceil((1.0 - endgame_fraction) * total_iterations))))
        nominal_endgame_cycles = int(max(0, total_iterations - main_iterations))
        global_iteration_end = int(min(total_iterations, start_iteration + run_iteration_count - 1))
        if finalization_only:
            campaign_status = "finalizing"
            global_iteration_end = 0

        def _campaign_state_payload(
            *,
            phase: str,
            iteration: int,
            current_metrics_local: dict[str, float],
            current_eval_local: dict,
        ) -> dict:
            return {
                "version": 3,
                "row_prefix": str(row_prefix),
                "campaign_state_stem": str(campaign_state_stem),
                "worksheet_tenant_id": str(tenant_id),
                "worksheet_id": str(worksheet_id),
                "worksheet_name": None if worksheet_name is None else str(worksheet_name),
                "worksheet_number": None if worksheet_number is None else int(worksheet_number),
                "campaign_status": str(campaign_status),
                "phase": str(phase),
                "iteration": int(iteration),
                "next_iteration": int(iteration + 1),
                "campaign_target_iterations": int(total_iterations),
                "campaign_total_iterations": int(total_iterations),
                "last_clean_iteration": int(last_clean_iteration),
                "final_validation_complete": bool(final_validation_complete),
                "result_json_written": bool(result_json_written),
                "final_result_json_path": str(final_result_json_path),
                "api_reauth_count": int(runtime.api_reauth_count),
                "api_retry_count": int(runtime.api_retry_count),
                "recovered_study_count": int(runtime.recovered_study_count),
                "current_vector": {k: float(v) for k, v in current_vector.items()},
                "current_metrics": {k: float(v) for k, v in current_metrics_local.items()},
                "current_study_id": str(current_eval_local.get("study_id", "")),
                "current_job_id": str(current_eval_local.get("job_id", "")),
                "trust_radius": float(trust_radius),
                "focus_scores": {k: float(v) for k, v in focus_scores.items()},
                "mutation_scales": {k: float(v) for k, v in mutation_scales.items()},
                "rng_state": dict(rng.bit_generator.state),
                "calibration_residuals": [float(v) for v in calibration_residuals],
                "calibration_bias": float(calibration_bias),
                "calibration_mae": float(calibration_mae),
                "seen_job_ids": sorted(str(v) for v in seen_job_ids),
                "batch_points": int(batch_points),
                "study_wall_ewma_seconds": float(study_wall_ewma),
                "concurrent_starts": int(concurrent_starts),
                "concurrency_cooldown": int(concurrency_cooldown),
                "no_improve_streak": int(no_improve_streak),
                "islands": [
                    {
                        "id": int(i.get("id", 0)),
                        "vector": {k: float(v) for k, v in dict(i.get("vector") or {}).items()},
                        "metrics": {k: float(v) for k, v in dict(i.get("metrics") or {}).items()},
                        "trust_radius": float(i.get("trust_radius", float("nan"))),
                        "mutation_scales": {k: float(v) for k, v in dict(i.get("mutation_scales") or {}).items()},
                    }
                    for i in islands
                ],
                "best_legal_vector": None if best_legal_vector is None else {k: float(v) for k, v in best_legal_vector.items()},
                "best_legal_metrics": None
                if best_legal_metrics is None
                else {k: float(v) for k, v in best_legal_metrics.items()},
                "best_verified_vector": None
                if best_verified_vector is None
                else {k: float(v) for k, v in best_verified_vector.items()},
                "best_verified_metrics": None
                if best_verified_metrics is None
                else {k: float(v) for k, v in best_verified_metrics.items()},
                "best_verified_lap": float(best_verified_lap),
                "elite_archive": [dict(row) for row in elite_archive],
                "best_archive_legal_lap": float(raceeng_best_archive_legal_lap(elite_archive)),
                "incumbent_reference_lap": float(incumbent_reference_lap),
                "incumbent_reference_bundle": incumbent_reference_bundle,
                "baseline_bundle": baseline_bundle,
                "final_validation_bundle": final_validation_bundle_state,
                "rolling_noise_sigma": float(rolling_noise_sigma),
                "endgame_sigma_mult": float(endgame_sigma_mult),
                "endgame_no_improve_streak": int(endgame_no_improve_streak),
                "extension_cycles_used": int(extension_cycles_used),
                "worksheet_rows_written": int(runtime.worksheet_rows_written),
                "worksheet_rows_reinserted": int(runtime.worksheet_rows_reinserted),
                "worksheet_reconcile_retry_count": int(runtime.worksheet_reconcile_retry_count),
                "static_configs": {k: dict(v) for k, v in static_config_state.items()},
                "pending_studies": [
                    {k: v for k, v in dict(entry).items()}
                    for entry in sorted(
                        runtime.pending_studies.values(),
                        key=lambda row: (str(row.get("logical_row_name", "")), str(row.get("launch_id", ""))),
                    )
                ],
                "timestamp_utc": datetime.now(UTC).isoformat(),
            }

        def _write_checkpoint(
            *,
            phase: str,
            iteration: int,
            current_metrics_local: dict[str, float],
            current_eval_local: dict,
        ):
            payload = _campaign_state_payload(
                phase=phase,
                iteration=iteration,
                current_metrics_local=current_metrics_local,
                current_eval_local=current_eval_local,
            )
            Path(output_files["checkpoint_json"]).write_text(json.dumps(payload, indent=2), encoding="utf-8")
            raceeng_write_campaign_state(
                campaign_state_stem=campaign_state_stem,
                state_payload=payload,
                history_X=history_X,
                history_obj=history_obj,
                history_cons=history_cons,
            )
            pd.DataFrame(runtime.reconnect_rows).to_csv(output_files["api_reconnect_trace_csv"], index=False)
            pd.DataFrame(runtime.worksheet_reconcile_rows).to_csv(output_files["worksheet_reconcile_trace_csv"], index=False)

        _write_checkpoint(
            phase="resume" if resume_state else "bootstrap",
            iteration=int(max(last_clean_iteration, start_iteration - 1)),
            current_metrics_local=current_metrics,
            current_eval_local=current_eval,
        )

        for iteration in range(start_iteration, min(main_iterations, global_iteration_end) + 1):
            iter_start = perf_counter()
            iter_row_prefix = f"{row_prefix}-it{iteration:02d}"
            profile = raceeng_constraint_profile(iteration, total_iterations)
            soft_bounds = profile["bounds"]
            hardness = float(profile["hardness"])
            soft_penalty = float(35.0 + 125.0 * hardness)
            for row in profile["rows"]:
                rr = {"iteration": int(iteration)}
                rr.update(row)
                constraint_rows.append(rr)

            pending_recovery = await raceeng_recover_pending_studies(
                session=session,
                tenant_id=tenant_id,
                specs=specs,
                runtime=runtime,
            )
            merged_pending_points = _merge_recovered_joint_runs(list(pending_recovery.get("joint_runs", [])))
            if merged_pending_points > 0:
                obj_model, constraint_models = raceeng_fit_surrogates_from_history(specs, history_X, history_obj, history_cons)
                print(f"Merged {merged_pending_points} recovered pending points before iteration {iteration}.")

            print(f"\n========== Iteration {iteration} [phase=main] ==========")
            print(f"Trust radius: {trust_radius:.3f} | hardening={hardness:.3f}")
            campaign_eta_seconds = float("nan")
            if timing_rows:
                recent = [float(r.get("iteration_wall_seconds", float("nan"))) for r in timing_rows[-5:]]
                recent = [v for v in recent if math.isfinite(v)]
                if recent:
                    campaign_eta_seconds = float(np.mean(np.array(recent, dtype=float)) * max(0, total_iterations - iteration + 1))
                    print(f"Approx campaign ETA: {campaign_eta_seconds/60.0:.1f} min")

            current_metrics = current_eval["metrics"]
            current_lap = float(current_metrics[RACEENG_OBJECTIVE])
            current_legal = raceeng_is_legal(current_metrics)
            current_soft_legal = raceeng_is_legal_with_bounds(current_metrics, soft_bounds)
            current_violation = raceeng_total_violation(current_metrics)
            current_soft_violation = raceeng_total_violation_with_bounds(current_metrics, soft_bounds)
            current_soft_score = float(current_lap + soft_penalty * current_soft_violation)
            print(
                "Current: "
                f"tLapTotal={current_lap:.6f}, hard_legal={current_legal}, "
                f"hard_violation={current_violation:.6g}, soft_violation={current_soft_violation:.6g}"
            )
            if best_legal_metrics is not None:
                best_so_far = float(best_legal_metrics.get(RACEENG_OBJECTIVE, float("nan")))
                if math.isfinite(best_so_far):
                    gain = float(baseline_lap - best_so_far) if math.isfinite(baseline_lap) else float("nan")
                    gain_txt = "nan" if not math.isfinite(gain) else f"{gain:+.6f}"
                    print(f"Best legal so far: tLapTotal={best_so_far:.6f} (vs baseline median {gain_txt})")

            if current_legal:
                if best_legal_metrics is None or current_lap < float(best_legal_metrics[RACEENG_OBJECTIVE]):
                    best_legal_metrics = dict(current_metrics)
                    best_legal_vector = dict(current_vector)

            center_row = {
                "iteration": iteration,
                "parameter": "__current__",
                "point": "center",
                "value": float("nan"),
                "study_id": current_eval["study_id"],
                "exploration_id": None,
                "job_id": current_eval["job_id"],
                "legal": bool(current_legal),
                "hard_legal": bool(current_legal),
                "soft_legal": bool(current_soft_legal),
                "total_violation": float(current_violation),
                "soft_violation": float(current_soft_violation),
                "point_index": -1,
                "local_point_index": -1,
                "strategy": "current",
            }
            center_row.update({k: float(current_metrics.get(k, float("nan"))) for k in RACEENG_METRICS})
            center_row.update({f"param_{k}": float(v) for k, v in current_vector.items()})
            sweep_rows.append(center_row)

            active_specs = list(specs)
            active_names = {s.name for s in active_specs}
            print(f"Active parameters ({len(active_specs)}/{len(specs)}): {', '.join(s.name for s in active_specs)}")

            base_car_id = current_eval["car_id"]
            launch_count = max(1, min(int(concurrent_starts), len(islands)))
            start_idx = (int(iteration) - 1) % len(islands)
            launch_indices = [int((start_idx + j) % len(islands)) for j in range(launch_count)]
            iteration_eta_seconds = float("nan")
            if math.isfinite(study_wall_ewma):
                iteration_eta_seconds = float(max(30.0, study_wall_ewma * max(1.0, 0.75 + 0.25 * launch_count)))
                print(f"Approx iteration ETA: {iteration_eta_seconds/60.0:.1f} min")
            print(
                "Controller: "
                f"starts={launch_count}/{len(islands)}; "
                f"batch_points={batch_points}; "
                f"ewma_wait={'nan' if not math.isfinite(study_wall_ewma) else f'{study_wall_ewma:.1f}s'}"
            )

            async def _run_joint_launch(
                island_idx: int,
                tag: str,
                vec: dict[str, float],
                island_trust_radius: float,
                island_mutation_scales: dict[str, float],
            ) -> tuple[int, str, dict[str, float], dict]:
                current_car_config = apply_raceeng_vector(car_payload, specs, vec)["config"]
                jr = await raceeng_run_joint_parameter_exploration(
                    session=session,
                    tenant_id=tenant_id,
                    worksheet_id=worksheet_id,
                    row_name=f"{iter_row_prefix}-{tag}-jointSweep",
                    static_config_ids=static_config_ids,
                    base_car_id=base_car_id,
                    current_car_config=current_car_config,
                    specs=specs,
                    active_specs=active_specs,
                    current_vector=vec,
                    timeout_seconds=timeout_seconds,
                    points_per_parameter=int(batch_points),
                    trust_radius=float(island_trust_radius),
                    mutation_scales=dict(island_mutation_scales),
                    rng=rng,
                    current_repeat_count=3,
                    obj_model=obj_model,
                    constraint_models=constraint_models,
                    soft_bounds=soft_bounds,
                    hardness=hardness,
                    partial_min_completed_fraction=0.82,
                    partial_min_completed_jobs=120,
                    partial_min_succeeded_jobs=90,
                    partial_poll_seconds=8.0,
                    progress_label="",
                    runtime=runtime,
                    phase="main",
                    iteration=iteration,
                )
                return int(island_idx), str(tag), dict(vec), dict(jr)

            joint_tasks: list[asyncio.Task] = []
            for local_ord, island_idx in enumerate(launch_indices):
                island = islands[island_idx]
                tag = f"is{int(island['id']):02d}"
                vec = dict(island["vector"])
                task = asyncio.create_task(
                    _run_joint_launch(
                        island_idx=int(island_idx),
                        tag=tag,
                        vec=vec,
                        island_trust_radius=float(island["trust_radius"]),
                        island_mutation_scales=dict(island["mutation_scales"]),
                    )
                )
                joint_tasks.append(task)

            recent_iter_seconds = [float(r.get("iteration_wall_seconds", float("nan"))) for r in timing_rows[-5:]]
            recent_iter_seconds = [v for v in recent_iter_seconds if math.isfinite(v)]
            mean_iter_seconds = (
                float(np.mean(np.array(recent_iter_seconds, dtype=float))) if recent_iter_seconds else float("nan")
            )
            finished_studies = 0
            completed_study_waits: list[float] = []
            joint_runs: list[dict] = []
            for done_task in asyncio.as_completed(joint_tasks):
                try:
                    island_idx, tag, vec, jr = await done_task
                    jr["_island_idx"] = int(island_idx)
                    jr["_tag"] = str(tag)
                    joint_runs.append(jr)
                    finished_studies += 1

                    wait_s = float(jr.get("wait_seconds", float("nan")))
                    if math.isfinite(wait_s):
                        completed_study_waits.append(wait_s)
                    study_ref_s = (
                        float(statistics.median(completed_study_waits))
                        if completed_study_waits
                        else (float(study_wall_ewma) if math.isfinite(study_wall_ewma) else float("nan"))
                    )
                    studies_left = max(0, len(joint_tasks) - finished_studies)
                    iter_remaining_s = (
                        float(study_ref_s * (float(studies_left) / float(max(1, launch_count))))
                        if math.isfinite(study_ref_s)
                        else float("nan")
                    )
                    iterations_remaining = max(0, int(total_iterations - iteration))
                    iter_ref_s = float(mean_iter_seconds)
                    if not math.isfinite(iter_ref_s):
                        if math.isfinite(study_ref_s):
                            iter_ref_s = float(max(30.0, study_ref_s))
                        elif math.isfinite(iteration_eta_seconds):
                            iter_ref_s = float(iteration_eta_seconds)
                    total_eta_s = (
                        float(iter_remaining_s + float(iterations_remaining) * iter_ref_s)
                        if math.isfinite(iter_remaining_s) and math.isfinite(iter_ref_s)
                        else float("nan")
                    )
                    print(
                        f"[it {iteration:03d} {tag}] study finished ({finished_studies}/{len(joint_tasks)}); "
                        f"iterations_remaining={iterations_remaining}; "
                        f"total_eta~{raceeng_format_eta(total_eta_s)}"
                    )
                except Exception as ex:
                    print(f"Joint run failed in iteration {iteration}: {type(ex).__name__}: {ex}")
            if not joint_runs:
                raise RuntimeError(f"All joint runs failed in iteration {iteration}")

            point_results = []
            for jr in joint_runs:
                point_results.extend(list(jr.get("point_results", [])))
            parameter_results = dict(joint_runs[0]["parameter_results"])
            if not point_results:
                raise RuntimeError("No sweep points executed in joint exploration")

            joint_study_ids = ",".join(str(jr.get("study_id", "")) for jr in joint_runs)
            joint_exploration_ids = ",".join(str(jr.get("exploration_id", "")) for jr in joint_runs)
            joint_duration_seconds = float(
                np.nanmax(np.array([float(jr.get("duration_seconds", float("nan"))) for jr in joint_runs], dtype=float))
            )
            joint_wait_seconds = float(
                np.nanmax(np.array([float(jr.get("wait_seconds", float("nan"))) for jr in joint_runs], dtype=float))
            )
            joint_completed_job_count = int(sum(int(jr.get("completed_job_count", 0)) for jr in joint_runs))
            joint_job_count = int(sum(int(jr.get("job_count", 0)) for jr in joint_runs))
            joint_succeeded_simulation_count = int(sum(int(jr.get("succeeded_simulation_count", 0)) for jr in joint_runs))
            joint_is_complete = bool(all(bool(jr.get("is_complete", False)) for jr in joint_runs))

            noise_sigma, noise_groups = raceeng_estimate_noise_sigma(point_results, spec_names)
            improvement_threshold = float(max(0.0020, 2.0 * noise_sigma, 0.50 * calibration_mae))
            noise_rows.append(
                {
                    "iteration": int(iteration),
                    "noise_sigma_tLapTotal": float(noise_sigma),
                    "noise_group_count": int(noise_groups),
                    "calibration_bias": float(calibration_bias),
                    "calibration_mae": float(calibration_mae),
                    "improvement_threshold": float(improvement_threshold),
                }
            )
            rolling_noise_sigma = raceeng_update_ewma(rolling_noise_sigma, float(noise_sigma), alpha=0.30)

            gradients: dict[str, dict[str, float]] = {}
            for spec in specs:
                gradients[spec.name] = dict(
                    parameter_results.get(spec.name, {}).get("gradient", {m: 0.0 for m in RACEENG_METRICS})
                )

            best_sweep_legal_metrics = None
            best_sweep_legal_vector = None
            hard_legal_point_count = 0
            soft_legal_point_count = 0
            finite_point_count = 0
            objective_rows_by_param: dict[str, list[float]] = {s.name: [] for s in specs}
            X_rows = []
            y_obj = []
            y_cons: dict[str, list[float]] = {c.name: [] for c in RACEENG_CONSTRAINTS}

            for point in point_results:
                spec_name = str(point["parameter"])
                point_value = float(point["value"]) if point["value"] is not None else float("nan")
                point_metrics = dict(point["metrics"])
                point_legal = raceeng_is_legal(point_metrics)
                point_soft_legal = raceeng_is_legal_with_bounds(point_metrics, soft_bounds)
                point_violation = raceeng_total_violation(point_metrics)
                point_soft_violation = raceeng_total_violation_with_bounds(point_metrics, soft_bounds)

                point_vector = dict(point.get("vector") or current_vector)
                point_job_id = point.get("job_id")
                if point_job_id and str(point_job_id) in seen_job_ids:
                    continue
                if point_job_id:
                    seen_job_ids.add(str(point_job_id))

                if math.isfinite(float(point_metrics.get(RACEENG_OBJECTIVE, float("nan")))):
                    finite_point_count += 1
                    y_obj.append(float(point_metrics[RACEENG_OBJECTIVE]))
                    X_rows.append([float(point_vector[s.name]) for s in specs])
                    for c in RACEENG_CONSTRAINTS:
                        y_cons[c.name].append(float(point_metrics.get(c.name, float("nan"))))
                    for s in specs:
                        objective_rows_by_param[s.name].append(float(point_metrics[RACEENG_OBJECTIVE]))
                    history_X.append([float(point_vector[s.name]) for s in specs])
                    history_obj.append(float(point_metrics[RACEENG_OBJECTIVE]))
                    for c in RACEENG_CONSTRAINTS:
                        history_cons[c.name].append(float(point_metrics.get(c.name, float("nan"))))
                if point_legal:
                    hard_legal_point_count += 1
                if point_soft_legal:
                    soft_legal_point_count += 1

                row = {
                    "iteration": iteration,
                    "parameter": spec_name,
                    "point": f"p{int(point['local_point_index']) + 1}",
                    "value": point_value,
                    "study_id": joint_study_ids,
                    "exploration_id": joint_exploration_ids,
                    "job_id": point["job_id"],
                    "legal": bool(point_legal),
                    "hard_legal": bool(point_legal),
                    "soft_legal": bool(point_soft_legal),
                    "total_violation": float(point_violation),
                    "soft_violation": float(point_soft_violation),
                    "point_index": int(point["point_index"]),
                    "local_point_index": int(point["local_point_index"]),
                    "active": True,
                    "strategy": str(point.get("strategy", "batch")),
                }
                row.update({k: float(point_metrics.get(k, float("nan"))) for k in RACEENG_METRICS})
                row.update({f"param_{k}": float(v) for k, v in point_vector.items()})
                sweep_rows.append(row)

                if point_legal:
                    if best_legal_metrics is None or float(point_metrics[RACEENG_OBJECTIVE]) < float(
                        best_legal_metrics[RACEENG_OBJECTIVE]
                    ):
                        best_legal_metrics = dict(point_metrics)
                        best_legal_vector = dict(point_vector)
                    if best_sweep_legal_metrics is None or float(point_metrics[RACEENG_OBJECTIVE]) < float(
                        best_sweep_legal_metrics[RACEENG_OBJECTIVE]
                    ):
                        best_sweep_legal_metrics = dict(point_metrics)
                        best_sweep_legal_vector = dict(point_vector)
                    elite_archive = raceeng_update_elite_archive(
                        elite_archive,
                        raceeng_make_archive_entry(
                            specs=specs,
                            vector=point_vector,
                            metrics=point_metrics,
                            source_phase="main",
                            source_iteration=int(iteration),
                            source_label=str(point_job_id or f"it{iteration:02d}_main"),
                            verified=False,
                        ),
                        specs=specs,
                        max_size=elite_archive_size,
                    )

            X_arr = np.array(history_X, dtype=float) if history_X else np.zeros((0, len(specs)), dtype=float)
            y_obj_arr = np.array(history_obj, dtype=float) if history_obj else np.zeros(0, dtype=float)
            obj_model = raceeng_fit_ridge_linear(X_arr, y_obj_arr)
            constraint_models = {
                c.name: raceeng_fit_ridge_linear(X_arr, np.array(history_cons[c.name], dtype=float))
                for c in RACEENG_CONSTRAINTS
            }
            if obj_model is not None and X_rows:
                X_recent = np.array(X_rows, dtype=float)
                y_recent = np.array(y_obj, dtype=float)
                y_pred_recent, _ = raceeng_predict_ridge_linear(obj_model, X_recent)
                mask_recent = np.isfinite(y_recent) & np.isfinite(y_pred_recent)
                if np.any(mask_recent):
                    residuals_recent = (y_recent[mask_recent] - y_pred_recent[mask_recent]).tolist()
                    calibration_residuals.extend([float(r) for r in residuals_recent])
                    calibration_residuals = calibration_residuals[-500:]
                    calibration_bias = float(np.median(np.array(calibration_residuals, dtype=float)))
                    calibration_mae = float(np.median(np.abs(np.array(calibration_residuals, dtype=float))))

            surrogate_rows.append(
                {
                    "iteration": int(iteration),
                    "metric": RACEENG_OBJECTIVE,
                    "train_count": 0 if obj_model is None else int(obj_model["train_count"]),
                    "r2": float("nan") if obj_model is None else float(obj_model["r2"]),
                    "resid_std": float("nan") if obj_model is None else float(obj_model["resid_std"]),
                }
            )
            for c in RACEENG_CONSTRAINTS:
                model = constraint_models[c.name]
                surrogate_rows.append(
                    {
                        "iteration": int(iteration),
                        "metric": c.name,
                        "train_count": 0 if model is None else int(model["train_count"]),
                        "r2": float("nan") if model is None else float(model["r2"]),
                        "resid_std": float("nan") if model is None else float(model["resid_std"]),
                    }
                )
            obj_train_count = 0 if obj_model is None else int(obj_model["train_count"])
            obj_r2 = float("nan") if obj_model is None else float(obj_model["r2"])
            obj_resid_std = float("nan") if obj_model is None else float(obj_model["resid_std"])

            coef_obj = np.zeros(len(specs), dtype=float)
            if obj_model is not None:
                beta = np.array(obj_model["beta"], dtype=float)
                sigma = np.array(obj_model["x_sigma"], dtype=float)
                if beta.size >= len(specs) + 1:
                    coef_obj = beta[1 : 1 + len(specs)] / sigma

            for spec in active_specs:
                summary = parameter_results.get(spec.name)
                if summary is None:
                    continue
                idx = spec_names.index(spec.name)
                effect = abs(float(coef_obj[idx])) * float(spec.delta)
                prev_scale = float(mutation_scales.get(spec.name, 1.0))
                target_scale = 0.90 if effect > float(np.nanmedian(np.abs(coef_obj) * np.array([s.delta for s in specs]))) else 1.20
                if spec.name.startswith("front_r") or spec.name.startswith("rear_r"):
                    target_scale *= 1.25
                mutation_scales[spec.name] = float(max(0.55, min(4.00, 0.70 * prev_scale + 0.30 * target_scale)))

                sens_row = {
                    "iteration": iteration,
                    "parameter": spec.name,
                    "center": float(summary["center"]),
                    "low": float(summary["low"]),
                    "high": float(summary["high"]),
                    "n_points": int(summary["n_points"]),
                    "n_finite_objective": int(summary["n_finite_objective"]),
                    "study_id": summary["study_id"],
                    "exploration_id": summary["exploration_id"],
                    "effect_score": float(effect),
                    "mutation_scale": float(mutation_scales[spec.name]),
                }
                for m in RACEENG_METRICS:
                    sens_row[f"d_{m}_d_{spec.name}"] = float(summary["gradient"].get(m, 0.0))
                sensitivity_rows.append(sens_row)

            next_focus_scores: dict[str, float] = {}
            for spec in specs:
                grad_obj = abs(float(gradients.get(spec.name, {}).get(RACEENG_OBJECTIVE, 0.0))) * max(1e-9, spec.delta)
                param_objectives = objective_rows_by_param.get(spec.name) or [float("inf")]
                best_param_lap = min(param_objectives)
                improvement = 0.0 if not math.isfinite(best_param_lap) else max(0.0, current_lap - best_param_lap)
                base_score = 0.65 * grad_obj + 0.35 * improvement
                carry = 0.35 * float(focus_scores.get(spec.name, 0.0))
                next_focus_scores[spec.name] = float(base_score + carry)
            focus_scores = next_focus_scores

            # Select next incumbent directly from legal points in the batch.
            prev_global_vector = dict(current_vector)
            decision = "reject_batch"
            step_vector = {s.name: 0.0 for s in specs}
            improved_islands = 0
            islands_with_no_legal = 0

            for jr in joint_runs:
                island_idx = int(jr.get("_island_idx", 0))
                island = islands[island_idx]
                island_points = list(jr.get("point_results", []))
                island_legal = [
                    p
                    for p in island_points
                    if bool(raceeng_is_legal(dict(p.get("metrics") or {})))
                    and math.isfinite(float((p.get("metrics") or {}).get(RACEENG_OBJECTIVE, float("nan"))))
                ]
                if not island_legal:
                    islands_with_no_legal += 1
                    island["trust_radius"] = float(min(3.2, float(island["trust_radius"]) * 1.10))
                    continue
                island_best = min(
                    island_legal,
                    key=lambda p: float((p.get("metrics") or {}).get(RACEENG_OBJECTIVE, float("inf"))),
                )
                island_best_lap = float((island_best.get("metrics") or {}).get(RACEENG_OBJECTIVE, float("inf")))
                island_current_lap = float((island.get("metrics") or {}).get(RACEENG_OBJECTIVE, float("inf")))
                island_current_legal = raceeng_is_legal(dict(island.get("metrics") or {}))
                if (not island_current_legal) or (island_best_lap < island_current_lap - improvement_threshold):
                    island["vector"] = dict(island_best.get("vector") or island["vector"])
                    island["metrics"] = dict(island_best.get("metrics") or island["metrics"])
                    island["trust_radius"] = float(min(2.8, float(island["trust_radius"]) * 1.06))
                    improved_islands += 1
                else:
                    island["trust_radius"] = float(max(0.35, float(island["trust_radius"]) * 0.88))

                # Keep island mutation scales in sync with learned global scaling trend.
                island_mut = dict(island.get("mutation_scales") or {})
                for name, val in mutation_scales.items():
                    prev_val = float(island_mut.get(name, val))
                    island_mut[name] = float(max(0.55, min(4.0, 0.75 * prev_val + 0.25 * float(val))))
                island["mutation_scales"] = island_mut

            accepted_metrics = None
            accepted_vector = None
            if best_sweep_legal_metrics is not None:
                best_lap = float(best_sweep_legal_metrics[RACEENG_OBJECTIVE])
                if (not current_legal) or (best_lap < current_lap - improvement_threshold):
                    accepted_metrics = dict(best_sweep_legal_metrics)
                    accepted_vector = dict(best_sweep_legal_vector)

            if accepted_vector is not None:
                for s in specs:
                    step_vector[s.name] = float(accepted_vector[s.name] - prev_global_vector[s.name])
                current_vector = dict(accepted_vector)
                current_metrics = dict(accepted_metrics)
                current_car_id, _ = await raceeng_materialize_vector_car(
                    session=session,
                    base_car_payload=car_payload,
                    specs=specs,
                    vector=current_vector,
                    name_prefix=f"{iter_row_prefix}-accepted",
                    runtime=runtime,
                )
                current_eval = {
                    "study_id": joint_study_ids,
                    "car_id": current_car_id,
                    "job_id": "__batch__",
                    "metrics": current_metrics,
                    "duration_seconds": float(joint_duration_seconds),
                    "succeeded_simulation_count": int(joint_succeeded_simulation_count),
                }
                decision = "accept_batch_legal"
            elif islands_with_no_legal >= len(joint_runs):
                decision = "reject_no_hard_legal_in_batch"
            else:
                decision = "reject_batch"

            trust_radius = float(np.mean(np.array([float(i["trust_radius"]) for i in islands], dtype=float)))
            batch_best_legal_lap = (
                float("nan") if best_sweep_legal_metrics is None else float(best_sweep_legal_metrics[RACEENG_OBJECTIVE])
            )
            batch_improvement = (
                0.0
                if not math.isfinite(batch_best_legal_lap)
                else float(max(0.0, float(current_lap) - float(batch_best_legal_lap)))
            )
            hard_legal_yield = float(hard_legal_point_count) / float(max(1, len(point_results)))
            soft_legal_yield = float(soft_legal_point_count) / float(max(1, len(point_results)))
            finite_yield = float(finite_point_count) / float(max(1, len(point_results)))
            no_improve_streak = 0 if decision == "accept_batch_legal" else int(no_improve_streak + 1)
            island_snapshots: list[str] = []
            island_best_laps: list[float] = []
            for island in islands:
                im = dict(island.get("metrics") or {})
                ilap = float(im.get(RACEENG_OBJECTIVE, float("nan")))
                island_best_laps.append(ilap)
                ilap_txt = "nan" if not math.isfinite(ilap) else f"{ilap:.6f}"
                island_snapshots.append(
                    f"is{int(island.get('id', 0)):02d}:lap={ilap_txt},legal={raceeng_is_legal(im)},tr={float(island.get('trust_radius', float('nan'))):.2f}"
                )
            if island_snapshots:
                print("Island states: " + " | ".join(island_snapshots))

            # Partially harvested studies stay in the persistent pending registry and are recovered later.
            if not joint_is_complete:
                for jr in joint_runs:
                    if bool(jr.get("is_complete", False)):
                        continue
                    print(
                        f"Deferred completion of partial study {jr['study_id']} "
                        f"for later recovery (launch_id={jr.get('launch_id')})."
                    )

            iteration_wall_seconds = float(perf_counter() - iter_start)
            prev_study_wall_ewma = float(study_wall_ewma)
            wait_sample = float(joint_wait_seconds) if math.isfinite(float(joint_wait_seconds)) else float(joint_duration_seconds)
            if math.isfinite(wait_sample):
                if math.isfinite(prev_study_wall_ewma):
                    study_wall_ewma = float(0.72 * prev_study_wall_ewma + 0.28 * wait_sample)
                else:
                    study_wall_ewma = float(wait_sample)

            # Adaptive controller for concurrency.
            prev_concurrent_starts = int(concurrent_starts)
            concurrency_reason = "hold"
            if force_two_main_starts:
                concurrent_starts = int(min(2, island_count))
                concurrency_cooldown = 0
                concurrency_reason = "force_two_main_starts"
            else:
                if concurrency_cooldown > 0:
                    concurrency_cooldown = int(concurrency_cooldown - 1)
                if concurrency_cooldown == 0 and math.isfinite(wait_sample) and math.isfinite(prev_study_wall_ewma):
                    if wait_sample > float(1.30 * prev_study_wall_ewma) and concurrent_starts > 1:
                        concurrent_starts = int(concurrent_starts - 1)
                        concurrency_reason = "backoff_slow_studies"
                        concurrency_cooldown = 1
                    elif (
                        wait_sample < float(0.82 * prev_study_wall_ewma)
                        and hard_legal_yield >= 0.06
                        and no_improve_streak >= 1
                        and concurrent_starts < island_count
                    ):
                        concurrent_starts = int(concurrent_starts + 1)
                        concurrency_reason = "expand_fast_studies"
                        concurrency_cooldown = 1
                concurrent_starts = int(max(1, min(island_count, concurrent_starts)))

            # Adaptive controller for per-parameter point density.
            prev_batch_points = int(batch_points)
            batch_points_reason = "hold"
            if hard_legal_yield < 0.025 and batch_points < 10:
                batch_points = int(batch_points + 1)
                batch_points_reason = "raise_low_legal_yield"
            elif no_improve_streak >= 2 and batch_points < 10:
                batch_points = int(batch_points + 1)
                batch_points_reason = "raise_after_stall"
            elif (
                decision == "accept_batch_legal"
                and batch_improvement > float(2.50 * improvement_threshold)
                and hard_legal_yield > 0.10
                and batch_points > 4
            ):
                batch_points = int(batch_points - 1)
                batch_points_reason = "lower_after_strong_gain"
            elif decision == "accept_batch_legal" and hard_legal_yield > 0.25 and batch_points > 5:
                batch_points = int(batch_points - 1)
                batch_points_reason = "lower_high_legal_yield"
            batch_points = int(max(4, min(10, batch_points)))

            eta_error_seconds = (
                float("nan")
                if not math.isfinite(iteration_eta_seconds)
                else float(iteration_wall_seconds - float(iteration_eta_seconds))
            )
            print(
                "Iteration summary: "
                f"wall={iteration_wall_seconds/60.0:.1f}m; "
                f"best_delta={batch_improvement:.5f}; "
                f"hard_legal_yield={hard_legal_yield:.1%}; "
                f"next_starts={concurrent_starts} ({concurrency_reason}); "
                f"next_batch_points={batch_points} ({batch_points_reason})"
            )
            best_legal_lap_so_far = float("nan")
            if best_legal_metrics is not None:
                best_legal_lap_so_far = float(best_legal_metrics.get(RACEENG_OBJECTIVE, float("nan")))
            best_gain_vs_baseline = (
                float("nan")
                if (not math.isfinite(best_legal_lap_so_far) or not math.isfinite(baseline_lap))
                else float(baseline_lap - best_legal_lap_so_far)
            )
            finite_island_laps = [float(v) for v in island_best_laps if math.isfinite(float(v))]
            island_lap_min = float(min(finite_island_laps)) if finite_island_laps else float("nan")
            island_lap_med = float(statistics.median(finite_island_laps)) if finite_island_laps else float("nan")
            pending_study_count = int(len(runtime.pending_studies))
            print(
                "Model/queue: "
                f"obj_r2={obj_r2:.3f} obj_resid={obj_resid_std:.5f} train={obj_train_count}; "
                f"pending_studies={pending_study_count}; "
                f"best_legal_gain_vs_baseline={best_gain_vs_baseline:+.6f}"
            )

            timing_rows.append(
                {
                    "phase": "main",
                    "iteration": int(iteration),
                    "joint_duration_seconds": float(joint_duration_seconds),
                    "partial_wait_seconds": float(joint_wait_seconds),
                    "iteration_wall_seconds": float(iteration_wall_seconds),
                    "joint_point_count": int(len(point_results)),
                    "joint_completed_job_count": int(joint_completed_job_count),
                    "joint_job_count": int(joint_job_count),
                    "joint_succeeded_simulation_count": int(joint_succeeded_simulation_count),
                    "is_complete_at_partial": bool(joint_is_complete),
                    "hard_legal_point_count": int(hard_legal_point_count),
                    "soft_legal_point_count": int(soft_legal_point_count),
                    "finite_point_count": int(finite_point_count),
                    "hard_legal_yield": float(hard_legal_yield),
                    "soft_legal_yield": float(soft_legal_yield),
                    "finite_yield": float(finite_yield),
                    "iteration_eta_seconds": float(iteration_eta_seconds),
                    "campaign_eta_seconds": float(campaign_eta_seconds),
                    "eta_error_seconds": float(eta_error_seconds),
                    "study_wait_ewma_seconds": float(study_wall_ewma),
                    "concurrent_starts": int(launch_count),
                    "batch_points": int(prev_batch_points),
                    "best_legal_lap_so_far": float(best_legal_lap_so_far),
                    "best_gain_vs_baseline": float(best_gain_vs_baseline),
                    "obj_model_train_count": int(obj_train_count),
                    "obj_model_r2": float(obj_r2),
                    "obj_model_resid_std": float(obj_resid_std),
                    "pending_study_count": int(pending_study_count),
                    "island_lap_min": float(island_lap_min),
                    "island_lap_median": float(island_lap_med),
                }
            )
            controller_rows.append(
                {
                    "phase": "main",
                    "iteration": int(iteration),
                    "concurrent_starts_prev": int(prev_concurrent_starts),
                    "concurrent_starts_next": int(concurrent_starts),
                    "concurrency_reason": str(concurrency_reason),
                    "batch_points_prev": int(prev_batch_points),
                    "batch_points_next": int(batch_points),
                    "batch_points_reason": str(batch_points_reason),
                    "hard_legal_yield": float(hard_legal_yield),
                    "soft_legal_yield": float(soft_legal_yield),
                    "finite_yield": float(finite_yield),
                    "batch_improvement": float(batch_improvement),
                    "no_improve_streak": int(no_improve_streak),
                    "study_wait_ewma_seconds": float(study_wall_ewma),
                    "last_wait_sample_seconds": float(wait_sample),
                    "iteration_wall_seconds": float(iteration_wall_seconds),
                    "best_legal_lap_so_far": float(best_legal_lap_so_far),
                    "best_gain_vs_baseline": float(best_gain_vs_baseline),
                    "obj_model_train_count": int(obj_train_count),
                    "obj_model_r2": float(obj_r2),
                    "obj_model_resid_std": float(obj_resid_std),
                    "pending_study_count": int(pending_study_count),
                    "island_lap_min": float(island_lap_min),
                    "island_lap_median": float(island_lap_med),
                }
            )

            iter_row = {
                "phase": "main",
                "iteration": int(iteration),
                "decision": str(decision),
                "trust_radius_next": float(trust_radius),
                "hardness": float(hardness),
                "noise_sigma_tlap": float(noise_sigma),
                "calibration_bias": float(calibration_bias),
                "calibration_mae": float(calibration_mae),
                "improvement_threshold": float(improvement_threshold),
                "current_lap_before": float(current_lap),
                "current_legal_before": bool(current_legal),
                "batch_best_legal_lap": float(batch_best_legal_lap),
                "batch_improvement": float(batch_improvement),
                "hard_legal_yield": float(hard_legal_yield),
                "soft_legal_yield": float(soft_legal_yield),
                "finite_yield": float(finite_yield),
                "hard_legal_point_count": int(hard_legal_point_count),
                "soft_legal_point_count": int(soft_legal_point_count),
                "finite_point_count": int(finite_point_count),
                "joint_sweep_study_id": joint_study_ids,
                "joint_sweep_exploration_id": joint_exploration_ids,
                "joint_sweep_point_count": int(len(point_results)),
                "joint_completed_job_count": int(joint_completed_job_count),
                "joint_job_count": int(joint_job_count),
                "is_complete_at_partial": bool(joint_is_complete),
                "joint_duration_seconds": float(joint_duration_seconds),
                "partial_wait_seconds": float(joint_wait_seconds),
                "iteration_wall_seconds": float(iteration_wall_seconds),
                "iteration_eta_seconds": float(iteration_eta_seconds),
                "campaign_eta_seconds": float(campaign_eta_seconds),
                "eta_error_seconds": float(eta_error_seconds),
                "study_wait_ewma_seconds": float(study_wall_ewma),
                "concurrent_starts_used": int(launch_count),
                "concurrent_starts_next": int(concurrent_starts),
                "batch_points_used": int(prev_batch_points),
                "batch_points_next": int(batch_points),
                "concurrency_reason": str(concurrency_reason),
                "batch_points_reason": str(batch_points_reason),
                "no_improve_streak": int(no_improve_streak),
                "improved_islands": int(improved_islands),
                "islands_with_no_legal": int(islands_with_no_legal),
                "best_legal_lap_so_far": float(best_legal_lap_so_far),
                "best_gain_vs_baseline": float(best_gain_vs_baseline),
                "obj_model_train_count": int(obj_train_count),
                "obj_model_r2": float(obj_r2),
                "obj_model_resid_std": float(obj_resid_std),
                "pending_study_count": int(pending_study_count),
                "island_lap_min": float(island_lap_min),
                "island_lap_median": float(island_lap_med),
            }
            for p, step in step_vector.items():
                iter_row[f"step_{p}"] = float(step)
            iteration_rows.append(iter_row)
            checkpoint_row = {
                "phase": "main",
                "iteration": int(iteration),
                "timestamp_utc": datetime.now(UTC).isoformat(),
                "current_lap": float(current_metrics.get(RACEENG_OBJECTIVE, float("nan"))),
                "best_legal_lap": float(
                    best_legal_metrics.get(RACEENG_OBJECTIVE, float("nan")) if best_legal_metrics is not None else float("nan")
                ),
                "best_verified_lap": float(best_verified_lap),
                "rolling_noise_sigma": float(rolling_noise_sigma),
                "endgame_stall": int(endgame_no_improve_streak),
                "extension_cycles_used": int(extension_cycles_used),
                "row_prefix": str(row_prefix),
            }
            checkpoint_rows.append(checkpoint_row)
            last_clean_iteration = int(iteration)
            campaign_status = "running"
            _write_checkpoint(
                phase="main",
                iteration=int(iteration),
                current_metrics_local=current_metrics,
                current_eval_local=current_eval,
            )

        # Dedicated endgame exploitation around incumbent using stochastic ES.
        endgame_cycle = int(max(0, max(start_iteration, main_iterations + 1) - main_iterations - 1))
        endgame_termination_reason = "completed_nominal"
        endgame_extension_allowed = bool(global_iteration_end >= total_iterations)
        while True:
            current_best_archive_lap = float(raceeng_best_archive_legal_lap(elite_archive))
            verification_debt = bool(
                raceeng_has_verification_debt(
                    elite_archive,
                    best_verified_lap=best_verified_lap,
                    threshold=endgame_verification_debt_threshold,
                )
            )
            next_cycle = int(endgame_cycle + 1)
            iteration = int(main_iterations + next_cycle)
            in_extension = bool(next_cycle > nominal_endgame_cycles)
            if iteration < max(start_iteration, main_iterations + 1):
                endgame_cycle = next_cycle
                continue
            if not in_extension:
                if iteration > global_iteration_end:
                    break
            else:
                if (
                    (not endgame_extension_allowed)
                    or next_cycle > nominal_endgame_cycles + endgame_extra_cycles_max + (3 if verification_debt else 0)
                    or (endgame_no_improve_streak >= endgame_stall_patience and not verification_debt)
                ):
                    break
            endgame_cycle = next_cycle
            extension_cycles_used = int(max(0, endgame_cycle - nominal_endgame_cycles))

            iter_start = perf_counter()
            iter_row_prefix = f"{row_prefix}-it{iteration:02d}"
            profile = raceeng_constraint_profile(total_iterations, total_iterations)
            soft_bounds = dict(profile["bounds"])
            hardness = 1.0
            for row in profile["rows"]:
                rr = {"iteration": int(iteration), "phase": "endgame"}
                rr.update(row)
                constraint_rows.append(rr)

            pending_recovery = await raceeng_recover_pending_studies(
                session=session,
                tenant_id=tenant_id,
                specs=specs,
                runtime=runtime,
            )
            merged_pending_points = _merge_recovered_joint_runs(list(pending_recovery.get("joint_runs", [])))
            if merged_pending_points > 0:
                obj_model, constraint_models = raceeng_fit_surrogates_from_history(specs, history_X, history_obj, history_cons)
                print(f"Merged {merged_pending_points} recovered pending points before endgame iteration {iteration}.")

            print(
                f"\n========== Iteration {iteration} [phase=endgame"
                f"{'+ext' if in_extension else ''}] =========="
            )
            print(
                f"Endgame state: cycle={endgame_cycle}, extension={extension_cycles_used}, "
                f"stall={endgame_no_improve_streak}/{endgame_stall_patience}, sigma_mult={endgame_sigma_mult:.3f}, "
                f"best_archive_legal={'nan' if not math.isfinite(current_best_archive_lap) else f'{current_best_archive_lap:.6f}'}, "
                f"verification_debt={verification_debt}"
            )

            active_specs_end = raceeng_select_endgame_active_specs(
                specs=specs,
                focus_scores=focus_scores,
                obj_model=obj_model,
                fraction=0.45,
                min_keep=18,
            )
            active_end_names = {s.name for s in active_specs_end}
            print(
                f"Endgame active parameters ({len(active_specs_end)}/{len(specs)}): "
                + ", ".join(s.name for s in active_specs_end)
            )

            archive_unverified_before = raceeng_best_archive_entry(elite_archive, require_unverified=True)
            archive_unverified_before_lap = (
                float(archive_unverified_before.get("lap", float("nan")))
                if isinstance(archive_unverified_before, dict)
                else float("nan")
            )
            challenger_entry = None
            if verification_debt or endgame_no_improve_streak >= 2:
                challenger_entry = raceeng_best_archive_entry(elite_archive, require_unverified=True)
                if challenger_entry is None:
                    challenger_entry = raceeng_best_archive_entry(elite_archive, require_unverified=False)

            lane_specs: list[tuple[str, str, dict[str, float]]] = [("is01", "incumbent", dict(current_vector))]
            if isinstance(challenger_entry, dict):
                challenger_vector = dict(challenger_entry.get("vector") or {})
                if raceeng_vector_distance(specs, challenger_vector, current_vector) > 0.20:
                    lane_specs.append(("is02", "challenger", challenger_vector))

            async def _run_endgame_lane(tag: str, lane_role: str, lane_vector: dict[str, float]) -> dict:
                lane_point_defs = raceeng_build_es_point_definitions(
                    specs=specs,
                    incumbent_vector=lane_vector,
                    active_specs=active_specs_end,
                    trust_radius=max(0.35, float(trust_radius) * 0.85),
                    sigma_mult=float(endgame_sigma_mult),
                    mutation_scales=mutation_scales,
                    rng=rng,
                    points_target=None,
                    obj_model=obj_model,
                    constraint_models=constraint_models,
                    soft_bounds=soft_bounds,
                )
                lane_run = await raceeng_run_joint_parameter_exploration(
                    session=session,
                    tenant_id=tenant_id,
                    worksheet_id=worksheet_id,
                    row_name=f"{iter_row_prefix}-{tag}-jointSweep",
                    static_config_ids=static_config_ids,
                    base_car_id=current_eval["car_id"],
                    current_car_config=apply_raceeng_vector(car_payload, specs, lane_vector)["config"],
                    specs=specs,
                    active_specs=active_specs_end,
                    current_vector=lane_vector,
                    timeout_seconds=timeout_seconds,
                    points_per_parameter=int(batch_points),
                    trust_radius=max(0.35, float(trust_radius) * 0.85),
                    mutation_scales=mutation_scales,
                    rng=rng,
                    current_repeat_count=3,
                    obj_model=obj_model,
                    constraint_models=constraint_models,
                    soft_bounds=soft_bounds,
                    hardness=1.0,
                    partial_min_completed_fraction=0.80,
                    partial_min_completed_jobs=96,
                    partial_min_succeeded_jobs=72,
                    partial_poll_seconds=8.0,
                    progress_label="",
                    point_defs_override=lane_point_defs,
                    runtime=runtime,
                    phase="endgame",
                    iteration=iteration,
                )
                lane_run["_lane_tag"] = str(tag)
                lane_run["_lane_role"] = str(lane_role)
                lane_run["_lane_vector"] = dict(lane_vector)
                return lane_run

            es_tasks = [
                asyncio.create_task(_run_endgame_lane(tag, lane_role, lane_vector))
                for tag, lane_role, lane_vector in lane_specs
            ]
            lane_runs: list[dict] = []
            for result in await asyncio.gather(*es_tasks, return_exceptions=True):
                if isinstance(result, Exception):
                    print(f"Endgame lane failed: {type(result).__name__}: {result}")
                    continue
                lane_runs.append(dict(result))

            point_results: list[dict] = []
            for lane_run in lane_runs:
                if not bool(lane_run.get("is_complete", False)):
                    print(
                        f"Deferred completion of partial endgame study {lane_run['study_id']} "
                        f"for later recovery (launch_id={lane_run.get('launch_id')})."
                    )
                for point in list(lane_run.get("point_results", [])):
                    row_point = dict(point)
                    row_point["_study_id"] = str(lane_run.get("study_id", ""))
                    row_point["_exploration_id"] = str(lane_run.get("exploration_id", ""))
                    row_point["_lane_tag"] = str(lane_run.get("_lane_tag", "is01"))
                    row_point["_lane_role"] = str(lane_run.get("_lane_role", "incumbent"))
                    point_results.append(row_point)
            if not point_results:
                print("Endgame ES returned zero points; stopping endgame early.")
                endgame_termination_reason = "no_points"
                break

            hard_legal_point_count = 0
            soft_legal_point_count = 0
            finite_point_count = 0
            legal_candidates: list[dict] = []
            for point in point_results:
                point_metrics = dict(point.get("metrics") or {})
                point_vector = dict(point.get("vector") or current_vector)
                point_job_id = point.get("job_id")
                if point_job_id and str(point_job_id) in seen_job_ids:
                    continue
                if point_job_id:
                    seen_job_ids.add(str(point_job_id))

                point_lap = float(point_metrics.get(RACEENG_OBJECTIVE, float("nan")))
                point_legal = bool(raceeng_is_legal(point_metrics))
                point_soft_legal = bool(raceeng_is_legal_with_bounds(point_metrics, soft_bounds))
                if point_legal:
                    hard_legal_point_count += 1
                if point_soft_legal:
                    soft_legal_point_count += 1
                if math.isfinite(point_lap):
                    finite_point_count += 1
                    history_X.append([float(point_vector[s.name]) for s in specs])
                    history_obj.append(float(point_lap))
                    for c in RACEENG_CONSTRAINTS:
                        history_cons[c.name].append(float(point_metrics.get(c.name, float("nan"))))
                    if point_legal:
                        legal_candidates.append(
                            {
                                "lap": float(point_lap),
                                "vector": dict(point_vector),
                                "metrics": dict(point_metrics),
                                "job_id": point.get("job_id"),
                                "lane_role": str(point.get("_lane_role", "incumbent")),
                                "lane_tag": str(point.get("_lane_tag", "is01")),
                            }
                        )
                        if best_legal_metrics is None or float(point_lap) < float(best_legal_metrics[RACEENG_OBJECTIVE]):
                            best_legal_metrics = dict(point_metrics)
                            best_legal_vector = dict(point_vector)
                        elite_archive = raceeng_update_elite_archive(
                            elite_archive,
                            raceeng_make_archive_entry(
                                specs=specs,
                                vector=point_vector,
                                metrics=point_metrics,
                                source_phase="endgame",
                                source_iteration=int(iteration),
                                source_label=str(point.get("job_id") or point.get("_lane_role") or "endgame_point"),
                                verified=False,
                            ),
                            specs=specs,
                            max_size=elite_archive_size,
                        )

                row = {
                    "phase": "endgame",
                    "iteration": int(iteration),
                    "parameter": str(point.get("parameter", "__es__")),
                    "point": f"p{int(point.get('local_point_index', 0)) + 1}",
                    "value": float(point.get("value", float("nan"))),
                    "study_id": str(point.get("_study_id", "")),
                    "exploration_id": str(point.get("_exploration_id", "")),
                    "job_id": point.get("job_id"),
                    "legal": bool(point_legal),
                    "hard_legal": bool(point_legal),
                    "soft_legal": bool(point_soft_legal),
                    "total_violation": float(raceeng_total_violation(point_metrics)),
                    "soft_violation": float(raceeng_total_violation_with_bounds(point_metrics, soft_bounds)),
                    "point_index": int(point.get("point_index", 0)),
                    "local_point_index": int(point.get("local_point_index", 0)),
                    "active": bool(str(point.get("parameter", "__es__")) == "__es__"),
                    "strategy": str(point.get("strategy", "es")),
                    "lane_role": str(point.get("_lane_role", "incumbent")),
                    "lane_tag": str(point.get("_lane_tag", "is01")),
                }
                row.update({k: float(point_metrics.get(k, float("nan"))) for k in RACEENG_METRICS})
                row.update({f"param_{k}": float(v) for k, v in point_vector.items()})
                sweep_rows.append(row)

            X_arr = np.array(history_X, dtype=float) if history_X else np.zeros((0, len(specs)), dtype=float)
            y_obj_arr = np.array(history_obj, dtype=float) if history_obj else np.zeros(0, dtype=float)
            obj_model = raceeng_fit_ridge_linear(X_arr, y_obj_arr)
            constraint_models = {
                c.name: raceeng_fit_ridge_linear(X_arr, np.array(history_cons[c.name], dtype=float))
                for c in RACEENG_CONSTRAINTS
            }
            obj_train_count = 0 if obj_model is None else int(obj_model["train_count"])
            obj_r2 = float("nan") if obj_model is None else float(obj_model["r2"])
            obj_resid_std = float("nan") if obj_model is None else float(obj_model["resid_std"])

            gradients_es: dict[str, dict[str, dict[str, float]]] = {}
            for lane_run in lane_runs:
                for spec_name, summary in dict(lane_run.get("parameter_results") or {}).items():
                    current_grad = dict(summary.get("gradient") or {})
                    existing = gradients_es.setdefault(
                        str(spec_name),
                        {"gradient": {m: 0.0 for m in RACEENG_METRICS}},
                    )
                    for metric_name, grad_value in current_grad.items():
                        prev_val = float(existing["gradient"].get(metric_name, 0.0))
                        grad_value = float(grad_value)
                        if abs(grad_value) >= abs(prev_val):
                            existing["gradient"][metric_name] = float(grad_value)
            for spec in specs:
                grad_obj = abs(
                    float(gradients_es.get(spec.name, {}).get("gradient", {}).get(RACEENG_OBJECTIVE, 0.0))
                ) * max(1e-9, float(spec.delta))
                carry = 0.45 * float(focus_scores.get(spec.name, 0.0))
                active_bonus = 0.15 if spec.name in active_end_names else 0.0
                focus_scores[spec.name] = float(carry + 0.55 * grad_obj + active_bonus)

            legal_candidates.sort(key=lambda r: float(r["lap"]))
            unique_candidates: list[dict] = []
            for cand in legal_candidates:
                if any(
                    raceeng_vector_distance(specs, dict(cand["vector"]), dict(existing["vector"])) <= 0.15
                    for existing in unique_candidates
                ):
                    continue
                unique_candidates.append(cand)
                if len(unique_candidates) >= endgame_verify_top_k:
                    break

            async def _verify_candidate(cand_idx: int, cand: dict) -> tuple[int, dict, dict]:
                cand_vector = dict(cand["vector"])
                cand_payload = apply_raceeng_vector(car_payload, specs, cand_vector)
                bundle = await raceeng_run_single_point_replicates(
                    session=session,
                    tenant_id=tenant_id,
                    worksheet_id=worksheet_id,
                    row_name=f"{iter_row_prefix}-verify{cand_idx:02d}",
                    static_config_ids=static_config_ids,
                    car_payload=cand_payload,
                    timeout_seconds=timeout_seconds,
                    repeat_count=3,
                    runtime=runtime,
                    phase="endgame_verify",
                    iteration=iteration,
                    bundle_role=f"verify{cand_idx:02d}",
                )
                return int(cand_idx), dict(cand_vector), dict(bundle)

            verify_results: list[tuple[int, dict, dict]] = []
            if unique_candidates:
                verify_tasks = [
                    asyncio.create_task(_verify_candidate(i + 1, cand))
                    for i, cand in enumerate(unique_candidates[:endgame_verify_top_k])
                ]
                gathered = await asyncio.gather(*verify_tasks, return_exceptions=True)
                for res in gathered:
                    if isinstance(res, Exception):
                        print(f"Endgame verification failed: {type(res).__name__}: {res}")
                        continue
                    verify_results.append(res)

            for cand_idx, cand_vector, bundle in verify_results:
                laps = [float(v) for v in bundle.get("laps", [])]
                sigma_bundle = raceeng_mad_sigma(laps)
                rolling_noise_sigma = raceeng_update_ewma(rolling_noise_sigma, sigma_bundle, alpha=0.35)
                verification_rows.append(
                    {
                        "phase": "endgame",
                        "iteration": int(iteration),
                        "endgame_cycle": int(endgame_cycle),
                        "candidate_rank": int(cand_idx),
                        "row_name": f"{iter_row_prefix}-verify{cand_idx:02d}",
                        "median_lap": float(bundle.get("median_lap", float("nan"))),
                        "median_legal": bool(bundle.get("median_legal", False)),
                        "legal_count": int(bundle.get("legal_count", 0)),
                        "repeat_count": int(bundle.get("repeat_count", 0)),
                        "noise_sigma_bundle": float(sigma_bundle),
                        "incumbent_reference_lap": float(incumbent_reference_lap),
                    }
                )
                if bool(bundle.get("median_legal", False)):
                    elite_archive = raceeng_update_elite_archive(
                        elite_archive,
                        raceeng_make_archive_entry(
                            specs=specs,
                            vector=cand_vector,
                            metrics=dict(bundle.get("median_metrics") or {}),
                            source_phase="endgame_verify",
                            source_iteration=int(iteration),
                            source_label=f"verify{cand_idx:02d}",
                            verified=True,
                            last_verified_median=float(bundle.get("median_lap", float("nan"))),
                        ),
                        specs=specs,
                        max_size=elite_archive_size,
                    )

            verified_legal = [
                (idx, vec, b)
                for idx, vec, b in verify_results
                if bool(b.get("median_legal")) and math.isfinite(float(b.get("median_lap", float("nan"))))
            ]
            verified_legal.sort(key=lambda t: float(t[2]["median_lap"]))
            incumbent_reference_lap_before = float(incumbent_reference_lap)
            acceptance_margin = float(
                max(
                    0.0015,
                    endgame_noise_margin_factor
                    * max(0.0, float(rolling_noise_sigma) if math.isfinite(float(rolling_noise_sigma)) else 0.0),
                )
            )
            decision = "endgame_reject"
            accepted_lap = float("nan")
            verified_accept = False
            if verified_legal:
                cand_idx, cand_vector, cand_bundle = verified_legal[0]
                cand_lap = float(cand_bundle["median_lap"])
                if (not math.isfinite(incumbent_reference_lap)) or (cand_lap < float(incumbent_reference_lap) - acceptance_margin):
                    current_vector = dict(cand_vector)
                    current_metrics = dict(cand_bundle["median_metrics"])
                    current_eval = dict(cand_bundle["representative_eval"])
                    incumbent_reference_lap = float(cand_lap)
                    incumbent_reference_bundle = dict(cand_bundle)
                    accepted_lap = float(cand_lap)
                    decision = "endgame_accept_verified"
                    verified_accept = True
                    if (best_legal_metrics is None) or (cand_lap < float(best_legal_metrics[RACEENG_OBJECTIVE])):
                        best_legal_metrics = dict(cand_bundle["median_metrics"])
                        best_legal_vector = dict(cand_vector)
                    if (best_verified_vector is None) or (not math.isfinite(best_verified_lap)) or (cand_lap < float(best_verified_lap)):
                        best_verified_vector = dict(cand_vector)
                        best_verified_metrics = dict(cand_bundle["median_metrics"])
                        best_verified_lap = float(cand_lap)

            archive_unverified_after = raceeng_best_archive_entry(elite_archive, require_unverified=True)
            archive_unverified_after_lap = (
                float(archive_unverified_after.get("lap", float("nan")))
                if isinstance(archive_unverified_after, dict)
                else float("nan")
            )
            unverified_improved = bool(
                math.isfinite(archive_unverified_after_lap)
                and (
                    (not math.isfinite(archive_unverified_before_lap))
                    or archive_unverified_after_lap < archive_unverified_before_lap - endgame_unverified_improvement_reset
                )
            )
            if verified_accept:
                endgame_no_improve_streak = 0
                endgame_sigma_mult = float(max(0.25, float(endgame_sigma_mult) * 0.92))
            elif unverified_improved:
                endgame_no_improve_streak = 0
                endgame_sigma_mult = float(endgame_sigma_mult)
                decision = "endgame_archive_progress"
            else:
                endgame_no_improve_streak += 1
                endgame_sigma_mult = float(min(2.5, float(endgame_sigma_mult) * 1.12))

            # Refresh incumbent reference periodically to reduce stale-noise bias.
            if (endgame_cycle % 3) == 0:
                ref_bundle = await raceeng_run_single_point_replicates(
                    session=session,
                    tenant_id=tenant_id,
                    worksheet_id=worksheet_id,
                    row_name=f"{iter_row_prefix}-incRef",
                    static_config_ids=static_config_ids,
                    car_payload=apply_raceeng_vector(car_payload, specs, current_vector),
                    timeout_seconds=timeout_seconds,
                    repeat_count=3,
                    runtime=runtime,
                    phase="endgame_incumbent_reference",
                    iteration=iteration,
                    bundle_role="incumbent_reference",
                )
                incumbent_reference_bundle = dict(ref_bundle)
                incumbent_reference_lap = float(ref_bundle.get("median_lap", incumbent_reference_lap))
                rolling_noise_sigma = raceeng_update_ewma(
                    rolling_noise_sigma,
                    raceeng_mad_sigma([float(v) for v in ref_bundle.get("laps", [])]),
                    alpha=0.35,
                )
                verification_rows.append(
                    {
                        "phase": "endgame",
                        "iteration": int(iteration),
                        "endgame_cycle": int(endgame_cycle),
                        "candidate_rank": 0,
                        "row_name": f"{iter_row_prefix}-incRef",
                        "median_lap": float(ref_bundle.get("median_lap", float("nan"))),
                        "median_legal": bool(ref_bundle.get("median_legal", False)),
                        "legal_count": int(ref_bundle.get("legal_count", 0)),
                        "repeat_count": int(ref_bundle.get("repeat_count", 0)),
                        "noise_sigma_bundle": float(raceeng_mad_sigma([float(v) for v in ref_bundle.get("laps", [])])),
                        "incumbent_reference_lap": float(incumbent_reference_lap),
                    }
                )

            trust_radius = float(max(0.30, min(2.8, float(trust_radius) * (1.02 if decision == "endgame_accept_verified" else 0.96))))
            current_lap = float(current_metrics.get(RACEENG_OBJECTIVE, float("nan")))
            hard_legal_yield = float(hard_legal_point_count) / float(max(1, len(point_results)))
            soft_legal_yield = float(soft_legal_point_count) / float(max(1, len(point_results)))
            finite_yield = float(finite_point_count) / float(max(1, len(point_results)))
            iteration_wall_seconds = float(perf_counter() - iter_start)
            joint_duration_seconds = float(
                np.nanmax(np.array([float(run.get("duration_seconds", float("nan"))) for run in lane_runs], dtype=float))
            )
            joint_wait_seconds = float(
                np.nanmax(np.array([float(run.get("wait_seconds", float("nan"))) for run in lane_runs], dtype=float))
            )
            joint_completed_job_count = int(sum(int(run.get("completed_job_count", 0)) for run in lane_runs))
            joint_job_count = int(sum(int(run.get("job_count", 0)) for run in lane_runs))
            joint_succeeded_simulation_count = int(sum(int(run.get("succeeded_simulation_count", 0)) for run in lane_runs))
            joint_is_complete = bool(all(bool(run.get("is_complete", False)) for run in lane_runs))
            joint_study_ids = ",".join(str(run.get("study_id", "")) for run in lane_runs)
            joint_exploration_ids = ",".join(str(run.get("exploration_id", "")) for run in lane_runs)
            wait_sample = float(joint_wait_seconds)
            if math.isfinite(wait_sample):
                study_wall_ewma = raceeng_update_ewma(study_wall_ewma, wait_sample, alpha=0.28)
            best_legal_lap_so_far = float("nan")
            if best_legal_metrics is not None:
                best_legal_lap_so_far = float(best_legal_metrics.get(RACEENG_OBJECTIVE, float("nan")))
            best_archive_legal_lap = float(raceeng_best_archive_legal_lap(elite_archive))
            verification_debt = bool(
                raceeng_has_verification_debt(
                    elite_archive,
                    best_verified_lap=best_verified_lap,
                    threshold=endgame_verification_debt_threshold,
                )
            )
            best_gain_vs_baseline = (
                float("nan")
                if (not math.isfinite(best_legal_lap_so_far) or not math.isfinite(baseline_lap))
                else float(baseline_lap - best_legal_lap_so_far)
            )
            print(
                "Endgame summary: "
                f"decision={decision}; accepted_lap={'nan' if not math.isfinite(accepted_lap) else f'{accepted_lap:.6f}'}; "
                f"inc_ref={incumbent_reference_lap:.6f}; stall={endgame_no_improve_streak}; "
                f"best_archive_legal={'nan' if not math.isfinite(best_archive_legal_lap) else f'{best_archive_legal_lap:.6f}'}; "
                f"verification_debt={verification_debt}; "
                f"rolling_noise_sigma={'nan' if not math.isfinite(rolling_noise_sigma) else f'{rolling_noise_sigma:.6f}'}"
            )

            endgame_rows.append(
                {
                    "phase": "endgame",
                    "iteration": int(iteration),
                    "endgame_cycle": int(endgame_cycle),
                    "in_extension": bool(in_extension),
                    "extension_cycles_used": int(extension_cycles_used),
                    "decision": str(decision),
                    "accepted_lap": float(accepted_lap),
                    "current_lap": float(current_lap),
                    "incumbent_reference_lap": float(incumbent_reference_lap),
                    "acceptance_margin": float(acceptance_margin),
                    "rolling_noise_sigma": float(rolling_noise_sigma),
                    "endgame_sigma_mult": float(endgame_sigma_mult),
                    "endgame_no_improve_streak": int(endgame_no_improve_streak),
                    "active_param_count": int(len(active_specs_end)),
                    "es_point_count": int(len(point_results)),
                    "hard_legal_yield": float(hard_legal_yield),
                    "soft_legal_yield": float(soft_legal_yield),
                    "finite_yield": float(finite_yield),
                    "iteration_wall_seconds": float(iteration_wall_seconds),
                    "study_wait_ewma_seconds": float(study_wall_ewma),
                    "best_legal_lap_so_far": float(best_legal_lap_so_far),
                    "best_archive_legal_lap": float(best_archive_legal_lap),
                    "verification_debt": bool(verification_debt),
                    "best_gain_vs_baseline": float(best_gain_vs_baseline),
                }
            )
            pending_study_count = int(len(runtime.pending_studies))
            timing_rows.append(
                {
                    "phase": "endgame",
                    "iteration": int(iteration),
                    "joint_duration_seconds": float(joint_duration_seconds),
                    "partial_wait_seconds": float(joint_wait_seconds),
                    "iteration_wall_seconds": float(iteration_wall_seconds),
                    "joint_point_count": int(len(point_results)),
                    "joint_completed_job_count": int(joint_completed_job_count),
                    "joint_job_count": int(joint_job_count),
                    "joint_succeeded_simulation_count": int(joint_succeeded_simulation_count),
                    "is_complete_at_partial": bool(joint_is_complete),
                    "hard_legal_point_count": int(hard_legal_point_count),
                    "soft_legal_point_count": int(soft_legal_point_count),
                    "finite_point_count": int(finite_point_count),
                    "hard_legal_yield": float(hard_legal_yield),
                    "soft_legal_yield": float(soft_legal_yield),
                    "finite_yield": float(finite_yield),
                    "iteration_eta_seconds": float("nan"),
                    "campaign_eta_seconds": float("nan"),
                    "eta_error_seconds": float("nan"),
                    "study_wait_ewma_seconds": float(study_wall_ewma),
                    "concurrent_starts": int(len(lane_runs)),
                    "batch_points": int(batch_points),
                    "best_legal_lap_so_far": float(best_legal_lap_so_far),
                    "best_archive_legal_lap": float(best_archive_legal_lap),
                    "best_gain_vs_baseline": float(best_gain_vs_baseline),
                    "obj_model_train_count": int(obj_train_count),
                    "obj_model_r2": float(obj_r2),
                    "obj_model_resid_std": float(obj_resid_std),
                    "pending_study_count": int(pending_study_count),
                    "island_lap_min": float("nan"),
                    "island_lap_median": float("nan"),
                }
            )
            controller_rows.append(
                {
                    "phase": "endgame",
                    "iteration": int(iteration),
                    "concurrent_starts_prev": int(len(lane_runs)),
                    "concurrent_starts_next": int(len(lane_runs)),
                    "concurrency_reason": "endgame_incumbent_plus_challenger" if len(lane_runs) > 1 else "endgame_single_es",
                    "batch_points_prev": int(batch_points),
                    "batch_points_next": int(batch_points),
                    "batch_points_reason": "hold_endgame",
                    "hard_legal_yield": float(hard_legal_yield),
                    "soft_legal_yield": float(soft_legal_yield),
                    "finite_yield": float(finite_yield),
                    "batch_improvement": float(
                        0.0 if not math.isfinite(accepted_lap) else max(0.0, incumbent_reference_lap_before - accepted_lap)
                    ),
                    "no_improve_streak": int(endgame_no_improve_streak),
                    "study_wait_ewma_seconds": float(study_wall_ewma),
                    "last_wait_sample_seconds": float(joint_wait_seconds),
                    "iteration_wall_seconds": float(iteration_wall_seconds),
                    "best_legal_lap_so_far": float(best_legal_lap_so_far),
                    "best_archive_legal_lap": float(best_archive_legal_lap),
                    "verification_debt": bool(verification_debt),
                    "best_gain_vs_baseline": float(best_gain_vs_baseline),
                    "obj_model_train_count": int(obj_train_count),
                    "obj_model_r2": float(obj_r2),
                    "obj_model_resid_std": float(obj_resid_std),
                    "pending_study_count": int(pending_study_count),
                    "island_lap_min": float("nan"),
                    "island_lap_median": float("nan"),
                }
            )
            iter_row = {
                "phase": "endgame",
                "iteration": int(iteration),
                "decision": str(decision),
                "trust_radius_next": float(trust_radius),
                "hardness": 1.0,
                "noise_sigma_tlap": float(rolling_noise_sigma),
                "calibration_bias": float(calibration_bias),
                "calibration_mae": float(calibration_mae),
                "improvement_threshold": float(acceptance_margin),
                "current_lap_before": float(current_lap),
                "current_legal_before": bool(raceeng_is_legal(current_metrics)),
                "batch_best_legal_lap": float("nan"),
                "batch_improvement": float(
                    0.0 if not math.isfinite(accepted_lap) else max(0.0, incumbent_reference_lap_before - accepted_lap)
                ),
                "hard_legal_yield": float(hard_legal_yield),
                "soft_legal_yield": float(soft_legal_yield),
                "finite_yield": float(finite_yield),
                "hard_legal_point_count": int(hard_legal_point_count),
                "soft_legal_point_count": int(soft_legal_point_count),
                "finite_point_count": int(finite_point_count),
                "joint_sweep_study_id": str(joint_study_ids),
                "joint_sweep_exploration_id": str(joint_exploration_ids),
                "joint_sweep_point_count": int(len(point_results)),
                "joint_completed_job_count": int(joint_completed_job_count),
                "joint_job_count": int(joint_job_count),
                "is_complete_at_partial": bool(joint_is_complete),
                "joint_duration_seconds": float(joint_duration_seconds),
                "partial_wait_seconds": float(joint_wait_seconds),
                "iteration_wall_seconds": float(iteration_wall_seconds),
                "iteration_eta_seconds": float("nan"),
                "campaign_eta_seconds": float("nan"),
                "eta_error_seconds": float("nan"),
                "study_wait_ewma_seconds": float(study_wall_ewma),
                "concurrent_starts_used": int(len(lane_runs)),
                "concurrent_starts_next": int(len(lane_runs)),
                "batch_points_used": int(batch_points),
                "batch_points_next": int(batch_points),
                "concurrency_reason": "endgame_incumbent_plus_challenger" if len(lane_runs) > 1 else "endgame_single_es",
                "batch_points_reason": "hold_endgame",
                "no_improve_streak": int(endgame_no_improve_streak),
                "improved_islands": 0,
                "islands_with_no_legal": 0,
                "best_legal_lap_so_far": float(best_legal_lap_so_far),
                "best_archive_legal_lap": float(best_archive_legal_lap),
                "verification_debt": bool(verification_debt),
                "best_gain_vs_baseline": float(best_gain_vs_baseline),
                "obj_model_train_count": int(obj_train_count),
                "obj_model_r2": float(obj_r2),
                "obj_model_resid_std": float(obj_resid_std),
                "pending_study_count": int(pending_study_count),
                "island_lap_min": float("nan"),
                "island_lap_median": float("nan"),
            }
            iteration_rows.append(iter_row)
            checkpoint_row = {
                "phase": "endgame",
                "iteration": int(iteration),
                "timestamp_utc": datetime.now(UTC).isoformat(),
                "current_lap": float(current_lap),
                "best_legal_lap": float(best_legal_lap_so_far),
                "best_verified_lap": float(best_verified_lap),
                "rolling_noise_sigma": float(rolling_noise_sigma),
                "endgame_stall": int(endgame_no_improve_streak),
                "extension_cycles_used": int(extension_cycles_used),
                "row_prefix": str(row_prefix),
            }
            checkpoint_rows.append(checkpoint_row)
            last_clean_iteration = int(iteration)
            campaign_status = "running"
            _write_checkpoint(
                phase="endgame",
                iteration=int(iteration),
                current_metrics_local=current_metrics,
                current_eval_local=current_eval,
            )

        if endgame_termination_reason == "completed_nominal":
            verification_debt = bool(
                raceeng_has_verification_debt(
                    elite_archive,
                    best_verified_lap=best_verified_lap,
                    threshold=endgame_verification_debt_threshold,
                )
            )
            if endgame_cycle >= nominal_endgame_cycles + endgame_extra_cycles_max and endgame_extra_cycles_max > 0:
                endgame_termination_reason = (
                    "extension_budget_reached_with_verification_debt"
                    if verification_debt
                    else "extension_budget_reached"
                )
            elif endgame_cycle > nominal_endgame_cycles and endgame_no_improve_streak >= endgame_stall_patience and not verification_debt:
                endgame_termination_reason = "stall_patience_reached"
            elif endgame_cycle >= nominal_endgame_cycles:
                endgame_termination_reason = "nominal_cycles_completed"

        final_recovery = await raceeng_recover_pending_studies(
            session=session,
            tenant_id=tenant_id,
            specs=specs,
            runtime=runtime,
        )
        _merge_recovered_joint_runs(list(final_recovery.get("joint_runs", [])))
        await raceeng_flush_worksheet_rows(runtime, force=True, reason="pre_finalization")
        if runtime.pending_studies:
            campaign_status = "finalizing"
            _write_checkpoint(
                phase="finalizing",
                iteration=int(max(last_clean_iteration, start_iteration - 1)),
                current_metrics_local=current_metrics,
                current_eval_local=current_eval,
            )
            raise RuntimeError(f"Finalization deferred; pending studies remain: {len(runtime.pending_studies)}")

        if best_legal_metrics is None or best_legal_vector is None:
            raise RuntimeError("No legal setup found")

        recovered_final_bundles = [
            dict(bundle)
            for bundle in list(final_recovery.get("bundles", []))
            if str(bundle.get("bundle_role", "")).startswith("final_validation")
        ]
        recovered_final_by_role = {
            str(bundle.get("bundle_role", "")): dict(bundle)
            for bundle in recovered_final_bundles
            if str(bundle.get("bundle_role", ""))
        }

        final_candidates: list[dict] = []
        if best_verified_vector is not None and best_verified_metrics is not None:
            final_candidates.append(
                {
                    "source": "best_verified",
                    "vector": dict(best_verified_vector),
                    "metrics": dict(best_verified_metrics),
                }
            )
        if final_candidate_policy == "top2":
            archive_candidate = raceeng_best_archive_entry(elite_archive, require_unverified=True)
            if archive_candidate is not None:
                archive_vector = dict(archive_candidate.get("vector") or {})
                if not final_candidates or raceeng_vector_distance(specs, archive_vector, dict(final_candidates[0]["vector"])) > 0.15:
                    final_candidates.append(
                        {
                            "source": "archive_unverified",
                            "vector": archive_vector,
                            "metrics": dict(archive_candidate.get("metrics") or {}),
                        }
                    )
        if not final_candidates:
            final_candidates.append(
                {
                    "source": "best_legal",
                    "vector": dict(best_legal_vector),
                    "metrics": dict(best_legal_metrics),
                }
            )
        final_candidates = final_candidates[:2]

        final_candidate_results: list[dict] = []
        if final_validation_complete and isinstance(final_validation_bundle_state, dict):
            final_candidate_results = [
                {
                    "source": str(final_validation_bundle_state.get("final_selected_source", final_validation_bundle_state.get("candidate_source", "final"))),
                    "bundle": dict(final_validation_bundle_state),
                }
            ]
        else:
            campaign_status = "finalizing"

            async def _run_final_candidate(cand_idx: int, cand: dict) -> dict:
                bundle_role = f"final_validation_{cand_idx:02d}"
                if bundle_role in recovered_final_by_role:
                    bundle = dict(recovered_final_by_role[bundle_role])
                else:
                    bundle = await raceeng_run_single_point_replicates(
                        session=session,
                        tenant_id=tenant_id,
                        worksheet_id=worksheet_id,
                        row_name=f"{row_prefix}-finalCand{cand_idx:02d}",
                        static_config_ids=static_config_ids,
                        car_payload=apply_raceeng_vector(car_payload, specs, dict(cand["vector"])),
                        timeout_seconds=timeout_seconds,
                        repeat_count=3,
                        runtime=runtime,
                        phase="final_validation",
                        iteration=max(total_iterations, last_clean_iteration),
                        bundle_role=bundle_role,
                    )
                return {
                    "source": str(cand["source"]),
                    "bundle_role": str(bundle_role),
                    "vector": dict(cand["vector"]),
                    "bundle": dict(bundle),
                }

            final_runs = await asyncio.gather(
                *[asyncio.create_task(_run_final_candidate(i + 1, cand)) for i, cand in enumerate(final_candidates)],
                return_exceptions=True,
            )
            for res in final_runs:
                if isinstance(res, Exception):
                    print(f"Final candidate validation failed: {type(res).__name__}: {res}")
                    continue
                final_candidate_results.append(dict(res))

        legal_final_candidates = [
            row
            for row in final_candidate_results
            if bool((row.get("bundle") or {}).get("median_legal", False))
            and math.isfinite(float((row.get("bundle") or {}).get("median_lap", float("nan"))))
        ]
        if legal_final_candidates:
            legal_final_candidates.sort(key=lambda row: float((row.get("bundle") or {}).get("median_lap", float("inf"))))
            selected_final = dict(legal_final_candidates[0])
        elif final_candidate_results:
            final_candidate_results.sort(
                key=lambda row: float((row.get("bundle") or {}).get("median_lap", float("inf")))
            )
            selected_final = dict(final_candidate_results[0])
        else:
            raise RuntimeError("No final candidate validation bundles succeeded")

        final_bundle = dict(selected_final["bundle"])
        final_candidate_source = str(selected_final.get("source", "final"))
        final_candidate_vector = dict(selected_final.get("vector") or best_legal_vector)
        final_metrics = dict(final_bundle["median_metrics"])
        final_lap = float(final_bundle["median_lap"])
        final_laps = [float(v) for v in final_bundle["laps"]]
        final_legal_count = int(final_bundle["legal_count"])
        final_repeat_count = int(final_bundle["repeat_count"])
        final_legal = bool(final_bundle["median_legal"])
        final_validation_complete = True
        final_validation_bundle_state = dict(final_bundle)
        final_validation_bundle_state["candidate_source"] = str(final_candidate_source)
        final_validation_bundle_state["final_selected_source"] = str(final_candidate_source)
        improvement_vs_baseline = (
            float("nan")
            if (not math.isfinite(final_lap) or not math.isfinite(baseline_lap))
            else float(baseline_lap - final_lap)
        )
        print(
            "Final validation median-of-3: "
            f"laps={[round(v, 6) for v in final_laps]}, "
            f"median={final_lap:.6f}, legal_runs={final_legal_count}/{final_repeat_count}, median_hard_legal={final_legal}, "
            f"improvement_vs_baseline_median={improvement_vs_baseline:+.6f}"
        )
        if final_legal:
            best_legal_metrics = dict(final_metrics)
            best_legal_vector = dict(final_candidate_vector)
            elite_archive = raceeng_update_elite_archive(
                elite_archive,
                raceeng_make_archive_entry(
                    specs=specs,
                    vector=final_candidate_vector,
                    metrics=final_metrics,
                    source_phase="final_validation",
                    source_iteration=max(total_iterations, last_clean_iteration),
                    source_label=str(final_candidate_source),
                    verified=True,
                    last_verified_median=float(final_lap),
                ),
                specs=specs,
                max_size=elite_archive_size,
            )
            if (best_verified_vector is None) or (
                math.isfinite(float(final_lap))
                and (not math.isfinite(float(best_verified_lap)) or float(final_lap) < float(best_verified_lap))
            ):
                best_verified_vector = dict(final_candidate_vector)
                best_verified_metrics = dict(final_metrics)
                best_verified_lap = float(final_lap)

        best_car_payload = apply_raceeng_vector(car_payload, specs, best_legal_vector)
        Path(output_files["best_car_json"]).write_text(json.dumps(best_car_payload, indent=2), encoding="utf-8")

        sweep_df = pd.DataFrame(sweep_rows)
        sens_df = pd.DataFrame(sensitivity_rows)
        iter_df = pd.DataFrame(iteration_rows)
        surrogate_df = pd.DataFrame(surrogate_rows)
        constraint_df = pd.DataFrame(constraint_rows)
        noise_df = pd.DataFrame(noise_rows)
        timing_df = pd.DataFrame(timing_rows)
        controller_df = pd.DataFrame(controller_rows)
        endgame_df = pd.DataFrame(endgame_rows)
        verification_df = pd.DataFrame(verification_rows)
        checkpoint_df = pd.DataFrame(checkpoint_rows)
        elite_archive_df = pd.DataFrame(
            [
                {
                    "rank": int(idx + 1),
                    "lap": float(row.get("lap", float("nan"))),
                    "verified": bool(row.get("verified", False)),
                    "last_verified_median": float(row.get("last_verified_median", float("nan"))),
                    "source_phase": str(row.get("source_phase", "")),
                    "source_iteration": int(row.get("source_iteration", 0)),
                    "source_label": str(row.get("source_label", "")),
                    **{f"param_{k}": float(v) for k, v in dict(row.get("vector") or {}).items()},
                }
                for idx, row in enumerate(elite_archive)
            ]
        )
        sweep_df.to_csv(output_files["sweep_results_csv"], index=False)
        sens_df.to_csv(output_files["sensitivities_csv"], index=False)
        iter_df.to_csv(output_files["iteration_summary_csv"], index=False)
        surrogate_df.to_csv(output_files["surrogate_diagnostics_csv"], index=False)
        constraint_df.to_csv(output_files["constraint_schedule_csv"], index=False)
        noise_df.to_csv(output_files["noise_estimates_csv"], index=False)
        timing_df.to_csv(output_files["timing_summary_csv"], index=False)
        controller_df.to_csv(output_files["controller_trace_csv"], index=False)
        endgame_df.to_csv(output_files["endgame_cycle_summary_csv"], index=False)
        verification_df.to_csv(output_files["verification_replicates_csv"], index=False)
        checkpoint_df.to_csv(output_files["checkpoint_trace_csv"], index=False)
        pd.DataFrame(runtime.reconnect_rows).to_csv(output_files["api_reconnect_trace_csv"], index=False)
        pd.DataFrame(runtime.worksheet_reconcile_rows).to_csv(output_files["worksheet_reconcile_trace_csv"], index=False)
        elite_archive_df.to_csv(output_files["elite_archive_csv"], index=False)

        last_global_iteration = int(max([int(r.get("iteration", 0)) for r in iteration_rows], default=start_iteration - 1))
        campaign_status = "completed"
        result_json_written = False

        final_candidate_medians = {
            str(row.get("source", "final")): float((row.get("bundle") or {}).get("median_lap", float("nan")))
            for row in final_candidate_results
        }
        final_candidate_sources = [str(row.get("source", "final")) for row in final_candidate_results]

        result_payload = {
            "timestamp_utc": datetime.now(UTC).isoformat(),
            "worksheet_tenant_id": tenant_id,
            "worksheet_id": worksheet_id,
            "worksheet_name": None if worksheet_name is None else str(worksheet_name),
            "run_worksheet_number": None if worksheet_number is None else int(worksheet_number),
            "row_prefix": row_prefix,
            "algorithm": "surrogate_partial_harvest_multistart_v3_endgame_es",
            "run_chunk_iterations_requested": int(run_iteration_count),
            "campaign_target_iterations": int(total_iterations),
            "max_iterations": int(run_iteration_count),
            "start_iteration": int(start_iteration),
            "end_iteration": int(last_global_iteration),
            "campaign_total_iterations": int(total_iterations),
            "executed_iterations_total": int(last_global_iteration),
            "main_iterations": int(main_iterations),
            "nominal_endgame_cycles": int(nominal_endgame_cycles),
            "endgame_cycles_executed": int(endgame_cycle),
            "extension_cycles_used": int(extension_cycles_used),
            "endgame_termination_reason": str(endgame_termination_reason),
            "sweep_points_per_parameter": int(sweep_points_per_parameter),
            "focus_fraction": float(focus_fraction),
            "endgame_fraction": float(endgame_fraction),
            "endgame_verify_top_k": int(endgame_verify_top_k),
            "endgame_extra_cycles_max": int(endgame_extra_cycles_max),
            "endgame_stall_patience": int(endgame_stall_patience),
            "endgame_noise_margin_factor": float(endgame_noise_margin_factor),
            "elite_archive_size": int(elite_archive_size),
            "endgame_verification_debt_threshold": float(endgame_verification_debt_threshold),
            "endgame_unverified_improvement_reset": float(endgame_unverified_improvement_reset),
            "final_candidate_policy": str(final_candidate_policy),
            "force_two_main_starts": bool(force_two_main_starts),
            "concurrent_starts_per_iteration_initial": int(min(2, island_count)),
            "concurrent_starts_per_iteration_final": int(concurrent_starts),
            "batch_points_per_parameter_initial": int(sweep_points_per_parameter),
            "batch_points_per_parameter_final": int(batch_points),
            "best_legal_vector": {k: float(v) for k, v in best_legal_vector.items()},
            "best_legal_metrics": {k: float(best_legal_metrics.get(k, float("nan"))) for k in RACEENG_METRICS},
            "best_legal_violation_breakdown": raceeng_legality_breakdown(best_legal_metrics),
            "best_verified_vector": None if best_verified_vector is None else {k: float(v) for k, v in best_verified_vector.items()},
            "best_verified_metrics": None
            if best_verified_metrics is None
            else {k: float(best_verified_metrics.get(k, float("nan"))) for k in RACEENG_METRICS},
            "best_verified_lap": float(best_verified_lap),
            "best_archive_legal_lap": float(raceeng_best_archive_legal_lap(elite_archive)),
            "verification_debt_cleared": bool(
                not raceeng_has_verification_debt(
                    elite_archive,
                    best_verified_lap=best_verified_lap,
                    threshold=endgame_verification_debt_threshold,
                )
            ),
            "elite_archive": [dict(row) for row in elite_archive],
            "final_candidate_source": str(final_candidate_source),
            "final_candidates_tested": int(len(final_candidate_results)),
            "final_candidate_sources": list(final_candidate_sources),
            "final_candidate_medians": dict(final_candidate_medians),
            "final_selected_source": str(final_candidate_source),
            "baseline_lap": float(baseline_lap),
            "baseline_laps": [float(v) for v in baseline_laps],
            "baseline_hard_legal_runs": int(baseline_legal_count),
            "final_validation_lap": float(final_lap),
            "final_validation_laps": [float(v) for v in final_laps],
            "final_validation_hard_legal_runs": int(final_legal_count),
            "improvement_vs_baseline_median": float(improvement_vs_baseline),
            "rolling_noise_sigma_final": float(rolling_noise_sigma),
            "endgame_no_improve_streak_final": int(endgame_no_improve_streak),
            "api_reauth_count": int(runtime.api_reauth_count),
            "api_retry_count": int(runtime.api_retry_count),
            "recovered_study_count": int(runtime.recovered_study_count),
            "pending_study_count_final": int(len(runtime.pending_studies)),
            "worksheet_rows_written": int(runtime.worksheet_rows_written),
            "worksheet_rows_reinserted": int(runtime.worksheet_rows_reinserted),
            "worksheet_reconcile_retry_count": int(runtime.worksheet_reconcile_retry_count),
            "campaign_completed_cleanly": bool(
                final_validation_complete and not runtime.pending_studies
            ),
            "finalization_only_pass_used": bool(runtime.finalization_only_pass_used),
            "outputs": dict(output_files),
        }
        Path(output_files["result_json"]).write_text(json.dumps(result_payload, indent=2), encoding="utf-8")
        result_json_written = True
        final_result_json_path = str(output_files["result_json"])
        final_state_payload = _campaign_state_payload(
            phase="completed",
            iteration=int(max(last_global_iteration, last_clean_iteration)),
            current_metrics_local=final_metrics,
            current_eval_local=final_bundle.get("representative_eval") or current_eval,
        )
        Path(output_files["checkpoint_json"]).write_text(json.dumps(final_state_payload, indent=2), encoding="utf-8")
        raceeng_write_campaign_state(
            campaign_state_stem=campaign_state_stem,
            state_payload=final_state_payload,
            history_X=history_X,
            history_obj=history_obj,
            history_cons=history_cons,
        )
        return result_payload, sweep_df, sens_df, iter_df
    finally:
        await raceeng_shutdown_worksheet_writer(runtime)


## Run Orchestration

This section wires the notebook settings into a concrete run, resolves or creates the destination worksheet, loads the default challenge inputs, estimates the remaining simulation budget, and enforces the user consent prompt before any new studies are launched.


In [ ]:
import asyncio
import copy
import json
import math
import re
import traceback
from datetime import UTC, datetime
from pathlib import Path

import canopy
from canopy.openapi import (
    GetTenantWorksheetLabelDefinitionsQueryResultLabelDefinitions,
    NewWorksheetDataOutline,
    StudyApi,
    WorksheetApi,
    WorksheetPostWorksheetRequest,
)

# This notebook is intended as customer-facing example code rather than a production SDK.
# Review the settings, warnings, and run-budget summary before launching simulations.
CUSTOMER_SETTINGS = RUN_CONFIG.customer
ADVANCED_SETTINGS = RUN_CONFIG.advanced
DEFAULT_CONFIG_URLS = dict(RUN_CONFIG.default_config_urls)

TARGET_TENANT_ID = str(CUSTOMER_SETTINGS.target_tenant_id)
WORKSHEET_DESTINATION_BASE = str(CUSTOMER_SETTINGS.worksheet_destination)
AUTO_INCREMENT_NAMED_WORKSHEETS = bool(CUSTOMER_SETTINGS.auto_increment_named_worksheets)
CREATE_WORKSHEET_IF_MISSING = bool(CUSTOMER_SETTINGS.create_worksheet_if_missing)
WORKSHEET_REGISTRY_FILE = str(ADVANCED_SETTINGS.worksheet_registry_file)

BASE_RUN_PREFIX = str(CUSTOMER_SETTINGS.base_run_prefix)
TOTAL_TARGET_ITERATIONS = int(CUSTOMER_SETTINGS.total_target_iterations)

MAX_AUTO_RESTARTS = int(ADVANCED_SETTINGS.max_auto_restarts)
RETRY_DELAY_SECONDS = int(ADVANCED_SETTINGS.retry_delay_seconds)

STUDY_TIMEOUT_SECONDS = int(ADVANCED_SETTINGS.study_timeout_seconds)
SWEEP_POINTS_PER_PARAMETER = int(ADVANCED_SETTINGS.sweep_points_per_parameter)
FOCUS_FRACTION = float(ADVANCED_SETTINGS.focus_fraction)
ENDGAME_FRACTION = float(ADVANCED_SETTINGS.endgame_fraction)
ENDGAME_VERIFY_TOP_K = int(ADVANCED_SETTINGS.endgame_verify_top_k)
ENDGAME_EXTRA_CYCLES_MAX = int(ADVANCED_SETTINGS.endgame_extra_cycles_max)
ENDGAME_STALL_PATIENCE = int(ADVANCED_SETTINGS.endgame_stall_patience)
ENDGAME_NOISE_MARGIN_FACTOR = float(ADVANCED_SETTINGS.endgame_noise_margin_factor)
ELITE_ARCHIVE_SIZE = int(ADVANCED_SETTINGS.elite_archive_size)
ENDGAME_VERIFICATION_DEBT_THRESHOLD = float(ADVANCED_SETTINGS.endgame_verification_debt_threshold)
ENDGAME_UNVERIFIED_IMPROVEMENT_RESET = float(ADVANCED_SETTINGS.endgame_unverified_improvement_reset)
FINAL_CANDIDATE_POLICY = str(ADVANCED_SETTINGS.final_candidate_policy)
FORCE_TWO_MAIN_STARTS = bool(ADVANCED_SETTINGS.force_two_main_starts)
WORKSHEET_ROW_BATCH_SECONDS = float(ADVANCED_SETTINGS.worksheet_row_batch_seconds)
WORKSHEET_ROW_BATCH_SIZE = int(ADVANCED_SETTINGS.worksheet_row_batch_size)

ROW_RE_V2 = re.compile(
    r"^(?P<prefix>.+)-it(?P<it>\d+)-(?P<tag>is\d+)-jointSweep(?:-retry\d+)?$"
)
ROW_RE_V1 = re.compile(r"^(?P<prefix>.+)-it(?P<it>\d+)-(?P<kind>candidate|legalFallback|current)$")
ROW_LAUNCH_SUFFIX_RE = re.compile(r"^(?P<base>.+?)--L[0-9a-fA-F]{8}$")
KIND_PRIORITY = {"jointSweep": 3, "legalFallback": 2, "candidate": 1, "current": 0}


class RaceEngFatalNotebookError(RuntimeError):
    pass


def raceeng_is_fatal_notebook_error(ex: BaseException) -> bool:
    return isinstance(
        ex,
        (
            RaceEngFatalNotebookError,
            NameError,
            AttributeError,
            TypeError,
            KeyError,
            UnboundLocalError,
            AssertionError,
            ImportError,
        ),
    )


def _is_probable_worksheet_id(value: str) -> bool:
    return bool(re.fullmatch(r"[0-9a-fA-F]{32}", str(value or "").strip()))


def _strip_launch_suffix(name: str) -> str:
    text = str(name or "")
    match = ROW_LAUNCH_SUFFIX_RE.match(text)
    return str(match.group("base")) if match is not None else text


def _campaign_state_stem_for_worksheet(worksheet_id: str) -> str:
    return f"{CAMPAIGN_STATE_STEM_BASE}_{raceeng_safe_file_stem(worksheet_id)}"


def _campaign_is_complete(campaign_state: dict | None) -> bool:
    if not isinstance(campaign_state, dict):
        return False
    return bool(
        str(campaign_state.get("campaign_status", "")) == "completed"
        and bool(campaign_state.get("final_validation_complete", False))
        and bool(campaign_state.get("result_json_written", False))
        and not list(campaign_state.get("pending_studies") or [])
    )


def _load_matching_campaign_state(campaign_state_stem: str, worksheet_id: str) -> dict | None:
    state = raceeng_load_campaign_state(campaign_state_stem)
    if not isinstance(state, dict):
        return None
    state_worksheet_id = str(state.get("worksheet_id") or "").strip()
    if state_worksheet_id and state_worksheet_id != str(worksheet_id):
        print(
            f"Ignoring campaign state '{campaign_state_stem}' because worksheet_id={state_worksheet_id} "
            f"does not match current worksheet_id={worksheet_id}."
        )
        return None
    return state


def _load_worksheet_registry(path: str) -> dict:
    p = Path(path)
    if not p.exists():
        return {}
    try:
        data = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        return {}
    return data if isinstance(data, dict) else {}


def _save_worksheet_registry(path: str, data: dict):
    Path(path).write_text(json.dumps(data, indent=2), encoding="utf-8")


def _next_worksheet_number_from_registry(path: str, pattern: re.Pattern[str]) -> int:
    registry = _load_worksheet_registry(path)
    values = []
    for name in registry.keys():
        match = pattern.match(str(name))
        if match is not None:
            values.append(int(match.group("num")))
    return int(max(values, default=0) + 1)


if AUTO_INCREMENT_NAMED_WORKSHEETS and not _is_probable_worksheet_id(WORKSHEET_DESTINATION_BASE):
    WORKSHEET_NAME_RE = re.compile(rf"^{re.escape(WORKSHEET_DESTINATION_BASE)} (?P<num>\d+)$")
    RUN_WORKSHEET_NUMBER = _next_worksheet_number_from_registry(WORKSHEET_REGISTRY_FILE, WORKSHEET_NAME_RE)
    WORKSHEET_DESTINATION = f"{WORKSHEET_DESTINATION_BASE} {RUN_WORKSHEET_NUMBER}"
else:
    WORKSHEET_NAME_RE = re.compile(r"^(?P<num>-1)$")
    RUN_WORKSHEET_NUMBER = None
    WORKSHEET_DESTINATION = WORKSHEET_DESTINATION_BASE

_ws_stem_token = (
    f"ws{int(RUN_WORKSHEET_NUMBER)}"
    if RUN_WORKSHEET_NUMBER is not None
    else f"ws_{raceeng_safe_file_stem(WORKSHEET_DESTINATION)}"
)
CAMPAIGN_STATE_STEM_BASE = f"raceeng_campaign_{raceeng_safe_file_stem(BASE_RUN_PREFIX)}_{_ws_stem_token}"


async def _close_sessions(*sessions):
    seen = set()
    for session in sessions:
        if session is None:
            continue
        sid = id(session)
        if sid in seen:
            continue
        seen.add(sid)
        try:
            await session.close()
        except Exception:
            pass


def _estimate_main_study_point_count(spec_count: int, points_per_parameter: int) -> int:
    return int(max(120, max(1, spec_count) * max(3, int(points_per_parameter))))


def _estimate_endgame_lane_point_count(spec_count: int) -> int:
    active_count = int(max(18, math.ceil(0.45 * max(1, spec_count))))
    return int(max(48, min(168, max(96, 4 * active_count))))


def _format_credit_estimate(value: float) -> str:
    rounded = round(float(value), 2)
    if abs(rounded - round(rounded)) < 1e-9:
        return f"{int(round(rounded)):,}"
    if abs(rounded * 10 - round(rounded * 10)) < 1e-9:
        return f"{rounded:,.1f}"
    return f"{rounded:,.2f}"


def _raceeng_prompt_yes_cancel_tk(*, title: str, message: str, yes_text: str = "Yes", cancel_text: str = "Cancel") -> bool:
    import tkinter as tk
    from tkinter import ttk

    result = {"value": False}
    root = tk.Tk()
    root.title(title)
    root.attributes("-topmost", True)
    root.resizable(True, True)

    frame = ttk.Frame(root, padding=12)
    frame.grid(row=0, column=0, sticky="nsew")
    root.columnconfigure(0, weight=1)
    root.rowconfigure(0, weight=1)
    frame.columnconfigure(0, weight=1)
    frame.columnconfigure(1, weight=1)
    frame.rowconfigure(0, weight=1)

    text = tk.Text(frame, width=96, height=18, wrap="word")
    text.grid(row=0, column=0, columnspan=2, sticky="nsew", pady=(0, 10))
    text.insert("1.0", message)
    text.configure(state="disabled")

    def _accept():
        result["value"] = True
        root.quit()

    def _cancel():
        root.quit()

    root.protocol("WM_DELETE_WINDOW", _cancel)

    yes_button = tk.Button(
        frame,
        text=yes_text,
        command=_accept,
        width=16,
        bg="#c62828",
        fg="white",
        activebackground="#b71c1c",
        activeforeground="white",
    )
    cancel_button = tk.Button(frame, text=cancel_text, command=_cancel, width=16)
    yes_button.grid(row=1, column=0, sticky="ew", padx=(0, 6))
    cancel_button.grid(row=1, column=1, sticky="ew")

    root.update_idletasks()
    root.mainloop()
    try:
        root.destroy()
    except tk.TclError:
        pass
    return bool(result["value"])


def _estimate_remaining_sim_budget(
    car_payload: dict,
    *,
    start_iteration: int,
    resume_state: dict | None,
    finalization_only: bool,
) -> dict:
    spec_count = int(len(build_raceeng_parameter_specs(car_payload["config"])))
    island_count = 3 if spec_count >= 3 else (2 if spec_count >= 2 else 1)
    nominal_main_launches = int(min(2, island_count))
    upper_main_launches = int(max(1, island_count))

    total_iterations = int(max(1, TOTAL_TARGET_ITERATIONS))
    main_iterations = int(max(1, math.ceil((1.0 - ENDGAME_FRACTION) * total_iterations)))
    nominal_endgame_cycles = int(max(0, total_iterations - main_iterations))

    if finalization_only:
        remaining_main_iterations = 0
        remaining_nominal_endgame_cycles = 0
    else:
        remaining_main_iterations = (
            0
            if start_iteration > main_iterations
            else int(main_iterations - start_iteration + 1)
        )
        endgame_start_iteration = int(max(start_iteration, main_iterations + 1))
        remaining_nominal_endgame_cycles = (
            0
            if endgame_start_iteration > total_iterations
            else int(total_iterations - endgame_start_iteration + 1)
        )

    upper_endgame_cycles = int(max(0, remaining_nominal_endgame_cycles) + ENDGAME_EXTRA_CYCLES_MAX + 3)

    baseline_done = isinstance((resume_state or {}).get("baseline_bundle"), dict)
    final_validation_done = bool((resume_state or {}).get("final_validation_complete", False)) or isinstance(
        (resume_state or {}).get("final_validation_bundle"),
        dict,
    )

    baseline_nominal = 0 if baseline_done or finalization_only else 3
    baseline_upper = 0 if baseline_done or finalization_only else 6

    main_point_count = _estimate_main_study_point_count(spec_count, SWEEP_POINTS_PER_PARAMETER)
    main_nominal = int(remaining_main_iterations * nominal_main_launches * main_point_count)
    main_upper = int(remaining_main_iterations * upper_main_launches * (2 * main_point_count))

    endgame_lane_point_count = _estimate_endgame_lane_point_count(spec_count)
    verify_bundle_points = int(3 * ENDGAME_VERIFY_TOP_K)
    incumbent_reference_nominal = int(math.ceil(max(0, remaining_nominal_endgame_cycles) / 3.0) * 3)
    incumbent_reference_upper = int(max(0, upper_endgame_cycles) * 3)
    endgame_nominal = int(
        remaining_nominal_endgame_cycles * (endgame_lane_point_count + verify_bundle_points)
        + incumbent_reference_nominal
    )
    endgame_upper = int(
        upper_endgame_cycles * (4 * endgame_lane_point_count + 2 * verify_bundle_points)
        + incumbent_reference_upper
    )

    final_validation_nominal = 0 if final_validation_done else 6
    final_validation_upper = 0 if final_validation_done else 12

    nominal_total = int(baseline_nominal + main_nominal + endgame_nominal + final_validation_nominal)
    upper_total = int(baseline_upper + main_upper + endgame_upper + final_validation_upper)
    nominal_credit_estimate = float(nominal_total)
    upper_credit_estimate = float(upper_total)
    return {
        "spec_count": int(spec_count),
        "island_count": int(island_count),
        "remaining_main_iterations": int(remaining_main_iterations),
        "remaining_nominal_endgame_cycles": int(remaining_nominal_endgame_cycles),
        "upper_endgame_cycles": int(upper_endgame_cycles),
        "main_point_count": int(main_point_count),
        "endgame_lane_point_count": int(endgame_lane_point_count),
        "baseline_nominal": int(baseline_nominal),
        "baseline_upper": int(baseline_upper),
        "main_nominal": int(main_nominal),
        "main_upper": int(main_upper),
        "endgame_nominal": int(endgame_nominal),
        "endgame_upper": int(endgame_upper),
        "final_validation_nominal": int(final_validation_nominal),
        "final_validation_upper": int(final_validation_upper),
        "nominal_total": int(nominal_total),
        "upper_total": int(upper_total),
        "nominal_credit_estimate": float(nominal_credit_estimate),
        "upper_credit_estimate": float(upper_credit_estimate),
    }


def _prompt_for_run_consent(
    *,
    budget: dict,
    worksheet_id: str,
    worksheet_name: str,
    source_desc: str,
    resumed: bool,
) -> None:
    banner = "=" * 88
    context_label = "Remaining simulations from this point" if resumed else "Simulation budget estimate for this run"
    message = "\n".join(
        [
            banner,
            "WARNING: This notebook may launch a large number of Canopy simulations.",
            (
                f"{context_label}: nominal ~{int(budget['nominal_total']):,}, "
                f"conservative upper bound ~{int(budget['upper_total']):,}."
            ),
            (
                f"Estimated compute credits (1 dynamic lap simulation = 1 compute credit): "
                f"nominal ~{_format_credit_estimate(budget['nominal_credit_estimate'])}, "
                f"conservative upper bound ~{_format_credit_estimate(budget['upper_credit_estimate'])}."
            ),
            (
                "Breakdown: "
                f"baseline~{int(budget['baseline_nominal']):,}, "
                f"main~{int(budget['main_nominal']):,}, "
                f"endgame~{int(budget['endgame_nominal']):,}, "
                f"final-validation~{int(budget['final_validation_nominal']):,}."
            ),
            f"Nominal campaign target iterations: {int(TOTAL_TARGET_ITERATIONS):,}.",
            f"Maximum additional endgame extension cycles: {int(ENDGAME_EXTRA_CYCLES_MAX):,}.",
            "The conservative upper bound includes endgame extension budget, verification work, retries, and final validation.",
            (
                "Study sizing assumptions: "
                f"{int(budget['remaining_main_iterations'])} main iterations at ~{int(budget['main_point_count']):,} sims/study, "
                f"{int(budget['remaining_nominal_endgame_cycles'])} nominal endgame cycles at ~{int(budget['endgame_lane_point_count']):,} sims/lane."
            ),
            f"Worksheet target: {worksheet_name} ({worksheet_id})",
            f"Resume source: {source_desc}",
            (
                "This notebook is AI-generated example code intended for inspiration and further vibe-coding applications, "
                "including worksheet interactions, API reconnect handling, auto-resume flows, and optimisation algorithms."
            ),
            "Review and adapt it before using it in any production or customer workflow.",
            banner,
        ]
    )
    print()
    print(message)
    try:
        consent = _raceeng_prompt_yes_cancel_tk(
            title="RaceEng Challenge Launch Warning",
            message=message,
            yes_text="Yes",
            cancel_text="Cancel",
        )
    except Exception as ex:
        raise RuntimeError(f"Consent dialog unavailable ({type(ex).__name__}: {ex}).") from ex
    if not consent:
        raise RaceEngUserCancelledRun("Run cancelled because the user did not consent to the estimated simulation and compute-credit budget.")


async def _create_blank_named_worksheet(
    session,
    tenant_id: str,
    worksheet_name: str,
    runtime: RaceEngRuntime | None = None,
) -> str:
    request = WorksheetPostWorksheetRequest(
        name=worksheet_name,
        properties=[],
        outline=NewWorksheetDataOutline(
            rows=[],
            label_definitions=GetTenantWorksheetLabelDefinitionsQueryResultLabelDefinitions(
                simulation_label_definitions=[],
                config_label_definitions=[],
            ),
        ),
        notes="",
    )
    if runtime is not None:
        created = await raceeng_call_with_retry(
            runtime,
            f"worksheet_create:{worksheet_name}",
            lambda s: WorksheetApi(s.async_client).worksheet_post_worksheet(
                tenant_id=tenant_id,
                worksheet_post_worksheet_request=request,
            ),
        )
    else:
        session.authentication.authenticate()
        worksheet_api = WorksheetApi(session.async_client)
        created = await worksheet_api.worksheet_post_worksheet(
            tenant_id=tenant_id,
            worksheet_post_worksheet_request=request,
        )
    return created.worksheet.worksheet_id


async def resolve_or_create_destination_worksheet(
    session,
    tenant_id: str,
    destination: str,
    create_if_missing: bool,
    registry_path: str,
    runtime: RaceEngRuntime | None = None,
    defer_create_until_confirmed: bool = False,
) -> tuple[str | None, str, str]:
    token = str(destination or "").strip()
    if not token:
        raise ValueError("WORKSHEET_DESTINATION must be non-empty")

    async def _get_worksheet(target_id: str):
        if runtime is not None:
            return await raceeng_call_with_retry(
                runtime,
                f"worksheet_get:{target_id}",
                lambda s: WorksheetApi(s.async_client).worksheet_get_worksheet(tenant_id, target_id),
            )
        session.authentication.authenticate()
        worksheet_api = WorksheetApi(session.async_client)
        return await worksheet_api.worksheet_get_worksheet(tenant_id, target_id)

    if _is_probable_worksheet_id(token):
        result = await _get_worksheet(token)
        return token, str(result.worksheet.name or token), "id"

    registry = _load_worksheet_registry(registry_path)
    existing_id = registry.get(token)
    if isinstance(existing_id, str) and _is_probable_worksheet_id(existing_id):
        try:
            result = await _get_worksheet(existing_id)
            return existing_id, str(result.worksheet.name or token), "registry"
        except Exception:
            registry.pop(token, None)

    if not create_if_missing:
        raise RuntimeError(
            f"Worksheet '{token}' not found in {registry_path}. "
            "Provide a worksheet ID or enable CREATE_WORKSHEET_IF_MISSING."
        )
    if defer_create_until_confirmed:
        return None, token, "deferred-create"

    new_id = await _create_blank_named_worksheet(
        session=session,
        tenant_id=tenant_id,
        worksheet_name=token,
        runtime=runtime,
    )
    registry[token] = new_id
    _save_worksheet_registry(registry_path, registry)
    return new_id, token, "created"


def _extract_row_car_id(row):
    for cfg in (row.configs or []):
        if getattr(cfg, "config_type", None) != "car":
            continue
        ref = getattr(cfg, "reference", None)
        tenant_ref = None if ref is None else getattr(ref, "tenant", None)
        target_id = None if tenant_ref is None else getattr(tenant_ref, "target_id", None)
        if target_id:
            return str(target_id)
    return None


async def _study_success_count(runtime: RaceEngRuntime, tenant_id: str, study_id: str | None) -> int:
    if not study_id:
        return 0
    try:
        meta = await raceeng_call_with_retry(
            runtime,
            f"study_metadata:{study_id}:resume_parser",
            lambda s: StudyApi(s.async_client).study_get_study_metadata(tenant_id, study_id),
            max_attempts=4,
            base_delay_seconds=1.0,
            max_delay_seconds=8.0,
        )
        study_doc = canopy.get_study_document(runtime.session, meta.study)
        return int(getattr(study_doc, "succeeded_job_count", 0) or 0)
    except Exception:
        return 0


def _parse_iteration_row_name(name: str):
    name = _strip_launch_suffix(name)
    m2 = ROW_RE_V2.match(name)
    if m2 is not None:
        return {
            "prefix": m2.group("prefix"),
            "iteration": int(m2.group("it")),
            "kind": "jointSweep",
            "tag": str(m2.group("tag")),
            "logical_name": name,
        }
    m1 = ROW_RE_V1.match(name)
    if m1 is not None:
        return {
            "prefix": m1.group("prefix"),
            "iteration": int(m1.group("it")),
            "kind": str(m1.group("kind")),
            "tag": None,
            "logical_name": name,
        }
    return None


async def _resolve_progress_state(runtime: RaceEngRuntime, tenant_id: str, worksheet_id: str, base_prefix: str):
    ws = (
        await raceeng_call_with_retry(
            runtime,
            f"worksheet_get:{worksheet_id}:progress",
            lambda s: WorksheetApi(s.async_client).worksheet_get_worksheet(tenant_id, worksheet_id),
        )
    ).worksheet
    rows = list(ws.outline.rows or [])

    parsed = []
    for idx, row in enumerate(rows):
        name = row.name or ""
        parsed_name = _parse_iteration_row_name(name)
        if parsed_name is None:
            continue
        prefix = str(parsed_name["prefix"])
        if not prefix.startswith(base_prefix):
            continue
        it = int(parsed_name["iteration"])
        kind = str(parsed_name["kind"])
        study_id = None
        if row.study is not None and row.study.reference is not None:
            study_id = row.study.reference.target_id
        succeeded = await _study_success_count(runtime, tenant_id, study_id)
        parsed.append(
            {
                "idx": idx,
                "row": row,
                "name": name,
                "logical_name": str(parsed_name["logical_name"]),
                "prefix": prefix,
                "iteration": it,
                "kind": kind,
                "tag": parsed_name["tag"],
                "study_id": study_id,
                "succeeded": succeeded,
            }
        )

    completed_iterations = sorted(
        {
            int(rec["iteration"])
            for rec in parsed
            if rec["succeeded"] > 0 and rec["kind"] in ("jointSweep", "candidate", "legalFallback")
        }
    )
    total_completed = int(len(completed_iterations))
    last_completed_iteration = int(max(completed_iterations, default=0))

    resume_candidates = [
        rec
        for rec in parsed
        if rec["succeeded"] > 0 and rec["kind"] in ("jointSweep", "candidate", "legalFallback", "current")
    ]
    if resume_candidates:
        resume_candidates.sort(
            key=lambda r: (
                int(r["iteration"]),
                int(r["idx"]),
                KIND_PRIORITY.get(str(r["kind"]), -1),
            )
        )
        resume_row = resume_candidates[-1]
    else:
        resume_row = None

    kind_counts: dict[str, int] = {}
    for rec in parsed:
        kind = str(rec["kind"])
        kind_counts[kind] = int(kind_counts.get(kind, 0) + 1)

    return {
        "total_completed": total_completed,
        "last_completed_iteration": last_completed_iteration,
        "parsed_count": len(parsed),
        "resume_row": resume_row,
        "kind_counts": kind_counts,
    }



async def _load_resume_car_payload(runtime: RaceEngRuntime, resume_row: dict) -> tuple[str, dict]:
    car_id = _extract_row_car_id(resume_row["row"])
    if not car_id:
        raise RuntimeError(f"Could not extract car config id from row {resume_row['name']}")

    loaded = await raceeng_call_with_retry(
        runtime,
        f"load_config:{car_id}:resume_payload",
        lambda s: canopy.load_config(s, car_id),
    )
    payload = raceeng_config_result_to_payload(loaded, fallback_name=resume_row.get("logical_name", "resume-car"))
    return car_id, payload


async def run_autoresume():
    session_boot = None
    boot_runtime = None
    campaign_state_stem = None
    campaign_paths = None
    default_payloads = None
    consent_granted = False
    pending_auth = globals().get("RACEENG_AUTH_DATA")
    if pending_auth is None:
        try:
            pending_auth = raceeng_prompt_for_authentication_data()
        except RaceEngUserCancelledRun as ex:
            raceeng_clear_cached_auth_data()
            print(str(ex))
            return

    try:
        session_boot = await authenticate_with_auth_data(pending_auth)
        globals()["RACEENG_AUTH_DATA"] = raceeng_clone_auth_data(pending_auth)
        boot_runtime = RaceEngRuntime(
            auth_data=raceeng_clone_auth_data(globals()["RACEENG_AUTH_DATA"]),
            session=session_boot,
            campaign_state_stem=None,
        )
        worksheet_id, worksheet_name, resolution_mode = await resolve_or_create_destination_worksheet(
            session=session_boot,
            tenant_id=TARGET_TENANT_ID,
            destination=WORKSHEET_DESTINATION,
            create_if_missing=bool(CREATE_WORKSHEET_IF_MISSING),
            registry_path=WORKSHEET_REGISTRY_FILE,
            runtime=boot_runtime,
            defer_create_until_confirmed=True,
        )
        if worksheet_id is not None:
            campaign_state_stem = _campaign_state_stem_for_worksheet(worksheet_id)
            campaign_paths = raceeng_campaign_state_paths(campaign_state_stem)
        default_payloads = {
            key: await load_default_config_payload_from_url(
                session_boot,
                url,
                runtime=boot_runtime,
            )
            for key, url in DEFAULT_CONFIG_URLS.items()
        }
        print(
            f"Destination worksheet resolved: id={worksheet_id}, name='{worksheet_name}', mode={resolution_mode}"
        )
        if worksheet_id is not None:
            print(
                f"Worksheet URL: https://portal.canopysimulations.com/worksheets/{TARGET_TENANT_ID}/{worksheet_id}"
            )
            print(f"Campaign state stem: {campaign_state_stem}")
        else:
            print("Worksheet creation is deferred until the launch warning is accepted.")
        if RUN_WORKSHEET_NUMBER is not None:
            print(f"Run worksheet number: {RUN_WORKSHEET_NUMBER}")
        print("Loaded Canopy cloud defaults:")
        for key in ["car", "weather", "track", "userMaths"]:
            payload = default_payloads[key]
            print(f"  {key:9s} {payload.get('name')}")
    except Exception:
        raceeng_clear_cached_auth_data()
        raise
    finally:
        await raceeng_shutdown_worksheet_writer(boot_runtime)
        await _close_sessions(
            None if boot_runtime is None else boot_runtime.session,
            session_boot,
        )

    if default_payloads is None:
        raceeng_clear_cached_auth_data()
        raise RuntimeError("Bootstrap failed before default config loading completed.")

    done = False

    for attempt in range(1, MAX_AUTO_RESTARTS + 1):
        session = None
        runtime = None
        try:
            auth_data = raceeng_clone_auth_data(globals().get("RACEENG_AUTH_DATA"))
            if auth_data is None:
                raise RuntimeError("Authentication data is not available in memory. Re-run the notebook cell to authenticate.")

            session = await authenticate_with_auth_data(auth_data)
            runtime = RaceEngRuntime(
                auth_data=raceeng_clone_auth_data(auth_data),
                session=session,
                campaign_state_stem=campaign_state_stem,
            )
            if worksheet_id is not None and campaign_state_stem is not None:
                worksheet_state = await _resolve_progress_state(runtime, TARGET_TENANT_ID, worksheet_id, BASE_RUN_PREFIX)
                campaign_state = _load_matching_campaign_state(campaign_state_stem, worksheet_id)
                campaign_complete = _campaign_is_complete(campaign_state)
            else:
                worksheet_state = {
                    "parsed_count": 0,
                    "kind_counts": {},
                    "last_completed_iteration": 0,
                    "resume_row": None,
                }
                campaign_state = None
                campaign_complete = False

            if isinstance(campaign_state, dict):
                completed = int(
                    campaign_state.get(
                        "last_clean_iteration",
                        int(campaign_state.get("next_iteration", 1)) - 1,
                    )
                )
                completed = max(0, completed)
                start_iteration = int(campaign_state.get("next_iteration", completed + 1))
                resume_state = campaign_state
                run_car_payload = copy.deepcopy(default_payloads["car"])
                source_desc = (
                    f"campaign-state(iter={completed}, next={start_iteration}, file={campaign_paths['json']})"
                )
            else:
                completed = int(worksheet_state.get("last_completed_iteration", 0))
                start_iteration = int(completed + 1)
                resume_state = None
                resume_row = worksheet_state["resume_row"]
                if resume_row is None:
                    run_car_payload = copy.deepcopy(default_payloads["car"])
                    source_desc = "base-car(default-cloud-config)"
                else:
                    source_car_id, run_car_payload = await _load_resume_car_payload(runtime, resume_row)
                    source_desc = f"{resume_row['name']} (car_id={source_car_id})"

            remaining = max(0, int(TOTAL_TARGET_ITERATIONS - completed))
            finalization_only = bool(
                remaining <= 0 and not campaign_complete and isinstance(campaign_state, dict)
            )
            run_iterations = 1 if finalization_only else remaining

            print(
                f"[auto {attempt}/{MAX_AUTO_RESTARTS}] completed={completed}, remaining={remaining}, parsed_rows={worksheet_state['parsed_count']}"
            )
            print(f"[auto {attempt}/{MAX_AUTO_RESTARTS}] parsed_kind_counts={worksheet_state.get('kind_counts', {})}")
            print(
                f"[auto {attempt}/{MAX_AUTO_RESTARTS}] campaign_complete={campaign_complete}, finalization_only={finalization_only}"
            )
            if isinstance(campaign_state, dict):
                print(
                    f"[auto {attempt}/{MAX_AUTO_RESTARTS}] campaign_state=loaded next_iteration={campaign_state.get('next_iteration')} "
                    f"last_clean_iteration={campaign_state.get('last_clean_iteration')} status={campaign_state.get('campaign_status')}"
                )
            resume_row = worksheet_state["resume_row"]
            if resume_row is not None:
                print(
                    f"[auto {attempt}/{MAX_AUTO_RESTARTS}] resume_row={resume_row['name']} "
                    f"(logical={resume_row['logical_name']}, iter={resume_row['iteration']}, "
                    f"kind={resume_row['kind']}, succeeded={resume_row['succeeded']})"
                )

            if remaining <= 0 and campaign_complete:
                print("Campaign already completed cleanly. Auto-resume finished.")
                done = True
                break
            if remaining <= 0 and not finalization_only:
                print("Target iterations reached but no resumable campaign state exists. Auto-resume finished.")
                done = True
                break
            if finalization_only:
                print("Nominal iterations reached but campaign finalization is incomplete; running finalization-only pass.")

            if not consent_granted:
                resumed = bool(isinstance((resume_state or {}).get("baseline_bundle"), dict) or resume_row is not None or completed > 0)
                budget = _estimate_remaining_sim_budget(
                    run_car_payload,
                    start_iteration=start_iteration,
                    resume_state=resume_state,
                    finalization_only=finalization_only,
                )
                _prompt_for_run_consent(
                    budget=budget,
                    worksheet_id=str(worksheet_id or "pending-create"),
                    worksheet_name=str(worksheet_name),
                    source_desc=str(source_desc),
                    resumed=resumed,
                )
                consent_granted = True
                if worksheet_id is None:
                    worksheet_id, worksheet_name, resolution_mode = await resolve_or_create_destination_worksheet(
                        session=session,
                        tenant_id=TARGET_TENANT_ID,
                        destination=WORKSHEET_DESTINATION,
                        create_if_missing=bool(CREATE_WORKSHEET_IF_MISSING),
                        registry_path=WORKSHEET_REGISTRY_FILE,
                        runtime=runtime,
                        defer_create_until_confirmed=False,
                    )
                    campaign_state_stem = _campaign_state_stem_for_worksheet(worksheet_id)
                    campaign_paths = raceeng_campaign_state_paths(campaign_state_stem)
                    runtime.campaign_state_stem = campaign_state_stem
                    print(
                        f"Worksheet created after consent: id={worksheet_id}, name='{worksheet_name}', mode={resolution_mode}"
                    )
                    print(
                        f"Worksheet URL: https://portal.canopysimulations.com/worksheets/{TARGET_TENANT_ID}/{worksheet_id}"
                    )
                    print(f"Campaign state stem: {campaign_state_stem}")

            row_prefix = f"{BASE_RUN_PREFIX}-auto-{datetime.now(UTC).strftime('%H%M%S')}-a{attempt:02d}"
            print(f"Starting run chunk: row_prefix={row_prefix}")
            print(f"Resume source: {source_desc}")

            session = await raceeng_runtime_ensure_session(runtime)
            result_payload, sweep_df, sens_df, iter_df = await import_configs_and_optimize_balanced_car(
                session=session,
                tenant_id=TARGET_TENANT_ID,
                worksheet_id=worksheet_id,
                worksheet_name=worksheet_name,
                worksheet_number=RUN_WORKSHEET_NUMBER,
                row_prefix=row_prefix,
                max_iterations=run_iterations,
                timeout_seconds=STUDY_TIMEOUT_SECONDS,
                car_payload=run_car_payload,
                weather_payload=copy.deepcopy(default_payloads["weather"]),
                track_payload=copy.deepcopy(default_payloads["track"]),
                user_maths_payload=copy.deepcopy(default_payloads["userMaths"]),
                auth_data=auth_data,
                sweep_points_per_parameter=SWEEP_POINTS_PER_PARAMETER,
                focus_fraction=FOCUS_FRACTION,
                endgame_fraction=ENDGAME_FRACTION,
                endgame_verify_top_k=ENDGAME_VERIFY_TOP_K,
                endgame_extra_cycles_max=ENDGAME_EXTRA_CYCLES_MAX,
                endgame_stall_patience=ENDGAME_STALL_PATIENCE,
                endgame_noise_margin_factor=ENDGAME_NOISE_MARGIN_FACTOR,
                elite_archive_size=ELITE_ARCHIVE_SIZE,
                endgame_verification_debt_threshold=ENDGAME_VERIFICATION_DEBT_THRESHOLD,
                endgame_unverified_improvement_reset=ENDGAME_UNVERIFIED_IMPROVEMENT_RESET,
                final_candidate_policy=FINAL_CANDIDATE_POLICY,
                force_two_main_starts=FORCE_TWO_MAIN_STARTS,
                worksheet_row_batch_seconds=WORKSHEET_ROW_BATCH_SECONDS,
                worksheet_row_batch_size=WORKSHEET_ROW_BATCH_SIZE,
                start_iteration=start_iteration,
                campaign_total_iterations=TOTAL_TARGET_ITERATIONS,
                resume_state=resume_state,
                campaign_state_stem=campaign_state_stem,
                finalization_only=finalization_only,
            )
            print("Run chunk completed cleanly.")
            print(json.dumps(result_payload, indent=2))

            campaign_state_post = _load_matching_campaign_state(campaign_state_stem, worksheet_id)
            if _campaign_is_complete(campaign_state_post) or bool(result_payload.get("campaign_completed_cleanly", False)):
                print("Campaign completed cleanly. Auto-resume finished.")
                done = True
                break
            if finalization_only:
                print("Finalization-only pass completed but campaign is still not marked complete; retry loop will continue.")
        except RaceEngUserCancelledRun as ex:
            print(str(ex))
            done = True
            raceeng_clear_cached_auth_data()
            return
        except Exception as ex:
            if raceeng_is_fatal_notebook_error(ex):
                print(f"Fatal notebook error: {type(ex).__name__}: {ex}")
                print(traceback.format_exc())
                done = True
                raceeng_clear_cached_auth_data()
                return
            print(f"Chunk failed: {type(ex).__name__}: {ex}")
            print(traceback.format_exc())
        finally:
            await raceeng_shutdown_worksheet_writer(runtime)
            await _close_sessions(None if runtime is None else runtime.session, session)

        await asyncio.sleep(float(RETRY_DELAY_SECONDS))

    if done:
        raceeng_clear_cached_auth_data()
        return
    raceeng_clear_cached_auth_data()
    raise RuntimeError("Auto-resume exhausted restart attempts before campaign completion")


## Run Entry Point

Run the next cell to start. The notebook will:
1. Prompt for credentials.
2. Resolve or create the destination worksheet.
3. Estimate the remaining simulation budget and ask for `yes` consent.
4. Start or resume the campaign.


In [ ]:
await run_autoresume()
